In [9]:
import webbrowser
from threading import Timer
import dash
from dash import dcc, html, Input, Output, State, dash_table
import pandas as pd
import base64
import io
import math
import random
import numpy as np
import plotly.graph_objects as go
from dash.exceptions import PreventUpdate

In [10]:
def energia_final(E, A, theta):
    theta_rad = math.radians(theta)
    return E * ((A*A + 1 + 2*A*math.cos(theta_rad)) / ((A+1)**2))

def monte_carlo(p, n0, pasos=50):
    neutrones = [n0]
    for _ in range(pasos):
        nuevos = sum(
            random.randint(0,2) if random.random() < p else 0
            for _ in range(neutrones[-1])
        )
        neutrones.append(nuevos)
    return neutrones

In [11]:
# ================= CREAR APLICACIÓN =================
app = dash.Dash(__name__)

# ================= CSS EMBEBIDO =================
app.index_string = """
<!DOCTYPE html>
<html>
    <head>
        {%metas%}
        <title>Laboratorio Nuclear</title>
        {%favicon%}
        {%css%}
        <link href="https://fonts.googleapis.com/css2?family=Montserrat:wght@300;400;500;600;700;800&display=swap" rel="stylesheet">
        <style>
        body {
          background: #0a0f0d;
          color: #e5e7eb;
          font-family: 'Montserrat', 'Segoe UI', sans-serif;
          margin: 0;
          padding: 0;
        }

        .header {
          display: grid;
          grid-template-columns: 1fr auto 1fr;
          align-items: center;
          padding: 20px 30px;
          border-bottom: 1px solid rgba(0,255,156,0.2);
          background: linear-gradient(135deg, #0a0f0d 0%, #121a17 100%);
        }

        .logo-left, .logo-right {
          height: 100px;
          width: auto;
          filter: brightness(0) invert(1) drop-shadow(0 0 8px rgba(0,255,156,0.6));
          transition: all 0.3s ease;
        }
        
        .logo-left:hover, .logo-right:hover {
          transform: scale(1.08);
          filter: brightness(0) invert(1) drop-shadow(0 0 15px rgba(0,255,156,0.9));
        }

        .logo-left {
          justify-self: start;
          background: rgba(0,255,156,0.08);
          border-radius: 15px;
          padding: 12px;
        }

        .logo-right {
          justify-self: end;
          background: rgba(0,255,156,0.08);
          border-radius: 15px;
          padding: 12px;
        }

        .main {
          max-width: 1400px;
          margin: auto;
          padding: 30px;
        }

        .cards__inner {
          display: flex;
          flex-wrap: wrap;
          gap: 25px;
          margin-top: 25px;
        }

        .card {
          flex: 1 1 320px;
          background: linear-gradient(135deg, #121a17 0%, #0e1512 100%);
          border: 1px solid rgba(0,255,156,0.3);
          padding: 25px;
          border-radius: 16px;
          box-shadow: 0 8px 25px rgba(0,0,0,0.4);
          transition: all 0.3s ease;
        }

        .card:hover {
          border-color: #00ff9c;
          box-shadow: 0 0 25px rgba(0,255,156,0.25);
          transform: translateY(-4px);
        }

        .card h3 {
          color: #00ff9c;
          margin-top: 0;
          margin-bottom: 18px;
          border-bottom: 2px solid rgba(0,255,156,0.3);
          padding-bottom: 10px;
          font-weight: 600;
          letter-spacing: -0.3px;
        }

        button {
          background: linear-gradient(135deg, #00ff9c 0%, #00cc7a 100%);
          color: #0a0f0d;
          border: none;
          padding: 12px;
          border-radius: 10px;
          font-weight: 700;
          font-family: 'Montserrat', sans-serif;
          cursor: pointer;
          margin-top: 12px;
          width: 100%;
          transition: all 0.3s ease;
          font-size: 14px;
        }

        button:hover {
          transform: translateY(-2px);
          box-shadow: 0 6px 20px rgba(0,255,156,0.35);
        }

        input {
          margin: 8px 0;
          padding: 10px;
          border-radius: 10px;
          background: #1e2a24;
          color: #e5e7eb;
          border: 1px solid rgba(0,255,156,0.3);
          width: 100%;
          box-sizing: border-box;
          transition: all 0.3s ease;
          font-family: 'Montserrat', sans-serif;
          font-size: 14px;
        }

        input:focus {
          outline: none;
          border-color: #00ff9c;
          box-shadow: 0 0 12px rgba(0,255,156,0.4);
        }

        label {
          display: block;
          margin-top: 12px;
          margin-bottom: 6px;
          color: #00ff9c;
          font-size: 0.85em;
          font-weight: 500;
          letter-spacing: 0.3px;
        }

        .dash-graph {
          background: transparent !important;
        }
        
        .dash-table-container {
          overflow-x: auto;
          margin-top: 12px;
        }
        
        .dash-spreadsheet-container {
          max-width: 100%;
          overflow-x: auto;
        }
        
        .upload-area {
          width: 100%;
          height: 70px;
          line-height: 70px;
          border-width: 2px;
          border-style: dashed;
          border-radius: 12px;
          text-align: center;
          margin: 12px 0;
          border-color: rgba(0,255,156,0.4);
          cursor: pointer;
          transition: all 0.3s ease;
          background: rgba(0,255,156,0.05);
          font-family: 'Montserrat', sans-serif;
          font-weight: 500;
        }
        
        .upload-area:hover {
          border-color: #00ff9c;
          background: rgba(0,255,156,0.12);
          transform: translateY(-2px);
        }
        
        hr {
          border-color: rgba(0,255,156,0.2);
          margin: 20px 0;
        }
        
        /* Estilos para la tabla */
        .dash-table-container .dash-spreadsheet-container .dash-spreadsheet-inner {
          background-color: #121a17 !important;
          font-family: 'Montserrat', sans-serif !important;
        }
        
        .dash-table-container .dash-spreadsheet-container .dash-spreadsheet-inner th {
          background-color: #0a0f0d !important;
          color: #00ff9c !important;
          border-color: rgba(0,255,156,0.3) !important;
          font-weight: 600 !important;
          font-family: 'Montserrat', sans-serif !important;
        }
        
        .dash-table-container .dash-spreadsheet-container .dash-spreadsheet-inner td {
          color: #e5e7eb !important;
          border-color: rgba(0,255,156,0.2) !important;
          font-family: 'Montserrat', sans-serif !important;
        }
        
        /* Estilos para el slider */
        .rc-slider-track {
          background-color: #00ff9c;
        }
        
        .rc-slider-handle {
          border-color: #00ff9c;
          background-color: #00ff9c;
        }
        
        .rc-slider-handle:hover {
          border-color: #00cc7a;
        }
        
        /* Estilos adicionales para mejor legibilidad */
        h1, h2, h3, h4, h5, h6 {
          font-family: 'Montserrat', sans-serif;
          font-weight: 600;
        }
        
        .dash-spreadsheet {
          font-family: 'Montserrat', sans-serif !important;
        }
        </style>
    </head>
    <body>
        {%app_entry%}
        <footer>
            {%config%}
            {%scripts%}
            <script>
            document.addEventListener("mousemove", (e) => {
              document.documentElement.style.setProperty("--x", e.clientX + "px");
              document.documentElement.style.setProperty("--y", e.clientY + "px");
            });
            </script>
            {%renderer%}
        </footer>
    </body>
</html>
"""



In [12]:
# ================= LAYOUT =================

app.layout = html.Div([

    # HEADER
    html.Div([

        html.Img(
            src="https://upload.wikimedia.org/wikipedia/commons/c/ca/Escudo-UNAM-escalable.svg",
            className="logo-left",
            style={
                "filter": "brightness(0) invert(1) drop-shadow(0 0 8px rgba(0,255,156,0.6))",
                "background": "rgba(0,255,156,0.1)",
                "borderRadius": "15px",
                "padding": "12px"
            }
        ),

        html.Div([
            html.H1("Laboratorio Virtual Nuclear", style={
                "margin": "0",
                "background": "linear-gradient(135deg, #00ff9c 0%, #00cc7a 100%)",
                "WebkitBackgroundClip": "text",
                "WebkitTextFillColor": "transparent",
                "backgroundClip": "text",
                "fontSize": "2.2em",
                "fontWeight": "700",
                "letterSpacing": "-0.5px"
            }),
            html.H4("Facultad de Ciencias — UNAM", style={
                "margin": "8px 0 0 0",
                "color": "#8a9b8f",
                "fontWeight": "500",
                "letterSpacing": "0.3px"
            })
        ], style={"textAlign": "center"}),

        html.Img(
            src="https://upload.wikimedia.org/wikipedia/commons/5/5b/Escudo-facultad-ciencias-unam-escalable.svg",
            className="logo-right",
            style={
                "filter": "brightness(0) invert(1) drop-shadow(0 0 8px rgba(0,255,156,0.6))",
                "background": "rgba(0,255,156,0.1)",
                "borderRadius": "15px",
                "padding": "12px"
            }
        ),

    ], className="header"),

    html.Div([

        # CARD 1 - Calculadora
        html.Div([
            html.H3("Calculadora de Energía"),
            html.Label("Energía inicial (E):"),
            dcc.Input(id='E', type='number', value=2),
            html.Label("Número másico (A):"),
            dcc.Input(id='A', type='number', value=1),
            html.Label("Ángulo (θ en grados):"),
            dcc.Input(id='theta', type='number', value=30),
            html.Button("Calcular", id='btn'),
            html.Div(id='resultado', style={
                "marginTop": "15px", 
                "fontWeight": "600", 
                "textAlign": "center",
                "padding": "12px",
                "background": "rgba(0,255,156,0.1)",
                "borderRadius": "10px",
                "border": "1px solid rgba(0,255,156,0.3)",
                "fontFamily": "'Montserrat', sans-serif"
            }),
            html.Hr(),
            html.H4("Cargar datos CSV", style={"marginBottom": "10px", "fontWeight": "600"}),
            dcc.Upload(
                id='upload-data', 
                children=html.Div([' Arrastra o selecciona un archivo CSV']),
                className="upload-area"
            ),
            html.Div(id='tabla')
        ], className="card"),

        # CARD 2 - Monte Carlo
        html.Div([
            html.H3("Simulación Monte Carlo"),
            html.Label("Probabilidad de fisión (p):"),
            dcc.Slider(id='p', min=0, max=1, step=0.01, value=0.5),
            html.Br(),
            html.Label("Número inicial de neutrones:"),
            dcc.Input(id='n0', type='number', value=10),
            dcc.Graph(id='grafica_mc', style={"height": "400px", "marginTop": "12px"})
        ], className="card"),

        # CARD 3 - Dispersión 3D
        html.Div([
            html.H3("Dispersión Nuclear 3D"),
            html.Label("Número másico (A):"),
            dcc.Input(id='A2', type='number', value=1),
            dcc.Graph(id='grafica_disp', style={"height": "400px", "marginTop": "12px"})
        ], className="card"),

        # CARD 4 - Árbol de decisiones
        html.Div([
            html.H3("Árbol de Decisiones"),
            dcc.Graph(id='arbol', style={"height": "400px"})
        ], className="card"),

        # CARD 5 - Animación
        html.Div([
            html.H3("Animación de Partícula"),
            dcc.Interval(id='interval', interval=1000),
            dcc.Graph(id='animacion', style={"height": "400px"})
        ], className="card"),

    ], className="cards__inner")

], className="main")



In [13]:
# ================= CALLBACKS =================

@app.callback(
    Output('resultado', 'children'),
    Input('btn', 'n_clicks'),
    State('E', 'value'),
    State('A', 'value'),
    State('theta', 'value'),
    prevent_initial_call=True
)
def calcular(n, E, A, theta):
    if n is None:
        raise PreventUpdate
    try:
        if None in [E, A, theta]:
            return "Por favor, ingrese valores válidos"
        resultado = energia_final(float(E), float(A), float(theta))
        return f"Energía final (E') = {resultado:.4f} MeV"
    except Exception as e:
        return f"Error: {str(e)}"

@app.callback(
    Output('tabla', 'children'),
    Input('upload-data', 'contents')
)
def parse_csv(contents):
    if contents is None:
        return ""
    try:
        _, content_string = contents.split(',')
        decoded = base64.b64decode(content_string)
        
        # Intentar detectar el separador
        try:
            df = pd.read_csv(io.StringIO(decoded.decode('utf-8')))
        except:
            df = pd.read_csv(io.StringIO(decoded.decode('utf-8')), sep=';')
        
        # Limitar el número de columnas para evitar desbordamiento
        MAX_COLUMNAS = 8
        if len(df.columns) > MAX_COLUMNAS:
            df = df.iloc[:, :MAX_COLUMNAS]
            col_warning = html.Div(f"Mostrando solo {MAX_COLUMNAS} de {len(df.columns)} columnas", 
                                   style={"color": "#ffaa00", "fontSize": "12px", "marginTop": "5px"})
        else:
            col_warning = html.Div()
        
        return html.Div([
            html.H5(f"Datos cargados: {len(df)} filas, {len(df.columns)} columnas", 
                   style={"color": "#00ff9c", "marginBottom": "5px", "fontWeight": "600"}),
            col_warning,
            dash_table.DataTable(
                data=df.to_dict('records'),
                columns=[{"name": i, "id": i} for i in df.columns],
                page_size=5,
                style_table={
                    'overflowX': 'auto',
                    'minWidth': '100%'
                },
                style_cell={
                    'minWidth': '50px',
                    'maxWidth': '150px',
                    'overflow': 'hidden',
                    'textOverflow': 'ellipsis',
                    'backgroundColor': '#121a17',
                    'color': '#e5e7eb',
                    'border': '1px solid rgba(0,255,156,0.2)',
                    'fontFamily': "'Montserrat', sans-serif"
                },
                style_header={
                    'backgroundColor': '#0a0f0d',
                    'color': '#00ff9c',
                    'fontWeight': 'bold',
                    'border': '1px solid rgba(0,255,156,0.3)',
                    'fontFamily': "'Montserrat', sans-serif"
                }
            )
        ])
    except Exception as e:
        return html.Div(f"Error al cargar archivo: {str(e)}", style={"color": "red"})

@app.callback(
    Output('grafica_mc', 'figure'),
    Input('p', 'value'),
    Input('n0', 'value')
)
def actualizar_mc(p, n0):
    if p is None or n0 is None:
        raise PreventUpdate
    
    data = monte_carlo(p, int(n0))
    fig = go.Figure(
        data=[go.Scatter(
            y=data, 
            line=dict(color="#00ff9c", width=2),
            mode='lines+markers',
            marker=dict(size=6, color="#00ff9c")
        )],
        layout=go.Layout(
            template="plotly_dark",
            title=dict(
                text=f"Simulación Monte Carlo (p={p:.2f}, n₀={n0})",
                font=dict(family="Montserrat, sans-serif", size=14, color="#e5e7eb")
            ),
            xaxis_title=dict(text="Pasos de tiempo", font=dict(family="Montserrat, sans-serif")),
            yaxis_title=dict(text="Número de neutrones", font=dict(family="Montserrat, sans-serif")),
            plot_bgcolor='rgba(0,0,0,0)',
            paper_bgcolor='rgba(0,0,0,0)',
            font=dict(family="Montserrat, sans-serif", color="#e5e7eb")
        )
    )
    return fig

@app.callback(
    Output('arbol', 'figure'),
    Input('p', 'value')
)
def arbol(p):
    if p is None:
        raise PreventUpdate
    
    # Crear estructura de árbol
    niveles = list(range(10))
    valores = []
    for nivel in niveles:
        for i in range(2**nivel):
            if random.random() < p:
                valores.append(nivel)
    
    fig = go.Figure(
        data=go.Scatter(
            x=valores, 
            y=[random.random() for _ in valores], 
            mode='markers',
            marker=dict(color="#00ff9c", size=8, symbol='circle')
        ),
        layout=go.Layout(
            template="plotly_dark",
            title=dict(
                text=f"Árbol de decisiones (p={p:.2f})",
                font=dict(family="Montserrat, sans-serif", size=14, color="#e5e7eb")
            ),
            xaxis_title=dict(text="Nivel", font=dict(family="Montserrat, sans-serif")),
            yaxis_title=dict(text="Valor aleatorio", font=dict(family="Montserrat, sans-serif")),
            showlegend=False,
            plot_bgcolor='rgba(0,0,0,0)',
            paper_bgcolor='rgba(0,0,0,0)',
            font=dict(family="Montserrat, sans-serif", color="#e5e7eb")
        )
    )
    return fig

@app.callback(
    Output('grafica_disp', 'figure'),
    Input('A2', 'value')
)
def dispersion_3D(A):
    if A is None:
        A = 1
    
    theta = np.linspace(0, 180, 30)
    phi = np.linspace(0, 360, 30)
    theta, phi = np.meshgrid(theta, phi)
    
    A_val = float(A)
    Ep = (1 * ((A_val*A_val + 1 + 2*A_val*np.cos(np.radians(theta))) / ((A_val+1)**2)))
    
    x = Ep * np.sin(np.radians(theta)) * np.cos(np.radians(phi))
    y = Ep * np.sin(np.radians(theta)) * np.sin(np.radians(phi))
    z = Ep * np.cos(np.radians(theta))
    
    fig = go.Figure(
        data=[go.Scatter3d(
            x=x.flatten(),
            y=y.flatten(),
            z=z.flatten(),
            mode='markers',
            marker=dict(
                size=3,
                color=Ep.flatten(),
                colorscale='Viridis',
                showscale=True,
                colorbar=dict(title="Energía", title_font=dict(family="Montserrat, sans-serif"))
            ),
            name='Dispersión'
        )],
        layout=go.Layout(
            template="plotly_dark",
            title=dict(
                text=f"Dispersión Nuclear (A={A_val})",
                font=dict(family="Montserrat, sans-serif", size=14, color="#e5e7eb")
            ),
            scene=dict(
                xaxis_title=dict(text='X', font=dict(family="Montserrat, sans-serif")),
                yaxis_title=dict(text='Y', font=dict(family="Montserrat, sans-serif")),
                zaxis_title=dict(text='Z', font=dict(family="Montserrat, sans-serif")),
                bgcolor='rgba(0,0,0,0)',
                xaxis=dict(color="#e5e7eb"),
                yaxis=dict(color="#e5e7eb"),
                zaxis=dict(color="#e5e7eb")
            ),
            plot_bgcolor='rgba(0,0,0,0)',
            paper_bgcolor='rgba(0,0,0,0)',
            font=dict(family="Montserrat, sans-serif", color="#e5e7eb")
        )
    )
    return fig

@app.callback(
    Output('animacion', 'figure'),
    Input('interval', 'n_intervals')
)
def animacion(n):
    # Movimiento browniano simple
    x = [0, random.uniform(-1, 1)]
    y = [0, random.uniform(-1, 1)]
    
    fig = go.Figure(
        data=go.Scatter(
            x=x, 
            y=y, 
            mode='lines+markers',
            line=dict(color="#00ff9c", width=2),
            marker=dict(size=10, color="#00ff9c", symbol='circle')
        ),
        layout=go.Layout(
            template="plotly_dark",
            title=dict(
                text=f"Movimiento de partícula (t={n+1})",
                font=dict(family="Montserrat, sans-serif", size=14, color="#e5e7eb")
            ),
            xaxis=dict(
                range=[-1.5, 1.5], 
                gridcolor='rgba(0,255,156,0.2)', 
                color="#e5e7eb",
                title_font=dict(family="Montserrat, sans-serif")
            ),
            yaxis=dict(
                range=[-1.5, 1.5], 
                gridcolor='rgba(0,255,156,0.2)', 
                color="#e5e7eb",
                title_font=dict(family="Montserrat, sans-serif")
            ),
            xaxis_title=dict(text="Posición X", font=dict(family="Montserrat, sans-serif")),
            yaxis_title=dict(text="Posición Y", font=dict(family="Montserrat, sans-serif")),
            plot_bgcolor='rgba(0,0,0,0)',
            paper_bgcolor='rgba(0,0,0,0)',
            font=dict(family="Montserrat, sans-serif", color="#e5e7eb")
        )
    )
    return fig


In [ ]:

# para correr

def open_browser():
    webbrowser.open("http://127.0.0.1:8050")

if __name__ == '__main__':
    Timer(1, open_browser).start()
    app.run(debug=True, use_reloader=False)

## ejemplos

## parte de app

In [1]:
import webbrowser
import math
import random
import numpy as np
from http.server import HTTPServer, BaseHTTPRequestHandler
import json
from threading import Timer
import urllib.parse

def energia_final(E, A, theta):
    """Calcula la energía final después de dispersión nuclear"""
    theta_rad = math.radians(theta)
    return E * ((A*A + 1 + 2*A*math.cos(theta_rad)) / ((A+1)**2))

def monte_carlo(p, n0, pasos=50):
    """Simulación Monte Carlo de población de neutrones"""
    neutrones = [n0]
    for _ in range(pasos):
        nuevos = sum(
            random.randint(0, 2) if random.random() < p else 0
            for _ in range(neutrones[-1])
        )
        neutrones.append(nuevos)
    return neutrones

HTML_FISION = '''
<!DOCTYPE html>
<html>
<head>
<meta charset="UTF-8">
<title>Simulador de Fision Nuclear</title>
<style>
body { margin:0; overflow:hidden; background:#010105; }
#app { width:100%; height:100vh; }
.back-btn {
    position:absolute;
    top:200px;
    left:20px;
    z-index:20;
    background:#111;
    color:#fff;
    border:1px solid #00ff9c;
    padding:10px 20px;
    cursor:pointer;
    font-family: monospace;
    font-size:14px;
    text-decoration:none;
    border-radius:5px;
    transition: all 0.2s;
}
.back-btn:hover {
    background:#00ff9c;
    color:#000;
}
.controls {
position:absolute;
top:20px;
left:120px;
z-index:10;
display: flex;
gap: 10px;
flex-wrap: wrap;
}
button {
padding:10px 15px;
background:#111;
color:#fff;
border:1px solid #444;
cursor:pointer;
font-family: monospace;
font-size: 14px;
transition: all 0.2s;
}
button:hover {
background:#333;
transform: scale(1.02);
}
.info {
position:absolute;
bottom:20px;
left:20px;
color:#fff;
background:rgba(0,0,0,0.8);
padding:12px;
font-family: monospace;
font-size:14px;
z-index:10;
border-radius:5px;
backdrop-filter: blur(5px);
max-width: 280px;
}
.info h3 {
margin:0 0 8px 0;
color: #ffaa66;
}
.info p {
margin:4px 0;
font-size: 12px;
}
.stats {
position:absolute;
top:20px;
right:20px;
color:#fff;
background:rgba(0,0,0,0.8);
padding:12px;
font-family: monospace;
font-size:14px;
z-index:10;
border-radius:5px;
text-align: right;
min-width: 200px;
backdrop-filter: blur(5px);
}
.stats .value {
color: #ffaa66;
font-size: 20px;
font-weight: bold;
}
.stats .label {
font-size: 12px;
color: #aaa;
}
.warning {
color: #ff8866;
animation: pulse 1s infinite;
font-weight: bold;
}
@keyframes pulse {
0%, 100% { opacity: 1; }
50% { opacity: 0.5; }
}
</style>
</head>
<body>
<a href="/" class="back-btn">← Volver al Laboratorio</a>
<div class="controls">
    <button id="fireBtn">Disparar neutron</button>
    <button id="resetBtn">Reiniciar simulacion</button>
    <button id="chainBtn">Iniciar reaccion en cadena</button>
</div>
<div class="stats">
    <h3>Estado del reactor</h3>
    <p><span class="label">Atomos de Uranio:</span> <span id="atomCount" class="value">0</span></p>
    <p><span class="label">Neutrones activos:</span> <span id="neutronCount" class="value">0</span></p>
    <p><span class="label">Fisiones totales:</span> <span id="fissionCount" class="value">0</span></p>
    <p><span class="label">Energia liberada:</span> <span id="energyReleased" class="value">0</span> MeV</p>
    <div id="chainWarning" style="display:none;" class="warning">
        REACCION EN CADENA ACTIVA
    </div>
</div>
<div class="info">
    <h3>FISION NUCLEAR - REACCION EN CADENA</h3>
    <p>n + U-235 -> Ba-141 + Kr-92 + 2-3 n + Energia (200 MeV)</p>
    <p>Los neutrones liberados pueden fisionar mas nucleos de Uranio</p>
    <p>Cada fision puede generar hasta 3 nuevas reacciones</p>
    <p>Total de atomos: 150 en disposicion ALEATORIA</p>
</div>
<div id="app"></div>
<script type="importmap">
{
  "imports": {
    "three": "https://esm.sh/three",
    "three/addons/": "https://esm.sh/three/addons/"
  }
}
</script>
<script type="module">
import * as THREE from "three";
import { OrbitControls } from "three/addons/controls/OrbitControls.js";

let scene, camera, renderer, controls;
let atoms = [];
let neutrons = [];
let fragments = [];
let energyParticles = [];
let fissionCount = 0;
let totalEnergy = 0;
let chainReactionActive = false;
let chainReactionInterval = null;

const ENERGY_PER_FISSION = 200;
const NEUTRON_SPEED = 0.35;
const FISSION_RADIUS = 1.3;

function init() {
    scene = new THREE.Scene();
    scene.background = new THREE.Color(0x010105);
    scene.fog = new THREE.FogExp2(0x010105, 0.0008);

    camera = new THREE.PerspectiveCamera(60, innerWidth/innerHeight, 0.1, 1000);
    camera.position.set(28, 22, 35);

    renderer = new THREE.WebGLRenderer({antialias: true});
    renderer.setSize(innerWidth, innerHeight);
    renderer.shadowMap.enabled = true;
    document.getElementById("app").appendChild(renderer.domElement);

    controls = new OrbitControls(camera, renderer.domElement);
    controls.enableDamping = true;
    controls.autoRotate = false;
    controls.enableZoom = true;
    controls.zoomSpeed = 1.2;
    controls.target.set(0, 0, 0);

    const ambientLight = new THREE.AmbientLight(0x222222);
    scene.add(ambientLight);

    const directionalLight = new THREE.DirectionalLight(0xffffff, 0.5);
    directionalLight.position.set(5, 10, 7);
    directionalLight.castShadow = true;
    scene.add(directionalLight);

    const backLight = new THREE.PointLight(0x4466ff, 0.3);
    backLight.position.set(-5, 0, -5);
    scene.add(backLight);

    const fillLight = new THREE.PointLight(0xffaa66, 0.2);
    fillLight.position.set(3, 5, 4);
    scene.add(fillLight);

    createInitialAtoms();
    animate();
    setInterval(updateUI, 100);
}

const tex = new THREE.TextureLoader().load(
    "https://threejs.org/examples/textures/sprites/disc.png"
);

const createShell = (count, radius, color) => {
    const pos = [];
    const col = [];
    
    const pointCount = Math.min(count * 6, 120);
    
    for (let i = 0; i < pointCount; i++){
        const r = radius + (Math.random() - 0.5) * 0.4;
        const theta = Math.acos(2 * Math.random() - 1);
        const phi = Math.random() * Math.PI * 2;
        
        const x = r * Math.sin(theta) * Math.cos(phi);
        const y = r * Math.sin(theta) * Math.sin(phi);
        const z = r * Math.cos(theta);
        
        pos.push(x, y, z);
        
        const c = new THREE.Color(color);
        const fade = 0.5 + Math.random() * 0.5;
        col.push(c.r * fade, c.g * fade, c.b * fade);
    }
    
    const geo = new THREE.BufferGeometry();
    geo.setAttribute("position", new THREE.Float32BufferAttribute(pos, 3));
    geo.setAttribute("color", new THREE.Float32BufferAttribute(col, 3));
    
    const mat = new THREE.PointsMaterial({
        size: 0.06,
        map: tex,
        transparent: true,
        vertexColors: true,
        blending: THREE.AdditiveBlending,
        depthWrite: false
    });
    
    return new THREE.Points(geo, mat);
};

function createAtom(position, color = 0xff3366) {
    const group = new THREE.Group();
    
    const nucleus = new THREE.Mesh(
        new THREE.SphereGeometry(0.55, 20, 20),
        new THREE.MeshStandardMaterial({ 
            color: color, 
            emissive: 0x331100, 
            emissiveIntensity: 0.12, 
            metalness: 0.6,
            roughness: 0.4
        })
    );
    nucleus.castShadow = true;
    group.add(nucleus);
    
    const shells = [
        {n: 2,  r: 1.1, color: 0xff6688},
        {n: 8,  r: 1.7, color: 0xff7799},
        {n: 18, r: 2.4, color: 0xff88aa},
        {n: 32, r: 3.1, color: 0xff99bb},
        {n: 21, r: 3.9, color: 0xffaacc},
        {n: 9,  r: 4.8, color: 0xffbbdd},
        {n: 2,  r: 5.7, color: 0xffccee}
    ];
    
    const clouds = shells.map(s => {
        const c = createShell(s.n, s.r, s.color);
        group.add(c);
        return c;
    });
    
    group.userData = {
        clouds: clouds,
        type: "uranium"
    };
    
    group.position.copy(position);
    scene.add(group);
    
    return group;
}

function createInitialAtoms() {
    atoms.forEach(atom => scene.remove(atom));
    atoms = [];
    
    const positions = [];
    const numAtoms = 150;
    
    const radius = 14;
    
    for(let i = 0; i < numAtoms; i++) {
        let x, y, z, distance;
        do {
            x = (Math.random() - 0.5) * radius * 2;
            y = (Math.random() - 0.5) * radius * 1.5;
            z = (Math.random() - 0.5) * radius * 2;
            distance = Math.sqrt(x*x + y*y + z*z);
        } while(distance > radius);
        
        positions.push(new THREE.Vector3(x, y, z));
    }
    
    positions.forEach(pos => {
        atoms.push(createAtom(pos));
    });
    
    console.log("Creados " + atoms.length + " atomos en posiciones aleatorias");
    
    fissionCount = 0;
    totalEnergy = 0;
    updateUI();
}

function createNeutron(position, direction, speed = NEUTRON_SPEED) {
    const neutronMesh = new THREE.Mesh(
        new THREE.SphereGeometry(0.16, 12, 12),
        new THREE.MeshStandardMaterial({ 
            color: 0xffdd99, 
            emissive: 0xff8844, 
            emissiveIntensity: 0.7 
        })
    );
    
    neutronMesh.position.copy(position);
    neutronMesh.castShadow = true;
    
    neutronMesh.userData = {
        velocity: direction.clone().normalize().multiplyScalar(speed)
    };
    
    scene.add(neutronMesh);
    neutrons.push(neutronMesh);
    updateUI();
}

function fission(atom, neutronPosition) {
    const atomPos = atom.position.clone();
    
    scene.remove(atom);
    const atomIndex = atoms.indexOf(atom);
    if(atomIndex !== -1) atoms.splice(atomIndex, 1);
    
    fissionCount++;
    totalEnergy += ENERGY_PER_FISSION;
    updateUI();
    
    if(chainReactionActive && atoms.length > 0) {
        document.getElementById("chainWarning").style.display = "block";
        document.body.style.animation = "pulse 0.3s";
        setTimeout(() => {
            document.body.style.animation = "";
        }, 300);
    }
    
    const bariumGroup = new THREE.Group();
    const bariumNucleus = new THREE.Mesh(
        new THREE.SphereGeometry(0.42, 18, 18),
        new THREE.MeshStandardMaterial({ color: 0x66cc99, emissive: 0x226644, emissiveIntensity: 0.2 })
    );
    bariumNucleus.castShadow = true;
    bariumGroup.add(bariumNucleus);
    
    const bariumShells = [
        {n: 2, r: 0.8, color: 0x88ffaa},
        {n: 8, r: 1.25, color: 0xaaffcc},
        {n: 18, r: 1.75, color: 0xccffdd},
        {n: 18, r: 2.3, color: 0xddffee},
        {n: 8, r: 2.9, color: 0xeeffff},
        {n: 2, r: 3.5, color: 0xaaffff}
    ];
    
    bariumShells.forEach(shell => {
        const cloud = createShell(shell.n, shell.r, shell.color);
        bariumGroup.add(cloud);
    });
    
    const kryptonGroup = new THREE.Group();
    const kryptonNucleus = new THREE.Mesh(
        new THREE.SphereGeometry(0.32, 18, 18),
        new THREE.MeshStandardMaterial({ color: 0xff9966, emissive: 0x442200, emissiveIntensity: 0.2 })
    );
    kryptonNucleus.castShadow = true;
    kryptonGroup.add(kryptonNucleus);
    
    const kryptonShells = [
        {n: 2, r: 0.7, color: 0xffaa88},
        {n: 8, r: 1.1, color: 0xffbb99},
        {n: 18, r: 1.6, color: 0xffccaa},
        {n: 8, r: 2.1, color: 0xffddbb}
    ];
    
    kryptonShells.forEach(shell => {
        const cloud = createShell(shell.n, shell.r, shell.color);
        kryptonGroup.add(cloud);
    });
    
    const direction = neutronPosition.clone().sub(atomPos).normalize();
    
    bariumGroup.position.copy(atomPos.clone().add(direction.clone().multiplyScalar(1.5)));
    kryptonGroup.position.copy(atomPos.clone().add(direction.clone().multiplyScalar(-1.5)));
    
    bariumGroup.userData = {
        velocity: direction.clone().multiplyScalar(0.05),
        type: "barium"
    };
    kryptonGroup.userData = {
        velocity: direction.clone().multiplyScalar(-0.05),
        type: "krypton"
    };
    
    scene.add(bariumGroup);
    scene.add(kryptonGroup);
    fragments.push(bariumGroup, kryptonGroup);
    
    const neutronCount = 2 + Math.floor(Math.random() * 2);
    
    for(let i = 0; i < neutronCount; i++) {
        const randomDir = new THREE.Vector3(
            (Math.random() - 0.5) * 2,
            (Math.random() - 0.5) * 2,
            (Math.random() - 0.5) * 2
        ).normalize();
        
        const speedMultiplier = chainReactionActive ? 1.15 : 1.0;
        createNeutron(atomPos, randomDir, NEUTRON_SPEED * speedMultiplier);
    }
    
    const particleCount = 60;
    const colors = [0xffaa44, 0xff6644, 0xffaa88, 0xffcc66];
    
    for(let i = 0; i < particleCount; i++) {
        const particle = new THREE.Mesh(
            new THREE.SphereGeometry(0.05, 4, 4),
            new THREE.MeshBasicMaterial({ 
                color: colors[Math.floor(Math.random() * colors.length)],
                transparent: true,
                blending: THREE.AdditiveBlending
            })
        );
        
        particle.position.copy(atomPos);
        
        const angle1 = Math.random() * Math.PI * 2;
        const angle2 = Math.random() * Math.PI * 2;
        const speed = 0.1 + Math.random() * 0.15;
        
        particle.userData = {
            velocity: new THREE.Vector3(
                Math.sin(angle1) * Math.cos(angle2) * speed,
                Math.sin(angle1) * Math.sin(angle2) * speed,
                Math.cos(angle1) * speed
            ),
            life: 1.0,
            decay: 0.01 + Math.random() * 0.01
        };
        
        scene.add(particle);
        energyParticles.push(particle);
    }
    
    const explosionLight = new THREE.PointLight(0xff8844, 1.8, 18);
    explosionLight.position.copy(atomPos);
    scene.add(explosionLight);
    
    let intensity = 1.8;
    const lightInterval = setInterval(() => {
        intensity -= 0.2;
        explosionLight.intensity = intensity;
        if(intensity <= 0) {
            clearInterval(lightInterval);
            scene.remove(explosionLight);
        }
    }, 40);
    
    const ringGeometry = new THREE.RingGeometry(0.5, 1.2, 32);
    const ringMaterial = new THREE.MeshBasicMaterial({ color: 0xffaa66, transparent: true, opacity: 0.7, side: THREE.DoubleSide });
    const shockRing = new THREE.Mesh(ringGeometry, ringMaterial);
    shockRing.position.copy(atomPos);
    scene.add(shockRing);
    
    let ringScale = 1;
    const ringInterval = setInterval(() => {
        ringScale += 0.25;
        shockRing.scale.set(ringScale, ringScale, 1);
        ringMaterial.opacity -= 0.04;
        if(ringMaterial.opacity <= 0) {
            clearInterval(ringInterval);
            scene.remove(shockRing);
        }
    }, 25);
}

function startChainReaction() {
    if(chainReactionActive) return;
    
    chainReactionActive = true;
    document.getElementById("chainWarning").style.display = "block";
    
    console.log("Iniciando reaccion en cadena con " + atoms.length + " atomos");
    
    if(atoms.length > 0 && neutrons.length === 0) {
        const start = new THREE.Vector3(0, 5, 25);
        const target = atoms[Math.floor(Math.random() * atoms.length)].position;
        const direction = new THREE.Vector3().subVectors(target, start);
        createNeutron(start, direction, NEUTRON_SPEED * 1.2);
    }
    
    chainReactionInterval = setInterval(() => {
        if(chainReactionActive && atoms.length > 0 && neutrons.length < 10) {
            const randomAtom = atoms[Math.floor(Math.random() * atoms.length)];
            const start = new THREE.Vector3(
                randomAtom.position.x + (Math.random() - 0.5) * 8,
                randomAtom.position.y + (Math.random() - 0.5) * 6 + 6,
                randomAtom.position.z + (Math.random() - 0.5) * 8 + 18
            );
            const direction = new THREE.Vector3().subVectors(randomAtom.position, start);
            createNeutron(start, direction, NEUTRON_SPEED * 1.25);
        }
        
        if(atoms.length === 0 && chainReactionActive) {
            stopChainReaction();
            console.log("Reaccion en cadena completada - Todos los atomos fisionados");
        }
    }, 1200);
}

function stopChainReaction() {
    chainReactionActive = false;
    document.getElementById("chainWarning").style.display = "none";
    if(chainReactionInterval) {
        clearInterval(chainReactionInterval);
        chainReactionInterval = null;
    }
}

function resetSimulation() {
    stopChainReaction();
    
    neutrons.forEach(n => scene.remove(n));
    fragments.forEach(f => scene.remove(f));
    energyParticles.forEach(p => scene.remove(p));
    atoms.forEach(a => scene.remove(a));
    
    neutrons = [];
    fragments = [];
    energyParticles = [];
    
    createInitialAtoms();
    updateUI();
}

function updateUI() {
    document.getElementById("atomCount").innerHTML = atoms.length;
    document.getElementById("neutronCount").innerHTML = neutrons.length;
    document.getElementById("fissionCount").innerHTML = fissionCount;
    document.getElementById("energyReleased").innerHTML = totalEnergy.toFixed(0);
}

function animate() {
    requestAnimationFrame(animate);
    
    atoms.forEach(atom => {
        if(atom.userData && atom.userData.clouds) {
            atom.userData.clouds.forEach((c, i) => {
                c.rotation.y += 0.0005 * (i + 1);
                c.rotation.x += 0.0003 * (i + 1);
            });
        }
    });
    
    for(let i = fragments.length - 1; i >= 0; i--) {
        const fragment = fragments[i];
        if(fragment.userData && fragment.userData.velocity) {
            fragment.position.add(fragment.userData.velocity);
            
            fragment.children.forEach(child => {
                if(child.isPoints) {
                    child.rotation.y += 0.012;
                    child.rotation.x += 0.008;
                    child.rotation.z += 0.005;
                }
            });
        }
    }
    
    for(let i = energyParticles.length - 1; i >= 0; i--) {
        const p = energyParticles[i];
        p.position.add(p.userData.velocity);
        p.userData.life -= p.userData.decay;
        
        const scale = p.userData.life * 0.6;
        p.scale.set(scale, scale, scale);
        p.material.opacity = p.userData.life;
        
        if(p.userData.life <= 0) {
            scene.remove(p);
            energyParticles.splice(i, 1);
        }
    }
    
    for(let i = neutrons.length - 1; i >= 0; i--) {
        const n = neutrons[i];
        n.position.add(n.userData.velocity);
        
        const pulse = 0.6 + Math.sin(Date.now() * 0.02) * 0.4;
        n.material.emissiveIntensity = pulse;
        
        let hit = false;
        
        for(let j = 0; j < atoms.length; j++) {
            const atom = atoms[j];
            if(atom && n.position.distanceTo(atom.position) < FISSION_RADIUS) {
                fission(atom, n.position);
                scene.remove(n);
                neutrons.splice(i, 1);
                hit = true;
                break;
            }
        }
        
        if(!hit && (Math.abs(n.position.x) > 40 || Math.abs(n.position.y) > 30 || Math.abs(n.position.z) > 40)) {
            scene.remove(n);
            neutrons.splice(i, 1);
        }
    }
    
    controls.update();
    renderer.render(scene, camera);
}

document.getElementById("fireBtn").onclick = () => {
    if(atoms.length === 0) {
        alert("No quedan atomos de Uranio. Reinicia la simulacion.");
        return;
    }
    
    const start = new THREE.Vector3(0, 5, 28);
    const target = atoms[Math.floor(Math.random() * atoms.length)].position;
    const direction = new THREE.Vector3().subVectors(target, start);
    createNeutron(start, direction);
};

document.getElementById("resetBtn").onclick = resetSimulation;
document.getElementById("chainBtn").onclick = startChainReaction;

window.addEventListener("resize", () => {
    camera.aspect = innerWidth / innerHeight;
    camera.updateProjectionMatrix();
    renderer.setSize(innerWidth, innerHeight);
});

init();
</script>
</body>
</html>
'''

HTML_PRINCIPAL = """
<!DOCTYPE html>
<html lang="es">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Laboratorio Nuclear - Simulador de Dispersión</title>
    
    <link href="https://fonts.googleapis.com/css2?family=League+Spartan:wght@400;500;600;700;800;900&display=swap" rel="stylesheet">
    <script src="https://cdn.plot.ly/plotly-3.1.0.min.js"></script>
    
    <style>
        *,
        *::after,
        *::before {
            box-sizing: border-box;
            margin: 0;
            padding: 0;
        }

        html, body {
            height: 100%;
            min-height: 100vh;
        }

        body {
            font-family: "League Spartan", system-ui, sans-serif;
            font-size: 1rem;
            line-height: 1.2;
            background-color: #212121;
            color: #ddd;
            padding: 2rem;
        }

        .main {
            max-width: 1400px;
            margin: 0 auto;
        }

        .main__heading {
            font-weight: 600;
            font-size: 2.5em;
            margin-bottom: 0.2em;
            text-align: center;
            background: linear-gradient(135deg, #00ff9c 0%, #667eea 100%);
            -webkit-background-clip: text;
            -webkit-text-fill-color: transparent;
            background-clip: text;
        }

        .subheading {
            text-align: center;
            color: #00ff9c;
            margin-bottom: 2em;
            font-size: 1.1em;
            opacity: 0.8;
        }

        .cards__inner {
            display: flex;
            flex-wrap: wrap;
            gap: 1.5em;
            margin-bottom: 2em;
        }

        .card {
            flex: 1 1 300px;
            padding: 1.5em;
            background: linear-gradient(135deg, #2b2b2b 0%, #1e1e1e 100%);
            border: 1px solid rgba(0, 255, 156, 0.2);
            border-radius: 15px;
            transition: all 0.3s ease;
        }

        .card:hover {
            transform: translateY(-5px);
            border-color: rgba(0, 255, 156, 0.5);
            box-shadow: 0 10px 30px rgba(0, 255, 156, 0.1);
        }

        .card__heading {
            font-size: 1.3em;
            font-weight: 700;
            margin-bottom: 1em;
            color: #00ff9c;
        }

        .input-group {
            margin-bottom: 1em;
        }

        .input-group label {
            display: block;
            margin-bottom: 0.3em;
            font-weight: 500;
            color: #aaa;
        }

        .input-group input {
            width: 100%;
            padding: 0.5em;
            background: #1a1a1a;
            border: 1px solid #333;
            border-radius: 8px;
            color: #fff;
            font-family: inherit;
            font-size: 0.9rem;
        }

        .input-group input:focus {
            outline: none;
            border-color: #00ff9c;
        }

        button {
            width: 100%;
            padding: 0.7em;
            background: linear-gradient(135deg, #00ff9c 0%, #667eea 100%);
            color: #1a1a1a;
            border: none;
            border-radius: 10px;
            font-family: "League Spartan", system-ui, sans-serif;
            font-weight: 700;
            cursor: pointer;
            transition: transform 0.2s;
        }

        button:hover {
            transform: scale(1.02);
        }

        .resultado {
            margin-top: 1em;
            padding: 0.8em;
            background: rgba(0, 255, 156, 0.1);
            border-radius: 8px;
            text-align: center;
            font-weight: 600;
            color: #00ff9c;
        }

        .graph-container {
            background: #1a1a1a;
            border-radius: 15px;
            padding: 1em;
            margin-bottom: 1.5em;
            border: 1px solid rgba(0, 255, 156, 0.2);
        }

        .graph-title {
            margin-bottom: 1em;
            font-weight: 600;
            color: #00ff9c;
            text-align: center;
        }

        .two-columns {
            display: grid;
            grid-template-columns: 1fr 1fr;
            gap: 1.5em;
            margin-bottom: 1.5em;
        }
        
        .fission-btn {
            background: linear-gradient(135deg, #ff6600 0%, #ff3300 100%);
            color: white;
            margin-bottom: 1em;
        }
        
        .fission-btn:hover {
            transform: scale(1.02);
        }

        @media (max-width: 768px) {
            .two-columns {
                grid-template-columns: 1fr;
            }
            body {
                padding: 1rem;
            }
        }
        
        .badge {
            display: inline-block;
            background: rgba(0, 255, 156, 0.2);
            padding: 0.2em 0.5em;
            border-radius: 5px;
            font-size: 0.8em;
            margin-top: 0.5em;
        }
    </style>
</head>
<body>
    <div class="main">
        <h1 class="main__heading">Laboratorio Nuclear</h1>
        <div class="subheading">Simulador de Dispersión de Neutrones | Fisica Nuclear</div>
        
        <button class="fission-btn" onclick="window.location.href='/fision'" style="margin-bottom: 20px; width: auto; padding: 12px 24px;">
            Acceder a Simulador de Fision Nuclear
        </button>

        <div class="cards__inner">
            <div class="card">
                <div class="card__heading">Energia Final</div>
                <div class="input-group">
                    <label>Energia inicial (E) MeV</label>
                    <input type="number" id="energia_inicial" value="1" step="0.1">
                </div>
                <div class="input-group">
                    <label>Numero masico (A)</label>
                    <input type="number" id="masa_A" value="1" step="0.5">
                </div>
                <div class="input-group">
                    <label>Angulo de dispersion (grados)</label>
                    <input type="number" id="angulo" value="0" step="10">
                </div>
                <button onclick="calcularEnergia()">Calcular Energia Final</button>
                <div id="resultado_energia" class="resultado">Ingrese valores</div>
            </div>

            <div class="card">
                <div class="card__heading">Monte Carlo</div>
                <div class="input-group">
                    <label>Probabilidad de multiplicacion (p)</label>
                    <input type="number" id="prob_p" value="0.5" step="0.05" min="0" max="1">
                </div>
                <div class="input-group">
                    <label>Neutrones iniciales (n)</label>
                    <input type="number" id="neutrones_n0" value="100">
                </div>
                <button onclick="simularMonteCarlo()">Ejecutar Simulacion</button>
                <div id="resultado_mc" class="resultado">Listo para simular</div>
            </div>

            <div class="card">
                <div class="card__heading">Parametros de Dispersion</div>
                <div class="input-group">
                    <label>Numero masico (A)</label>
                    <input type="number" id="masa_A_2d" value="2" step="0.5" min="1">
                </div>
                <div class="input-group">
                    <label>Energia inicial (E) MeV</label>
                    <input type="number" id="energia_E_2d" value="1" step="0.5">
                </div>
                <button onclick="actualizarGrafica2D()">Actualizar Grafica</button>
            </div>
        </div>

        <div class="graph-container">
            <div class="graph-title">Simulacion Monte Carlo</div>
            <div id="grafica_mc" style="height: 400px;"></div>
        </div>

        <div class="graph-container">
            <div class="graph-title">Energia vs Angulo de Dispersion</div>
            <div id="grafica_2d" style="height: 400px;"></div>
        </div>
    </div>

    <script>
        function energiaFinal(E, A, theta) {
            const thetaRad = theta * Math.PI / 180;
            return E * ((A*A + 1 + 2*A*Math.cos(thetaRad)) / ((A+1)**2));
        }
        
        function monteCarlo(p, n0, pasos=50) {
            let neutrones = [n0];
            for (let step = 0; step < pasos; step++) {
                let nuevos = 0;
                for (let i = 0; i < neutrones[neutrones.length-1]; i++) {
                    if (Math.random() < p) {
                        nuevos += Math.floor(Math.random() * 3);
                    }
                }
                neutrones.push(nuevos);
            }
            return neutrones;
        }
        
        function calcularEnergia() {
            const E = parseFloat(document.getElementById('energia_inicial').value);
            const A = parseFloat(document.getElementById('masa_A').value);
            const theta = parseFloat(document.getElementById('angulo').value);
            
            const resultado = energiaFinal(E, A, theta);
            document.getElementById('resultado_energia').innerHTML = 
                `Energia final (E') = ${resultado.toFixed(4)} MeV`;
        }
        
        function simularMonteCarlo() {
            const p = parseFloat(document.getElementById('prob_p').value);
            const n0 = parseInt(document.getElementById('neutrones_n0').value);
            
            const datos = monteCarlo(p, n0, 50);
            
            document.getElementById('resultado_mc').innerHTML = 
                `Simulacion completada: ${datos.length} pasos | Final: ${datos[datos.length-1]} neutrones`;
            
            const trace = {
                y: datos,
                mode: 'lines+markers',
                line: {color: '#00ff9c', width: 2},
                marker: {size: 6, color: '#00ff9c'}
            };
            
            const layout = {
                template: 'plotly_dark',
                title: {text: `Monte Carlo (p=${p}, n=${n0})`, font: {family: "League Spartan", size: 14}},
                xaxis: {title: "Pasos de tiempo", titlefont: {family: "League Spartan"}},
                yaxis: {title: "Numero de neutrones", titlefont: {family: "League Spartan"}},
                plot_bgcolor: 'rgba(0,0,0,0)',
                paper_bgcolor: 'rgba(0,0,0,0)'
            };
            
            Plotly.newPlot('grafica_mc', [trace], layout);
        }
        
        function actualizarGrafica2D() {
            const A = parseFloat(document.getElementById('masa_A_2d').value);
            const E = parseFloat(document.getElementById('energia_E_2d').value);
            
            const theta_vals = Array.from({length: 181}, (_, i) => i);
            const energia_vals = theta_vals.map(theta => energiaFinal(E, A, theta));
            
            const trace2D = {
                x: theta_vals,
                y: energia_vals,
                mode: 'lines',
                line: {color: '#00ff9c', width: 2}
            };
            
            const layout2D = {
                template: 'plotly_dark',
                title: {text: `Energia vs Angulo (A=${A})`, font: {family: "League Spartan", size: 14}},
                xaxis: {title: "Angulo (grados)", titlefont: {family: "League Spartan"}},
                yaxis: {title: "Energia final E' (MeV)", titlefont: {family: "League Spartan"}},
                plot_bgcolor: 'rgba(0,0,0,0)',
                paper_bgcolor: 'rgba(0,0,0,0)'
            };
            
            Plotly.newPlot('grafica_2d', [trace2D], layout2D);
        }
        
        window.onload = () => {
            calcularEnergia();
            simularMonteCarlo();
            actualizarGrafica2D();
        };
    </script>
</body>
</html>
"""

class Handler(BaseHTTPRequestHandler):
    def do_GET(self):
        if self.path == '/':
            self.send_response(200)
            self.send_header('Content-type', 'text/html; charset=utf-8')
            self.end_headers()
            self.wfile.write(HTML_PRINCIPAL.encode('utf-8'))
        elif self.path == '/fision':
            self.send_response(200)
            self.send_header('Content-type', 'text/html; charset=utf-8')
            self.end_headers()
            self.wfile.write(HTML_FISION.encode('utf-8'))
        else:
            self.send_response(404)
            self.end_headers()
    
    def log_message(self, format, *args):
        pass

def open_browser():
    webbrowser.open('http://localhost:8080')

def main():
    port = 8080
    server = HTTPServer(('localhost', port), Handler)
    print("Servidor nuclear iniciado en http://localhost:{}".format(port))
    print("Abriendo navegador...")
    print("Presiona Ctrl+C para detener el servidor")
    
    Timer(1, open_browser).start()
    
    try:
        server.serve_forever()
    except KeyboardInterrupt:
        print("\nCerrando servidor...")
        server.shutdown()

if __name__ == '__main__':
    main()

Servidor nuclear iniciado en http://localhost:8080
Abriendo navegador...
Presiona Ctrl+C para detener el servidor

Cerrando servidor...


## pruebas

In [1]:
import webbrowser
import math
import random
from http.server import HTTPServer, BaseHTTPRequestHandler
from threading import Timer

def energia_final(E, A, theta):
    theta_rad = math.radians(theta)
    return E * ((A*A + 1 + 2*A*math.cos(theta_rad)) / ((A+1)**2))

def monte_carlo(p, n0, pasos=50):
    neutrones = [n0]
    for _ in range(pasos):
        nuevos = sum(
            random.randint(0, 2) if random.random() < p else 0
            for _ in range(neutrones[-1])
        )
        neutrones.append(nuevos)
    return neutrones

HTML_PRINCIPAL = r"""<!DOCTYPE html>
<html lang="es">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Laboratorio Nuclear</title>
    <link href="https://fonts.googleapis.com/css2?family=Nunito:wght@300;400;600;700&display=swap" rel="stylesheet">
    <script src="https://cdn.plot.ly/plotly-3.1.0.min.js"></script>
    <script src="https://cdnjs.cloudflare.com/ajax/libs/mathjax/2.7.7/MathJax.js?config=TeX-MML-AM_CHTML"></script>
    <style>
        * {
            border: 0;
            box-sizing: border-box;
            margin: 0;
            padding: 0;
        }
        :root {
            --hue: 184;
            --bg: hsl(var(--hue), 10%, 90%);
            --fg: hsl(var(--hue), 66%, 24%);
            --primary: hsl(var(--hue), 66%, 44%);
            --gradient: linear-gradient(145deg, hsl(var(--hue), 10%, 85%), hsl(var(--hue), 10%, 100%));
            font-size: 16px;
        }
        body,
        button {
            color: var(--fg);
            font: 1em/1.5 "Nunito", sans-serif;
        }
        body {
            background: var(--bg);
            height: 100vh;
            display: grid;
            place-items: center;
            padding: 1.5em 0 0 0;
            position: relative;
        }
        body:after {
            content: "";
            display: block;
            height: 1.5em;
            width: 100%;
        }
        .app {
            background: hsl(var(--hue), 10%, 85%);
            border-radius: 3em;
            flex-direction: column;
            padding: 2.25em;
            width: 24.375em;
            height: auto;
            min-height: 52.75em;
            max-height: 90vh;
            overflow-y: auto;
            box-shadow: 0 8px 32px rgba(0, 0, 0, 0.1);
            border: 1px solid rgba(255, 255, 255, 0.5);
        }
        .app__gradients {
            position: absolute;
            width: 1px;
            height: 1px;
        }
        .icon {
            display: block;
            margin: auto;
            width: 1.5em;
            height: 1.5em;
        }
        .icon circle,
        .icon path {
            fill: currentColor;
            transition: fill 0.15s linear;
        }
        .icon ellipse,
        .icon polygon {
            stroke: currentColor;
            transition: stroke 0.15s linear;
        }
        .icon .no-fill {
            fill: none;
            stroke: currentColor;
        }
        .icon--red path {
            fill: hsl(3, 90%, 55%);
        }
        .icon--pulse {
            animation:
                bpm 1s linear,
                pulse 0.75s 1s linear infinite;
        }
        .ring,
        .sr-only {
            position: absolute;
        }
        .ring {
            display: block;
            inset: 0;
            width: 100%;
            height: auto;
        }
        .ring-fill,
        .ring-stroke {
            stroke: url("#ring");
        }
        .ring-stroke {
            animation-duration: 1s;
            animation-timing-function: ease-in-out;
        }
        .ring-track {
            stroke: hsl(var(--hue), 10%, 80%);
        }
        .sr-only {
            clip: rect(1px, 1px, 1px, 1px);
            overflow: hidden;
            width: 1px;
            height: 1px;
        }
        .header {
            display: flex;
            justify-content: space-between;
            align-items: center;
            margin-bottom: 1.5em;
        }
        .header__profile-btn,
        .header__notes-btn {
            width: 3em;
            height: 3em;
        }
        .header__profile-btn {
            border-radius: 1em;
            box-shadow: 0 0 0 0.125em inset;
        }
        .header__notes-btn {
            margin-inline-end: -1em;
        }
        .header__profile-btn:active,
        .header__notes-btn:active {
            transform: scale(0.9);
        }
        .header__profile-btn:focus {
            box-shadow: 0 0 0 0.125em var(--primary) inset;
        }
        .header__profile-icon {
            border-radius: 0.5em;
            margin: auto;
            width: 2em;
            height: 2em;
        }
        .header__notes-btn:focus .icon path {
            fill: var(--primary);
        }
        .header h1 {
            font-size: 1rem;
            text-align: center;
            flex: 1;
        }
        .main__date-nav {
            margin-bottom: 1.5em;
        }
        .main__date-arrow-btn,
        .main__date-edit-btn {
            height: 1.5em;
        }
        .main__date-arrow-btn {
            width: 1.5em;
        }
        .main__date-arrow-btn:active .icon path,
        .main__date-arrow-btn:focus .icon path {
            fill: var(--primary);
        }
        .main__date {
            text-transform: uppercase;
        }
        .main__date-edit-btn {
            min-width: 1.5em;
        }
        .main__date-edit-btn:active,
        .main__date-edit-btn:focus {
            color: var(--primary);
        }
        .main__stat-blocks {
            display: grid;
            grid-template-columns: repeat(3, 1fr);
            grid-gap: 1.5em;
            margin-bottom: 1.5em;
        }
        .main__stat-block {
            background: var(--gradient);
            border-radius: 1.5em;
            box-shadow:
                -0.75em -0.75em 2.25em hsl(0, 0%, 100%),
                0.75em 0.75em 2.25em hsl(var(--hue), 5%, 65%);
            padding: 0.75em;
            text-align: center;
            width: 100%;
        }
        .main__stat-block--lg {
            grid-column: 1 / 4;
            padding: 1.5em;
        }
        .main__stat-rows,
        .main__stat-row {
            margin-bottom: 1.5em;
        }
        .main__stat-row {
            align-items: center;
        }
        .main__stat-graph {
            margin: 0 auto 0.75em auto;
            position: relative;
            width: 3.75em;
            height: 3.75em;
        }
        .main__stat-graph .main__stat-detail {
            display: flex;
            flex-direction: column;
            justify-content: center;
            position: absolute;
            inset: 0;
        }
        .main__stat-block--lg .main__stat-graph {
            margin: auto;
            width: 11.25em;
            height: 11.25em;
        }
        .main__stat-block--lg .icon {
            margin: 0 auto;
            width: 2.25em;
            height: 2.25em;
        }
        .main__stat-row .main__stat-graph {
            background: var(--gradient);
            border-radius: 1em;
            box-shadow:
                -0.75em -0.75em 2.25em hsl(0, 0%, 100%),
                0.75em 0.75em 2.25em hsl(var(--hue), 5%, 65%);
            margin: 0;
            margin-inline-end: 1.5em;
        }
        .main__stat-value,
        .main__stat-unit {
            display: block;
        }
        .main__stat-value {
            font-size: 1.25em;
            line-height: 1.2;
        }
        .main__stat-block--lg .main__stat-value {
            font-size: 2em;
            line-height: 1.5;
        }
        .main__stat-unit,
        .main__stat-subtext {
            font-weight: 300;
        }
        .main__stat-subtext {
            color: hsl(var(--hue), 10%, 30%);
        }
        .footer {
            margin-top: auto;
        }
        .footer__nav-btn {
            background: var(--gradient);
            border-radius: 50%;
            box-shadow:
                1em 1em 2em hsl(var(--hue), 5%, 65%),
                -1em -1em 2em hsl(0, 0%, 100%);
            width: 3em;
            height: 3em;
        }
        .footer__nav-btn:active {
            box-shadow:
                0.75em 0.75em 1.5em hsl(var(--hue), 5%, 65%),
                -0.75em -0.75em 1.5em hsl(0, 0%, 100%);
            transform: scale(0.9);
        }
        .footer__nav-btn:focus .icon circle,
        .footer__nav-btn:focus .icon path {
            fill: var(--primary);
        }
        /* Dark theme */
        body.dark {
            --bg: hsl(var(--hue), 10%, 10%);
            --fg: hsl(var(--hue), 66%, 94%);
            --primary: hsl(var(--hue), 66%, 44%);
            --gradient: linear-gradient(145deg, hsl(var(--hue), 10%, 15%), hsl(var(--hue), 10%, 30%));
        }
        body.dark .app {
            background: hsl(var(--hue), 10%, 20%);
        }
        body.dark .icon--red path {
            fill: hsl(3, 90%, 65%);
        }
        body.dark .ring-track {
            stroke: hsl(var(--hue), 10%, 30%);
        }
        body.dark .main__stat-block,
        body.dark .main__stat-row .main__stat-graph {
            box-shadow:
                -0.75em -0.75em 2.25em hsl(var(--hue), 10%, 30%),
                0.75em 0.75em 2.25em hsl(var(--hue), 5%, 5%);
        }
        body.dark .main__stat-subtext {
            color: hsl(var(--hue), 10%, 70%);
        }
        body.dark .footer__nav-btn {
            box-shadow:
                -1em -1em 2em hsl(var(--hue), 10%, 30%),
                1em 1em 2em hsl(var(--hue), 5%, 5%);
        }
        body.dark .footer__nav-btn:active {
            box-shadow:
                -0.75em -0.75em 1.5em hsl(var(--hue), 10%, 30%),
                0.75em 0.75em 1.5em hsl(var(--hue), 5%, 5%);
        }
        @keyframes statFill {
            from { color: var(--fg); }
            to { color: hsl(var(--hue), 66%, 94%); }
        }
        @keyframes ringFill {
            from { r: 82px; stroke-width: 16; }
            to { r: 45px; stroke-width: 90; }
        }
        @keyframes stepCount {
            from { stroke-dashoffset: 515.22; }
            to { stroke-dashoffset: 0; }
        }
        @keyframes cals {
            from { stroke-dashoffset: 163.36; }
            to { stroke-dashoffset: 12.25; }
        }
        @keyframes miles {
            from { stroke-dashoffset: 163.36; }
            to { stroke-dashoffset: 35.39; }
        }
        @keyframes mins {
            from { stroke-dashoffset: 163.36; }
            to { stroke-dashoffset: 65.34; }
        }
        @keyframes bpm {
            from { transform: scale(0); }
            37.5% { transform: scale(1.2); }
            75%, to { transform: scale(1); }
        }
        @keyframes stepHrs {
            from { stroke-dashoffset: 131.95; }
            to { stroke-dashoffset: 52.78; }
        }
        @keyframes pulse {
            from, 75%, to { transform: scale(1); }
            25% { transform: scale(0.9); }
            50% { transform: scale(1.2); }
        }
        .nav-tabs {
            display: flex;
            gap: 0.5em;
            margin-bottom: 1.5em;
            justify-content: center;
        }
        .nav-btn {
            background: var(--gradient);
            border-radius: 1.5em;
            padding: 0.5em 1em;
            font-size: 0.7rem;
            font-weight: 600;
            cursor: pointer;
            box-shadow: -0.5em -0.5em 1.5em hsl(0, 0%, 100%), 0.5em 0.5em 1.5em hsl(var(--hue), 5%, 65%);
        }
        .nav-btn.active {
            background: var(--primary);
            color: white;
        }
        .tab-content {
            display: none;
        }
        .tab-content.active {
            display: block;
        }
        .card {
            background: var(--gradient);
            border-radius: 1.5em;
            padding: 1em;
            margin-bottom: 1em;
            box-shadow: -0.75em -0.75em 2.25em hsl(0, 0%, 100%), 0.75em 0.75em 2.25em hsl(var(--hue), 5%, 65%);
        }
        .card-title {
            font-weight: 700;
            margin-bottom: 0.5em;
            color: var(--primary);
        }
        .input-group {
            margin-bottom: 0.8em;
        }
        .input-group label {
            display: block;
            font-size: 0.7rem;
            font-weight: 600;
            margin-bottom: 0.2em;
        }
        .input-group input, .input-group select {
            width: 100%;
            padding: 0.4em 0.6em;
            background: hsl(var(--hue), 10%, 85%);
            border-radius: 1em;
            font-family: "Nunito", sans-serif;
            font-size: 0.8rem;
            border: 1px solid rgba(255, 255, 255, 0.3);
        }
        button {
            width: 100%;
            padding: 0.5em;
            background: var(--gradient);
            border-radius: 1.5em;
            font-family: "Nunito", sans-serif;
            font-weight: 600;
            cursor: pointer;
            margin-top: 0.3em;
            box-shadow: -0.5em -0.5em 1.5em hsl(0, 0%, 100%), 0.5em 0.5em 1.5em hsl(var(--hue), 5%, 65%);
            border: 1px solid rgba(255, 255, 255, 0.3);
        }
        button:active {
            transform: scale(0.95);
        }
        .btn-primary {
            background: var(--primary);
            color: white;
        }
        .resultado {
            margin-top: 0.5em;
            padding: 0.5em;
            background: hsl(var(--hue), 10%, 80%);
            border-radius: 1em;
            text-align: center;
            font-size: 0.7rem;
        }
        .graph-container {
            background: var(--gradient);
            border-radius: 1.5em;
            padding: 1em;
            margin-bottom: 1em;
            box-shadow: -0.75em -0.75em 2.25em hsl(0, 0%, 100%), 0.75em 0.75em 2.25em hsl(var(--hue), 5%, 65%);
        }
        .graph-title {
            font-size: 0.7rem;
            font-weight: 700;
            margin-bottom: 0.5em;
            text-align: center;
        }
        .simulation-canvas {
            width: 100%;
            height: 200px;
            background: hsl(var(--hue), 10%, 85%);
            border-radius: 1em;
        }
        .info-text {
            background: hsl(var(--hue), 10%, 80%);
            padding: 0.8em;
            border-radius: 1em;
            margin-bottom: 1em;
            font-size: 0.7rem;
            border-left: 3px solid var(--primary);
        }
        .formula {
            background: hsl(var(--hue), 10%, 80%);
            padding: 0.8em;
            border-radius: 1em;
            margin: 0.8em 0;
            font-size: 0.7rem;
            overflow-x: auto;
        }
        .nucleido-selector {
            display: flex;
            gap: 0.5em;
            margin-bottom: 0.8em;
            flex-wrap: wrap;
        }
        .nucleido-btn {
            flex: 1;
            padding: 0.3em;
            font-size: 0.7rem;
            margin: 0;
            background: hsl(var(--hue), 10%, 85%);
        }
        .nucleido-btn.active {
            background: var(--primary);
            color: white;
        }
        .flex-buttons {
            display: flex;
            gap: 0.5em;
        }
        .flex-buttons button {
            flex: 1;
        }
        
        /* Lightsaber Switch */
        .scene {
            display: flex;
            justify-content: center;
            align-items: center;
        }
        .images {
            position: absolute;
            opacity: 0;
            pointer-events: none;
        }
        .switch {
            display: flex;
            justify-content: center;
            align-items: center;
            width: 80px;
            cursor: pointer;
        }
        .switch input {
            position: absolute;
            opacity: 0;
        }
        .track {
            position: relative;
            width: 72px;
            height: 24px;
            background: #2c2c2c;
            border-radius: 60px;
            transition: 0.3s;
            box-shadow: inset 0 0 4px rgba(0,0,0,0.2);
        }
        body.dark .track {
            background: #1a1a1a;
        }
        .thumb {
            position: absolute;
            top: -6px;
            left: -6px;
            width: 36px;
            height: 36px;
            transition: 0.3s;
            display: flex;
            justify-content: center;
            align-items: center;
        }
        input:checked + .track .thumb {
            left: calc(100% - 36px + 6px);
        }
        .lightsaber {
            position: absolute;
            top: 0;
            left: 0;
            right: 0;
            bottom: 0;
            display: flex;
            justify-content: space-between;
            align-items: center;
            padding: 0 4px;
        }
        .lightsaber .light {
            width: 20px;
            height: 3px;
            background: #00ff9c;
            border-radius: 4px;
            box-shadow: 0 0 4px #00ff9c;
        }
        .lightsaber .dark {
            width: 20px;
            height: 3px;
            background: #ff3366;
            border-radius: 4px;
            box-shadow: 0 0 4px #ff3366;
        }
        .lightsaber .grip {
            width: 22px;
            height: 10px;
            background: #555;
            border-radius: 4px;
        }
        .circle {
            width: 36px;
            height: 36px;
            background: #1a1a1a;
            border-radius: 100%;
            display: flex;
            justify-content: center;
            align-items: center;
            transition: 0.2s;
            box-shadow: 0 2px 6px rgba(0,0,0,0.2);
        }
        body.dark .circle {
            background: #2a2a2a;
        }
        .sub-circle {
            width: 28px;
            height: 28px;
            background: #2c2c2c;
            border-radius: 100%;
        }
        body.dark .sub-circle {
            background: #3a3a3a;
        }
        .side {
            position: absolute;
            top: 0;
            left: 0;
            right: 0;
            bottom: 0;
            display: flex;
            justify-content: center;
            align-items: center;
            transition: 0.2s;
            transform: scale(0);
        }
        input:checked + .track .thumb .dark-side {
            transform: scale(1);
        }
        input:not(:checked) + .track .thumb .light-side {
            transform: scale(1);
        }
        .light-side .top {
            position: absolute;
            top: 4px;
            display: flex;
            gap: 4px;
        }
        .light-side .top .left,
        .light-side .top .right {
            width: 3px;
            height: 3px;
            background: #ffcc00;
            border-radius: 100%;
        }
        .light-side .center {
            display: flex;
            gap: 2px;
        }
        .light-side .center div {
            width: 2px;
            height: 10px;
            background: #ffcc00;
            border-radius: 2px;
        }
        .light-side .bottom {
            position: absolute;
            bottom: 4px;
        }
        .light-side .bottom .line {
            width: 12px;
            height: 2px;
            background: #ffcc00;
            border-radius: 2px;
        }
        .dark-side .circle {
            background: #ff3366;
        }
        .dark-side .sub-circle {
            background: #cc0033;
        }
        
        /* Scrollbar */
        .app::-webkit-scrollbar {
            width: 6px;
        }
        .app::-webkit-scrollbar-track {
            background: rgba(0, 0, 0, 0.1);
            border-radius: 10px;
        }
        .app::-webkit-scrollbar-thumb {
            background: var(--primary);
            border-radius: 10px;
        }
    </style>
</head>
<body>
    <div class="app">
        <div class="app__gradients">
            <svg style="width:0;height:0;position:absolute;" aria-hidden="true" focusable="false">
                <linearGradient id="ring" x1="0%" y1="0%" x2="100%" y2="100%">
                    <stop offset="0%" stop-color="hsl(184, 66%, 44%)" />
                    <stop offset="100%" stop-color="hsl(184, 66%, 74%)" />
                </linearGradient>
            </svg>
        </div>

        <div class="header">
            <button class="header__profile-btn" aria-label="Perfil">
                <svg class="icon header__profile-icon" viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="1.5">
                    <circle cx="12" cy="8" r="4" />
                    <path d="M5 20v-2a7 7 0 0 1 14 0v2" />
                </svg>
            </button>
            <h1>Laboratorio Nuclear</h1>
            <div class="scene">
                <div class="images">
                    <img src="https://assets.codepen.io/907471/darkside.svg?v42" class="dark">
                    <img src="https://assets.codepen.io/907471/lightside.svg?v42" class="light">
                </div>
                <label class="switch" id="switch" aria-label="Switch">
                    <input type="checkbox" id="themeToggle">
                    <div class="track">
                        <div class="lightsaber">
                            <div class="light"></div>
                            <div class="grip"></div>
                            <div class="dark"></div>
                        </div>
                        <div class="thumb">
                            <div class="side dark-side">
                                <div class="circle">
                                    <div class="sub-circle"></div>
                                </div>
                            </div>
                            <div class="side light-side">
                                <div class="circle">
                                    <div class="sub-circle"></div>
                                </div>
                                <div class="top">
                                    <div class="left"></div>
                                    <div class="right"></div>
                                </div>
                                <div class="center">
                                    <div class="item-1"></div>
                                    <div class="item-2"></div>
                                    <div class="item-3"></div>
                                    <div class="item-4"></div>
                                    <div class="item-5"></div>
                                </div>
                                <div class="bottom">
                                    <div class="line"></div>
                                </div>
                            </div>
                        </div>
                    </div>
                </label>
            </div>
        </div>

        <div class="main__stat-blocks">
            <div class="main__stat-block">
                <div class="main__stat-graph">
                    <svg class="ring" viewBox="0 0 120 120">
                        <circle class="ring-track" cx="60" cy="60" r="45" fill="none" stroke-width="8" />
                        <circle class="ring-stroke ring-stroke--steps" cx="60" cy="60" r="45" fill="none" stroke-width="8" stroke-dasharray="282.74" stroke-dashoffset="282.74" stroke-linecap="round" />
                    </svg>
                    <div class="main__stat-detail">
                        <span class="main__stat-value" id="statNeutrones">0</span>
                        <span class="main__stat-unit">n</span>
                    </div>
                </div>
                <span class="main__stat-subtext">Neutrones</span>
            </div>
            <div class="main__stat-block">
                <div class="main__stat-graph">
                    <svg class="ring" viewBox="0 0 120 120">
                        <circle class="ring-track" cx="60" cy="60" r="45" fill="none" stroke-width="8" />
                        <circle class="ring-stroke ring-stroke--cals" cx="60" cy="60" r="45" fill="none" stroke-width="8" stroke-dasharray="282.74" stroke-dashoffset="282.74" stroke-linecap="round" />
                    </svg>
                    <div class="main__stat-detail">
                        <span class="main__stat-value" id="statFisiones">0</span>
                        <span class="main__stat-unit">fis</span>
                    </div>
                </div>
                <span class="main__stat-subtext">Fisiones</span>
            </div>
            <div class="main__stat-block">
                <div class="main__stat-graph">
                    <svg class="ring" viewBox="0 0 120 120">
                        <circle class="ring-track" cx="60" cy="60" r="45" fill="none" stroke-width="8" />
                        <circle class="ring-stroke ring-stroke--miles" cx="60" cy="60" r="45" fill="none" stroke-width="8" stroke-dasharray="282.74" stroke-dashoffset="282.74" stroke-linecap="round" />
                    </svg>
                    <div class="main__stat-detail">
                        <span class="main__stat-value" id="statEnergia">0</span>
                        <span class="main__stat-unit">MeV</span>
                    </div>
                </div>
                <span class="main__stat-subtext">Energia</span>
            </div>
        </div>

        <div class="nav-tabs">
            <button class="nav-btn active" onclick="showTab('montecarlo')">Monte Carlo</button>
            <button class="nav-btn" onclick="showTab('dispersión')">Dispersion</button>
            <button class="nav-btn" onclick="showTab('fision')">Fision 3D</button>
        </div>

        <div id="montecarlo" class="tab-content active">
            <div class="info-text">
                <strong>Metodo Monte Carlo</strong><br>
                Simulacion de poblacion de neutrones usando numeros aleatorios.
            </div>
            <div class="card">
                <div class="card-title">Parametros</div>
                <div class="input-group">
                    <label>Probabilidad (p)</label>
                    <input type="number" id="prob_p" value="0.5" step="0.05" min="0" max="1">
                </div>
                <div class="input-group">
                    <label>Neutrones iniciales (n0)</label>
                    <input type="number" id="neutrones_n0" value="100">
                </div>
                <div class="input-group">
                    <label>Generaciones</label>
                    <input type="number" id="generaciones" value="50" min="10" max="100">
                </div>
                <button class="btn-primary" onclick="simularMonteCarlo()">Ejecutar</button>
                <div id="resultado_mc" class="resultado">Listo</div>
            </div>
            <div class="card">
                <div class="card-title">Interpretacion</div>
                <div class="info-text" style="margin:0">
                    <strong>Factor k:</strong><br>
                    k < 1: Subcritico | k = 1: Critico | k > 1: Supercritico
                </div>
            </div>
            <div class="graph-container">
                <div class="graph-title">Evolucion de la Poblacion</div>
                <div id="grafica_mc" style="height: 250px;"></div>
            </div>
            <div class="graph-container">
                <div class="graph-title">Visualizacion</div>
                <canvas id="mcCanvas" class="simulation-canvas"></canvas>
            </div>
        </div>

        <div id="dispersión" class="tab-content">
            <div class="info-text">
                <strong>Dispersion Elastica de Neutrones</strong><br>
                Energia final vs angulo de dispersion (0° a 360°)
            </div>
            <div class="formula">
                $$E_l' = \\frac{1}{2}E_l[(1+\\alpha)+(1-\\alpha)\\cos\\theta_C]$$
                $$\\alpha = \\left(\\frac{A-1}{A+1}\\right)^2$$
            </div>
            <div class="card">
                <div class="card-title">Parametros</div>
                <div class="input-group">
                    <label>Energia inicial (MeV)</label>
                    <input type="number" id="energia_inicial_elastica" value="1.0" step="0.1">
                </div>
                <div class="input-group">
                    <label>Nucleidos:</label>
                    <div class="nucleido-selector" id="nucleidosList">
                        <button type="button" class="nucleido-btn" data-a="1" data-name="H-1">H-1</button>
                        <button type="button" class="nucleido-btn" data-a="12" data-name="C-12">C-12</button>
                        <button type="button" class="nucleido-btn" data-a="23" data-name="Na-23">Na-23</button>
                        <button type="button" class="nucleido-btn" data-a="238" data-name="U-238">U-238</button>
                    </div>
                </div>
                <button class="btn-primary" onclick="agregarNucleido()">Agregar</button>
                <div class="flex-buttons">
                    <button onclick="limpiarGrafica()">Limpiar</button>
                </div>
                <div id="nucleidos_activos" class="resultado">Activos: Ninguno</div>
            </div>
            <div class="graph-container">
                <div class="graph-title">Energia vs Angulo (0° a 360°)</div>
                <div id="grafica_elastica" style="height: 300px;"></div>
            </div>
        </div>

        <div id="fision" class="tab-content">
            <div class="info-text">
                <strong>Fision Nuclear 3D</strong><br>
                n + U-235 -> Ba-141 + Kr-92 + 2-3 n + 200 MeV
            </div>
            <div class="card">
                <div class="card-title">Simulacion 3D</div>
                <button class="btn-primary" onclick="abrirFision()">Abrir Simulacion</button>
                <div class="info-text" style="margin-top:0.8em">
                    <strong>Controles:</strong><br>
                    Disparar neutron | Reaccion en cadena | Reiniciar<br>
                    Arrastrar: rotar | Scroll: zoom
                </div>
            </div>
        </div>
    </div>

    <script>
        let nucleidosActivos = [];

        const themeToggle = document.getElementById('themeToggle');
        
        function toggleTheme() {
            if (themeToggle.checked) {
                document.body.classList.add('dark');
                localStorage.setItem('theme', 'dark');
            } else {
                document.body.classList.remove('dark');
                localStorage.setItem('theme', 'light');
            }
            actualizarGraficaElastica();
            simularMonteCarlo();
        }

        themeToggle.addEventListener('change', toggleTheme);

        const savedTheme = localStorage.getItem('theme');
        if (savedTheme === 'dark') {
            document.body.classList.add('dark');
            themeToggle.checked = true;
        }

        function showTab(tabId) {
            document.querySelectorAll('.tab-content').forEach(tab => {
                tab.classList.remove('active');
            });
            document.getElementById(tabId).classList.add('active');
            
            document.querySelectorAll('.nav-btn').forEach(btn => {
                btn.classList.remove('active');
            });
            event.target.classList.add('active');
            
            if (tabId === 'montecarlo') {
                simularMonteCarlo();
            }
        }

        function energiaFinalElastica(E, A, theta) {
            const alpha = Math.pow((A - 1) / (A + 1), 2);
            const thetaRad = theta * Math.PI / 180;
            return 0.5 * E * ((1 + alpha) + (1 - alpha) * Math.cos(thetaRad));
        }

        function agregarNucleido() {
            const activeBtn = document.querySelector('.nucleido-btn.active');
            if (!activeBtn) {
                alert('Seleccione un nucleido');
                return;
            }
            
            const A = parseFloat(activeBtn.dataset.a);
            const name = activeBtn.dataset.name;
            const E = parseFloat(document.getElementById('energia_inicial_elastica').value);
            
            if (nucleidosActivos.some(n => n.A === A)) {
                alert(name + ' ya esta en la grafica');
                return;
            }
            
            nucleidosActivos.push({ A, name, E });
            actualizarListaNucleidos();
            actualizarGraficaElastica();
        }

        function actualizarListaNucleidos() {
            const container = document.getElementById('nucleidos_activos');
            if (nucleidosActivos.length === 0) {
                container.innerHTML = 'Activos: Ninguno';
                return;
            }
            container.innerHTML = 'Activos: ' + nucleidosActivos.map(n => n.name).join(', ');
        }

        function limpiarGrafica() {
            nucleidosActivos = [];
            actualizarListaNucleidos();
            actualizarGraficaElastica();
        }

        function actualizarGraficaElastica() {
            const theta_vals = Array.from({length: 361}, (_, i) => i);
            
            const traces = nucleidosActivos.map((nucleido, idx) => {
                const energy_ratio = theta_vals.map(theta => {
                    const E_final = energiaFinalElastica(nucleido.E, nucleido.A, theta);
                    return E_final / nucleido.E;
                });
                
                const colors = ['#00ff9c', '#667eea', '#ff6600', '#ff44cc'];
                
                return {
                    x: theta_vals,
                    y: energy_ratio,
                    mode: 'lines',
                    name: nucleido.name + ' (A=' + nucleido.A + ')',
                    line: { color: colors[idx % colors.length], width: 2 }
                };
            });
            
            const isDark = document.body.classList.contains('dark');
            const fontColor = isDark ? '#fff' : '#2c3e4f';
            
            const layout = {
                template: 'plotly_dark',
                title: { text: 'Energia vs Angulo de Dispersion', font: { size: 10, color: fontColor } },
                xaxis: { title: "Angulo (grados)", range: [0, 360], dtick: 45, tickfont: { color: fontColor }, titlefont: { color: fontColor } },
                yaxis: { title: "E'/E0", range: [0, 1.05], tickfont: { color: fontColor }, titlefont: { color: fontColor } },
                plot_bgcolor: 'rgba(0,0,0,0)',
                paper_bgcolor: 'rgba(0,0,0,0)',
                legend: { font: { size: 8, color: fontColor } }
            };
            
            Plotly.newPlot('grafica_elastica', traces, layout, { responsive: true });
        }

        function monteCarlo(p, n0, pasos) {
            let neutrones = [n0];
            for (let step = 0; step < pasos; step++) {
                let nuevos = 0;
                for (let i = 0; i < neutrones[neutrones.length-1]; i++) {
                    if (Math.random() < p) {
                        nuevos += Math.floor(Math.random() * 3);
                    }
                }
                neutrones.push(nuevos);
            }
            return neutrones;
        }

        function simularMonteCarlo() {
            const p = parseFloat(document.getElementById('prob_p').value);
            const n0 = parseInt(document.getElementById('neutrones_n0').value);
            const pasos = parseInt(document.getElementById('generaciones').value);
            
            const datos = monteCarlo(p, n0, pasos);
            const k_efectivo = datos[datos.length-1] / datos[datos.length-2];
            
            let estado = "";
            if (k_efectivo < 0.95) estado = "Subcritico";
            else if (k_efectivo < 1.05) estado = "Critico";
            else estado = "Supercritico";
            
            document.getElementById('resultado_mc').innerHTML = 
                'Gen:' + datos.length + ' | Final:' + datos[datos.length-1] + ' | k=' + k_efectivo.toFixed(4) + '<br>' + estado;
            
            document.getElementById('statNeutrones').innerHTML = datos[datos.length-1];
            
            const isDark = document.body.classList.contains('dark');
            const fontColor = isDark ? '#fff' : '#2c3e4f';
            
            const trace = {
                y: datos,
                x: Array.from({length: datos.length}, (_, i) => i),
                mode: 'lines+markers',
                line: {color: '#00ff9c', width: 2},
                marker: {size: 2},
                fill: 'tozeroy',
                fillcolor: 'rgba(0, 255, 156, 0.1)'
            };
            
            const layout = {
                template: 'plotly_dark',
                title: { text: 'Monte Carlo', font: { size: 10, color: fontColor } },
                xaxis: { title: "Generacion", tickfont: { color: fontColor }, titlefont: { color: fontColor } },
                yaxis: { title: "Neutrones", type: 'log', tickfont: { color: fontColor }, titlefont: { color: fontColor } },
                plot_bgcolor: 'rgba(0,0,0,0)',
                paper_bgcolor: 'rgba(0,0,0,0)'
            };
            
            Plotly.newPlot('grafica_mc', [trace], layout);
            visualizarMonteCarlo3D(datos);
        }

        function visualizarMonteCarlo3D(datos) {
            const canvas = document.getElementById('mcCanvas');
            const ctx = canvas.getContext('2d');
            
            function draw() {
                canvas.width = canvas.clientWidth;
                canvas.height = canvas.clientHeight;
                const w = canvas.width;
                const h = canvas.height;
                
                const isDark = document.body.classList.contains('dark');
                const bgColor = isDark ? '#2a2a2a' : '#d0d0d0';
                
                ctx.fillStyle = bgColor;
                ctx.fillRect(0, 0, w, h);
                
                const maxVal = Math.max(...datos);
                const barWidth = w / datos.length * 0.6;
                const barSpacing = w / datos.length * 0.4;
                
                for (let i = 0; i < datos.length; i++) {
                    const barHeight = Math.min((datos[i] / maxVal) * (h - 40), h - 40);
                    const x = i * (barWidth + barSpacing) + barSpacing/2;
                    const y = h - barHeight - 20;
                    
                    ctx.fillStyle = '#00ff9c';
                    ctx.fillRect(x, y, barWidth, barHeight);
                    
                    ctx.fillStyle = isDark ? '#ccc' : '#333';
                    ctx.font = '8px "Nunito"';
                    ctx.textAlign = 'center';
                    if (datos[i] < 100000) ctx.fillText(datos[i], x + barWidth/2, y - 2);
                    ctx.fillStyle = isDark ? '#888' : '#666';
                    ctx.fillText(i, x + barWidth/2, h - 8);
                }
            }
            
            draw();
            window.addEventListener('resize', () => draw());
        }

        function abrirFision() {
            window.open('/fision', '_blank');
        }

        document.querySelectorAll('.nucleido-btn').forEach(btn => {
            btn.addEventListener('click', function() {
                document.querySelectorAll('.nucleido-btn').forEach(b => b.classList.remove('active'));
                this.classList.add('active');
            });
        });

        document.getElementById('energia_inicial_elastica').addEventListener('input', () => {
            actualizarGraficaElastica();
        });

        window.onload = () => {
            simularMonteCarlo();
            document.querySelector('.nucleido-btn').classList.add('active');
            MathJax.Hub.Queue(["Typeset", MathJax.Hub]);
        };
    </script>
</body>
</html>
"""

HTML_FISION = r"""<!DOCTYPE html>
<html>
<head>
<meta charset="UTF-8">
<title>Simulador de Fision Nuclear 3D</title>
<style>
    * {
        border: 0;
        box-sizing: border-box;
        margin: 0;
        padding: 0;
    }
    :root {
        --hue: 184;
        --bg: hsl(var(--hue), 10%, 90%);
        --fg: hsl(var(--hue), 66%, 24%);
        --primary: hsl(var(--hue), 66%, 44%);
        --gradient: linear-gradient(145deg, hsl(var(--hue), 10%, 85%), hsl(var(--hue), 10%, 100%));
    }
    body { 
        margin: 0; 
        overflow: hidden; 
        background: var(--bg);
        font-family: "Nunito", sans-serif;
        position: relative;
    }
    body.dark {
        --bg: hsl(var(--hue), 10%, 10%);
        --fg: hsl(var(--hue), 66%, 94%);
        --gradient: linear-gradient(145deg, hsl(var(--hue), 10%, 15%), hsl(var(--hue), 10%, 30%));
    }
    #app { 
        width: 100%; 
        height: 100vh; 
    }
    .back-btn {
        position: absolute;
        top: 20px;
        left: 20px;
        z-index: 20;
        background: var(--gradient);
        color: var(--fg);
        padding: 8px 16px;
        cursor: pointer;
        font-family: "Nunito", sans-serif;
        font-size: 14px;
        font-weight: 600;
        text-decoration: none;
        border-radius: 1.5em;
        box-shadow: -0.5em -0.5em 1.5em hsl(0, 0%, 100%), 0.5em 0.5em 1.5em hsl(var(--hue), 5%, 65%);
        border: 1px solid rgba(255, 255, 255, 0.3);
    }
    .back-btn:active {
        transform: scale(0.95);
    }
    .controls {
        position: absolute;
        top: 20px;
        left: 120px;
        z-index: 10;
        display: flex;
        gap: 10px;
    }
    button {
        padding: 8px 16px;
        background: var(--gradient);
        color: var(--fg);
        border: none;
        cursor: pointer;
        font-family: "Nunito", sans-serif;
        font-size: 13px;
        font-weight: 600;
        border-radius: 1.5em;
        box-shadow: -0.5em -0.5em 1.5em hsl(0, 0%, 100%), 0.5em 0.5em 1.5em hsl(var(--hue), 5%, 65%);
        border: 1px solid rgba(255, 255, 255, 0.3);
    }
    button:active {
        transform: scale(0.95);
    }
    .info {
        position: absolute;
        bottom: 20px;
        left: 20px;
        color: var(--fg);
        background: var(--gradient);
        padding: 12px 16px;
        font-family: "Nunito", sans-serif;
        font-size: 12px;
        z-index: 10;
        border-radius: 1.5em;
        max-width: 280px;
        border: 1px solid rgba(255, 255, 255, 0.3);
    }
    .info h3 {
        margin: 0 0 8px 0;
        color: var(--primary);
    }
    .stats {
        position: absolute;
        top: 20px;
        right: 20px;
        color: var(--fg);
        background: var(--gradient);
        padding: 12px 16px;
        font-family: "Nunito", sans-serif;
        font-size: 12px;
        z-index: 10;
        border-radius: 1.5em;
        min-width: 180px;
        text-align: right;
        border: 1px solid rgba(255, 255, 255, 0.3);
    }
    .stats h3 {
        margin: 0 0 8px 0;
        color: var(--primary);
        text-align: left;
    }
    .stats .value {
        color: hsl(3, 90%, 55%);
        font-size: 18px;
        font-weight: bold;
    }
    .warning {
        color: hsl(3, 90%, 55%);
        animation: pulse 1s infinite;
        margin-top: 8px;
    }
    @keyframes pulse {
        0%, 100% { opacity: 1; }
        50% { opacity: 0.5; }
    }
    
    /* Lightsaber Switch */
    .scene {
        position: absolute;
        top: 20px;
        right: 20px;
        display: flex;
        justify-content: center;
        align-items: center;
        z-index: 30;
    }
    .images {
        position: absolute;
        opacity: 0;
        pointer-events: none;
    }
    .switch {
        display: flex;
        justify-content: center;
        align-items: center;
        width: 80px;
        cursor: pointer;
    }
    .switch input {
        position: absolute;
        opacity: 0;
    }
    .track {
        position: relative;
        width: 72px;
        height: 24px;
        background: #2c2c2c;
        border-radius: 60px;
        transition: 0.3s;
        box-shadow: inset 0 0 4px rgba(0,0,0,0.2);
    }
    body.dark .track {
        background: #1a1a1a;
    }
    .thumb {
        position: absolute;
        top: -6px;
        left: -6px;
        width: 36px;
        height: 36px;
        transition: 0.3s;
        display: flex;
        justify-content: center;
        align-items: center;
    }
    input:checked + .track .thumb {
        left: calc(100% - 36px + 6px);
    }
    .lightsaber {
        position: absolute;
        top: 0;
        left: 0;
        right: 0;
        bottom: 0;
        display: flex;
        justify-content: space-between;
        align-items: center;
        padding: 0 4px;
    }
    .lightsaber .light {
        width: 20px;
        height: 3px;
        background: #00ff9c;
        border-radius: 4px;
        box-shadow: 0 0 4px #00ff9c;
    }
    .lightsaber .dark {
        width: 20px;
        height: 3px;
        background: #ff3366;
        border-radius: 4px;
        box-shadow: 0 0 4px #ff3366;
    }
    .lightsaber .grip {
        width: 22px;
        height: 10px;
        background: #555;
        border-radius: 4px;
    }
    .circle {
        width: 36px;
        height: 36px;
        background: #1a1a1a;
        border-radius: 100%;
        display: flex;
        justify-content: center;
        align-items: center;
        transition: 0.2s;
        box-shadow: 0 2px 6px rgba(0,0,0,0.2);
    }
    body.dark .circle {
        background: #2a2a2a;
    }
    .sub-circle {
        width: 28px;
        height: 28px;
        background: #2c2c2c;
        border-radius: 100%;
    }
    body.dark .sub-circle {
        background: #3a3a3a;
    }
    .side {
        position: absolute;
        top: 0;
        left: 0;
        right: 0;
        bottom: 0;
        display: flex;
        justify-content: center;
        align-items: center;
        transition: 0.2s;
        transform: scale(0);
    }
    input:checked + .track .thumb .dark-side {
        transform: scale(1);
    }
    input:not(:checked) + .track .thumb .light-side {
        transform: scale(1);
    }
    .light-side .top {
        position: absolute;
        top: 4px;
        display: flex;
        gap: 4px;
    }
    .light-side .top .left,
    .light-side .top .right {
        width: 3px;
        height: 3px;
        background: #ffcc00;
        border-radius: 100%;
    }
    .light-side .center {
        display: flex;
        gap: 2px;
    }
    .light-side .center div {
        width: 2px;
        height: 10px;
        background: #ffcc00;
        border-radius: 2px;
    }
    .light-side .bottom {
        position: absolute;
        bottom: 4px;
    }
    .light-side .bottom .line {
        width: 12px;
        height: 2px;
        background: #ffcc00;
        border-radius: 2px;
    }
    .dark-side .circle {
        background: #ff3366;
    }
    .dark-side .sub-circle {
        background: #cc0033;
    }
</style>
</head>
<body>
    <div class="scene">
        <div class="images">
            <img src="https://assets.codepen.io/907471/darkside.svg?v42" class="dark">
            <img src="https://assets.codepen.io/907471/lightside.svg?v42" class="light">
        </div>
        <label class="switch" id="switch" aria-label="Switch">
            <input type="checkbox" id="themeToggleFision">
            <div class="track">
                <div class="lightsaber">
                    <div class="light"></div>
                    <div class="grip"></div>
                    <div class="dark"></div>
                </div>
                <div class="thumb">
                    <div class="side dark-side">
                        <div class="circle">
                            <div class="sub-circle"></div>
                        </div>
                    </div>
                    <div class="side light-side">
                        <div class="circle">
                            <div class="sub-circle"></div>
                        </div>
                        <div class="top">
                            <div class="left"></div>
                            <div class="right"></div>
                        </div>
                        <div class="center">
                            <div class="item-1"></div>
                            <div class="item-2"></div>
                            <div class="item-3"></div>
                            <div class="item-4"></div>
                            <div class="item-5"></div>
                        </div>
                        <div class="bottom">
                            <div class="line"></div>
                        </div>
                    </div>
                </div>
            </div>
        </label>
    </div>
    
    <a href="/" class="back-btn">Volver</a>
    <div class="controls">
        <button id="fireBtn">Disparar</button>
        <button id="resetBtn">Reiniciar</button>
        <button id="chainBtn">Cadena</button>
    </div>
    <div class="stats">
        <h3>Reactor</h3>
        <p>Uranio: <span id="atomCount" class="value">0</span></p>
        <p>Neutrones: <span id="neutronCount" class="value">0</span></p>
        <p>Fisiones: <span id="fissionCount" class="value">0</span></p>
        <p>Energia: <span id="energyReleased" class="value">0</span> MeV</p>
        <div id="chainWarning" style="display:none;" class="warning">CADENA ACTIVA</div>
    </div>
    <div class="info">
        <h3>Fision Nuclear</h3>
        <p>n + U-235 -> Ba-141 + Kr-92 + 2-3 n</p>
        <p>Energia: 200 MeV por fision</p>
        <p>150 atomos | Arrastrar: rotar | Scroll: zoom</p>
    </div>
    <div id="app"></div>
    <script>
        const themeToggleFision = document.getElementById('themeToggleFision');
        
        function toggleThemeFision() {
            if (themeToggleFision.checked) {
                document.body.classList.add('dark');
                localStorage.setItem('themeFision', 'dark');
            } else {
                document.body.classList.remove('dark');
                localStorage.setItem('themeFision', 'light');
            }
        }
        
        themeToggleFision.addEventListener('change', toggleThemeFision);
        
        const savedThemeFision = localStorage.getItem('themeFision');
        if (savedThemeFision === 'dark') {
            document.body.classList.add('dark');
            themeToggleFision.checked = true;
        }
    </script>
<script type="importmap">
{
  "imports": {
    "three": "https://esm.sh/three",
    "three/addons/": "https://esm.sh/three/addons/"
  }
}
</script>
<script type="module">
import * as THREE from "three";
import { OrbitControls } from "three/addons/controls/OrbitControls.js";

let scene, camera, renderer, controls;
let atoms = [];
let neutrons = [];
let fragments = [];
let energyParticles = [];
let fissionCount = 0;
let totalEnergy = 0;
let chainReactionActive = false;
let chainReactionInterval = null;

const ENERGY_PER_FISSION = 200;
const NEUTRON_SPEED = 0.35;
const FISSION_RADIUS = 1.3;

function init() {
    scene = new THREE.Scene();
    scene.background = new THREE.Color(0x101018);
    scene.fog = new THREE.FogExp2(0x101018, 0.0005);

    camera = new THREE.PerspectiveCamera(60, innerWidth/innerHeight, 0.1, 1000);
    camera.position.set(25, 20, 32);

    renderer = new THREE.WebGLRenderer({antialias: true});
    renderer.setSize(innerWidth, innerHeight);
    renderer.shadowMap.enabled = true;
    document.getElementById("app").appendChild(renderer.domElement);

    controls = new OrbitControls(camera, renderer.domElement);
    controls.enableDamping = true;
    controls.enableZoom = true;
    controls.zoomSpeed = 1.2;
    controls.target.set(0, 0, 0);

    const ambientLight = new THREE.AmbientLight(0x222222);
    scene.add(ambientLight);
    const directionalLight = new THREE.DirectionalLight(0xffffff, 0.6);
    directionalLight.position.set(5, 10, 7);
    directionalLight.castShadow = true;
    scene.add(directionalLight);
    const backLight = new THREE.PointLight(0x4466ff, 0.3);
    backLight.position.set(-5, 0, -5);
    scene.add(backLight);
    const fillLight = new THREE.PointLight(0xffaa66, 0.25);
    fillLight.position.set(3, 5, 4);
    scene.add(fillLight);
    const rimLight = new THREE.PointLight(0xff66aa, 0.2);
    rimLight.position.set(-3, 4, -4);
    scene.add(rimLight);

    createInitialAtoms();
    animate();
    setInterval(updateUI, 100);
}

const tex = new THREE.TextureLoader().load("https://threejs.org/examples/textures/sprites/disc.png");

const createShell = (count, radius, color) => {
    const pos = [];
    const col = [];
    const pointCount = Math.min(count * 6, 120);
    for (let i = 0; i < pointCount; i++){
        const r = radius + (Math.random() - 0.5) * 0.3;
        const theta = Math.acos(2 * Math.random() - 1);
        const phi = Math.random() * Math.PI * 2;
        const x = r * Math.sin(theta) * Math.cos(phi);
        const y = r * Math.sin(theta) * Math.sin(phi);
        const z = r * Math.cos(theta);
        pos.push(x, y, z);
        const c = new THREE.Color(color);
        const fade = 0.4 + Math.random() * 0.6;
        col.push(c.r * fade, c.g * fade, c.b * fade);
    }
    const geo = new THREE.BufferGeometry();
    geo.setAttribute("position", new THREE.Float32BufferAttribute(pos, 3));
    geo.setAttribute("color", new THREE.Float32BufferAttribute(col, 3));
    const mat = new THREE.PointsMaterial({ size: 0.07, map: tex, transparent: true, vertexColors: true, blending: THREE.AdditiveBlending, depthWrite: false });
    return new THREE.Points(geo, mat);
};

function createAtom(position, color = 0xff4466) {
    const group = new THREE.Group();
    const nucleus = new THREE.Mesh(new THREE.SphereGeometry(0.55, 32, 32), new THREE.MeshStandardMaterial({ color: color, emissive: 0x331100, emissiveIntensity: 0.15, metalness: 0.7, roughness: 0.3 }));
    nucleus.castShadow = true;
    group.add(nucleus);
    const shells = [{n:2,r:1.0,color:0xff6688},{n:8,r:1.6,color:0xff7799},{n:18,r:2.3,color:0xff88aa},{n:32,r:3.0,color:0xff99bb},{n:21,r:3.8,color:0xffaacc},{n:9,r:4.7,color:0xffbbdd},{n:2,r:5.6,color:0xffccee}];
    shells.forEach(s => { const c = createShell(s.n, s.r, s.color); group.add(c); });
    group.userData = { clouds: shells.map((_,i)=>group.children[i+1]), type: "uranium" };
    group.position.copy(position);
    scene.add(group);
    return group;
}

function createInitialAtoms() {
    atoms.forEach(atom => scene.remove(atom));
    atoms = [];
    const positions = [];
    const radius = 14;
    for(let i = 0; i < 150; i++) {
        let x, y, z, d;
        do { x = (Math.random() - 0.5) * radius * 2; y = (Math.random() - 0.5) * radius * 1.5; z = (Math.random() - 0.5) * radius * 2; d = Math.sqrt(x*x + y*y + z*z); } while(d > radius);
        positions.push(new THREE.Vector3(x, y, z));
    }
    positions.forEach(pos => atoms.push(createAtom(pos)));
    fissionCount = 0;
    totalEnergy = 0;
    updateUI();
}

function createNeutron(position, direction, speed = NEUTRON_SPEED) {
    const m = new THREE.Mesh(new THREE.SphereGeometry(0.18, 16, 16), new THREE.MeshStandardMaterial({ color: 0xffdd99, emissive: 0xff8844, emissiveIntensity: 0.8 }));
    m.position.copy(position);
    m.castShadow = true;
    m.userData = { velocity: direction.clone().normalize().multiplyScalar(speed) };
    scene.add(m);
    neutrons.push(m);
    updateUI();
}

function fission(atom, neutronPosition) {
    const atomPos = atom.position.clone();
    scene.remove(atom);
    const idx = atoms.indexOf(atom);
    if(idx !== -1) atoms.splice(idx, 1);
    fissionCount++;
    totalEnergy += ENERGY_PER_FISSION;
    updateUI();
    if(chainReactionActive && atoms.length > 0) document.getElementById("chainWarning").style.display = "block";
    
    const bariumGroup = new THREE.Group();
    const baNuc = new THREE.Mesh(new THREE.SphereGeometry(0.45, 24, 24), new THREE.MeshStandardMaterial({ color: 0x66cc99, emissive: 0x226644, emissiveIntensity: 0.25 }));
    bariumGroup.add(baNuc);
    [{n:2,r:0.8,c:0x88ffaa},{n:8,r:1.25,c:0xaaffcc},{n:18,r:1.75,c:0xccffdd},{n:18,r:2.3,c:0xddffee},{n:8,r:2.9,c:0xeeffff},{n:2,r:3.5,c:0xaaffff}].forEach(s => bariumGroup.add(createShell(s.n, s.r, s.c)));
    
    const kryptonGroup = new THREE.Group();
    const krNuc = new THREE.Mesh(new THREE.SphereGeometry(0.35, 24, 24), new THREE.MeshStandardMaterial({ color: 0xff9966, emissive: 0x442200, emissiveIntensity: 0.25 }));
    kryptonGroup.add(krNuc);
    [{n:2,r:0.7,c:0xffaa88},{n:8,r:1.1,c:0xffbb99},{n:18,r:1.6,c:0xffccaa},{n:8,r:2.1,c:0xffddbb}].forEach(s => kryptonGroup.add(createShell(s.n, s.r, s.c)));
    
    const dir = neutronPosition.clone().sub(atomPos).normalize();
    bariumGroup.position.copy(atomPos.clone().add(dir.clone().multiplyScalar(1.5)));
    kryptonGroup.position.copy(atomPos.clone().add(dir.clone().multiplyScalar(-1.5)));
    bariumGroup.userData = { velocity: dir.clone().multiplyScalar(0.06) };
    kryptonGroup.userData = { velocity: dir.clone().multiplyScalar(-0.06) };
    scene.add(bariumGroup);
    scene.add(kryptonGroup);
    fragments.push(bariumGroup, kryptonGroup);
    
    const nCount = 2 + Math.floor(Math.random() * 2);
    for(let i = 0; i < nCount; i++) {
        const rd = new THREE.Vector3((Math.random()-0.5)*2, (Math.random()-0.5)*2, (Math.random()-0.5)*2).normalize();
        createNeutron(atomPos, rd, NEUTRON_SPEED * (chainReactionActive ? 1.15 : 1.0));
    }
    
    for(let i = 0; i < 60; i++) {
        const p = new THREE.Mesh(new THREE.SphereGeometry(0.06, 6, 6), new THREE.MeshBasicMaterial({ color: [0xffaa44,0xff6644,0xffaa88][Math.floor(Math.random()*3)], transparent: true, blending: THREE.AdditiveBlending }));
        p.position.copy(atomPos);
        const a1 = Math.random() * Math.PI * 2, a2 = Math.random() * Math.PI * 2, sp = 0.12 + Math.random() * 0.18;
        p.userData = { velocity: new THREE.Vector3(Math.sin(a1)*Math.cos(a2)*sp, Math.sin(a1)*Math.sin(a2)*sp, Math.cos(a1)*sp), life: 1.0, decay: 0.012 };
        scene.add(p);
        energyParticles.push(p);
    }
    
    const explLight = new THREE.PointLight(0xff8844, 2.0, 20);
    explLight.position.copy(atomPos);
    scene.add(explLight);
    let int = 2.0;
    const li = setInterval(() => { int -= 0.25; explLight.intensity = int; if(int <= 0) { clearInterval(li); scene.remove(explLight); } }, 35);
}

function startChainReaction() {
    if(chainReactionActive) return;
    chainReactionActive = true;
    document.getElementById("chainWarning").style.display = "block";
    if(atoms.length > 0 && neutrons.length === 0) {
        const start = new THREE.Vector3(0, 5, 25);
        const target = atoms[Math.floor(Math.random() * atoms.length)].position;
        createNeutron(start, new THREE.Vector3().subVectors(target, start), NEUTRON_SPEED * 1.2);
    }
    chainReactionInterval = setInterval(() => {
        if(chainReactionActive && atoms.length > 0 && neutrons.length < 10) {
            const ra = atoms[Math.floor(Math.random() * atoms.length)];
            const start = new THREE.Vector3(ra.position.x + (Math.random()-0.5)*8, ra.position.y + (Math.random()-0.5)*6 + 6, ra.position.z + (Math.random()-0.5)*8 + 18);
            createNeutron(start, new THREE.Vector3().subVectors(ra.position, start), NEUTRON_SPEED * 1.25);
        }
        if(atoms.length === 0 && chainReactionActive) stopChainReaction();
    }, 1000);
}

function stopChainReaction() {
    chainReactionActive = false;
    document.getElementById("chainWarning").style.display = "none";
    if(chainReactionInterval) { clearInterval(chainReactionInterval); chainReactionInterval = null; }
}

function resetSimulation() {
    stopChainReaction();
    neutrons.forEach(n => scene.remove(n));
    fragments.forEach(f => scene.remove(f));
    energyParticles.forEach(p => scene.remove(p));
    atoms.forEach(a => scene.remove(a));
    neutrons = []; fragments = []; energyParticles = [];
    createInitialAtoms();
    updateUI();
}

function updateUI() {
    document.getElementById("atomCount").innerHTML = atoms.length;
    document.getElementById("neutronCount").innerHTML = neutrons.length;
    document.getElementById("fissionCount").innerHTML = fissionCount;
    document.getElementById("energyReleased").innerHTML = totalEnergy.toFixed(0);
    if(atoms.length === 0) document.getElementById("chainWarning").innerHTML = "COMPLETADA";
}

function animate() {
    requestAnimationFrame(animate);
    atoms.forEach(atom => { if(atom.userData && atom.userData.clouds) { atom.userData.clouds.forEach((c, i) => { c.rotation.y += 0.0008 * (i+1); c.rotation.x += 0.0005 * (i+1); }); } });
    for(let i=fragments.length-1; i>=0; i--) { const f = fragments[i]; if(f.userData && f.userData.velocity) { f.position.add(f.userData.velocity); f.children.forEach(c => { if(c.isPoints) { c.rotation.y += 0.015; c.rotation.x += 0.01; } }); } if(Math.abs(f.position.x) > 35 || Math.abs(f.position.y) > 25 || Math.abs(f.position.z) > 35) { scene.remove(f); fragments.splice(i,1); } }
    for(let i=energyParticles.length-1; i>=0; i--) { const p = energyParticles[i]; p.position.add(p.userData.velocity); p.userData.life -= p.userData.decay; const s = p.userData.life * 0.8; p.scale.set(s,s,s); p.material.opacity = p.userData.life; if(p.userData.life <= 0) { scene.remove(p); energyParticles.splice(i,1); } }
    for(let i=neutrons.length-1; i>=0; i--) { const n = neutrons[i]; n.position.add(n.userData.velocity); n.material.emissiveIntensity = 0.5 + Math.sin(Date.now()*0.02)*0.5; let hit = false; for(let j=0; j<atoms.length; j++) { const a = atoms[j]; if(a && n.position.distanceTo(a.position) < FISSION_RADIUS) { fission(a, n.position); scene.remove(n); neutrons.splice(i,1); hit = true; break; } } if(!hit && (Math.abs(n.position.x) > 45 || Math.abs(n.position.y) > 35 || Math.abs(n.position.z) > 45)) { scene.remove(n); neutrons.splice(i,1); } }
    controls.update();
    renderer.render(scene, camera);
}

document.getElementById("fireBtn").onclick = () => { if(atoms.length === 0) { alert("No quedan atomos. Reinicie."); return; } const start = new THREE.Vector3(0, 5, 28); const target = atoms[Math.floor(Math.random() * atoms.length)].position; createNeutron(start, new THREE.Vector3().subVectors(target, start)); };
document.getElementById("resetBtn").onclick = resetSimulation;
document.getElementById("chainBtn").onclick = startChainReaction;
window.addEventListener("resize", () => { camera.aspect = innerWidth / innerHeight; camera.updateProjectionMatrix(); renderer.setSize(innerWidth, innerHeight); });
init();
</script>
</body>
</html>
"""

class Handler(BaseHTTPRequestHandler):
    def do_GET(self):
        if self.path == '/':
            self.send_response(200)
            self.send_header('Content-type', 'text/html; charset=utf-8')
            self.end_headers()
            self.wfile.write(HTML_PRINCIPAL.encode('utf-8'))
        elif self.path == '/fision':
            self.send_response(200)
            self.send_header('Content-type', 'text/html; charset=utf-8')
            self.end_headers()
            self.wfile.write(HTML_FISION.encode('utf-8'))
        else:
            self.send_response(404)
            self.end_headers()
    
    def log_message(self, format, *args):
        pass

def open_browser():
    webbrowser.open('http://localhost:8080')

def main():
    port = 8080
    server = HTTPServer(('localhost', port), Handler)
    print("=" * 50)
    print(" LABORATORIO NUCLEAR")
    print("=" * 50)
    print(f" Servidor activo en: http://localhost:{port}")
    print("=" * 50)
    print(" Presiona Ctrl+C para detener")
    print("=" * 50)
    
    Timer(1, open_browser).start()
    
    try:
        server.serve_forever()
    except KeyboardInterrupt:
        print("\n Cerrando servidor...")
        server.shutdown()
        print(" Servidor detenido")

if __name__ == '__main__':
    main()

 LABORATORIO NUCLEAR
 Servidor activo en: http://localhost:8080
 Presiona Ctrl+C para detener

 Cerrando servidor...
 Servidor detenido


In [3]:
import webbrowser
import math
import random
from http.server import HTTPServer, BaseHTTPRequestHandler
from threading import Timer

def energia_final(E, A, theta):
    theta_rad = math.radians(theta)
    return E * ((A*A + 1 + 2*A*math.cos(theta_rad)) / ((A+1)**2))

def monte_carlo(p, n0, pasos=50):
    neutrones = [n0]
    for _ in range(pasos):
        nuevos = sum(
            random.randint(0, 2) if random.random() < p else 0
            for _ in range(neutrones[-1])
        )
        neutrones.append(nuevos)
    return neutrones

HTML_PRINCIPAL = """
<!DOCTYPE html>
<html lang="es">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Laboratorio Nuclear</title>
    <link href="https://fonts.googleapis.com/css2?family=Nunito:wght@300;400;600;700&display=swap" rel="stylesheet">
    <script src="https://cdn.plot.ly/plotly-3.1.0.min.js"></script>
    <script src="https://cdnjs.cloudflare.com/ajax/libs/mathjax/2.7.7/MathJax.js?config=TeX-MML-AM_CHTML"></script>
    <style>
        * {
            border: 0;
            box-sizing: border-box;
            margin: 0;
            padding: 0;
        }
        :root {
            --hue: 184;
            --bg: hsl(var(--hue), 10%, 90%);
            --fg: hsl(var(--hue), 66%, 24%);
            --primary: hsl(var(--hue), 66%, 44%);
            --gradient: linear-gradient(145deg, hsl(var(--hue), 10%, 85%), hsl(var(--hue), 10%, 100%));
            font-size: 16px;
        }
        body,
        button {
            color: var(--fg);
            font: 1em/1.5 "Nunito", sans-serif;
        }
        body {
            background: var(--bg);
            height: 100vh;
            display: grid;
            place-items: center;
            padding: 1.5em 0 0 0;
            position: relative;
        }
        body:after {
            content: "";
            display: block;
            height: 1.5em;
            width: 100%;
        }
        .app {
            background: hsl(var(--hue), 10%, 85%);
            border-radius: 3em;
            flex-direction: column;
            padding: 2.25em;
            width: 24.375em;
            height: auto;
            min-height: 52.75em;
            max-height: 90vh;
            overflow-y: auto;
            box-shadow: 0 8px 32px rgba(0, 0, 0, 0.1);
            border: 1px solid rgba(255, 255, 255, 0.5);
        }
        .app__gradients {
            position: absolute;
            width: 1px;
            height: 1px;
        }
        .icon {
            display: block;
            margin: auto;
            width: 1.5em;
            height: 1.5em;
        }
        .icon circle,
        .icon path {
            fill: currentColor;
            transition: fill 0.15s linear;
        }
        .icon ellipse,
        .icon polygon {
            stroke: currentColor;
            transition: stroke 0.15s linear;
        }
        .icon .no-fill {
            fill: none;
            stroke: currentColor;
        }
        .icon--red path {
            fill: hsl(3, 90%, 55%);
        }
        .icon--pulse {
            animation:
                bpm 1s linear,
                pulse 0.75s 1s linear infinite;
        }
        .ring,
        .sr-only {
            position: absolute;
        }
        .ring {
            display: block;
            inset: 0;
            width: 100%;
            height: auto;
        }
        .ring-fill,
        .ring-stroke {
            stroke: url("#ring");
        }
        .ring-stroke {
            animation-duration: 1s;
            animation-timing-function: ease-in-out;
        }
        .ring-track {
            stroke: hsl(var(--hue), 10%, 80%);
        }
        .sr-only {
            clip: rect(1px, 1px, 1px, 1px);
            overflow: hidden;
            width: 1px;
            height: 1px;
        }
        .header {
            display: flex;
            justify-content: space-between;
            align-items: center;
            margin-bottom: 1.5em;
        }
        .header__profile-btn,
        .header__notes-btn {
            width: 3em;
            height: 3em;
        }
        .header__profile-btn {
            border-radius: 1em;
            box-shadow: 0 0 0 0.125em inset;
        }
        .header__notes-btn {
            margin-inline-end: -1em;
        }
        .header__profile-btn:active,
        .header__notes-btn:active {
            transform: scale(0.9);
        }
        .header__profile-btn:focus {
            box-shadow: 0 0 0 0.125em var(--primary) inset;
        }
        .header__profile-icon {
            border-radius: 0.5em;
            margin: auto;
            width: 2em;
            height: 2em;
        }
        .header__notes-btn:focus .icon path {
            fill: var(--primary);
        }
        .header h1 {
            font-size: 1rem;
            text-align: center;
            flex: 1;
        }
        .main__date-nav {
            margin-bottom: 1.5em;
        }
        .main__date-arrow-btn,
        .main__date-edit-btn {
            height: 1.5em;
        }
        .main__date-arrow-btn {
            width: 1.5em;
        }
        .main__date-arrow-btn:active .icon path,
        .main__date-arrow-btn:focus .icon path {
            fill: var(--primary);
        }
        .main__date {
            text-transform: uppercase;
        }
        .main__date-edit-btn {
            min-width: 1.5em;
        }
        .main__date-edit-btn:active,
        .main__date-edit-btn:focus {
            color: var(--primary);
        }
        .main__stat-blocks {
            display: grid;
            grid-template-columns: repeat(3, 1fr);
            grid-gap: 1.5em;
            margin-bottom: 1.5em;
        }
        .main__stat-block {
            background: var(--gradient);
            border-radius: 1.5em;
            box-shadow:
                -0.75em -0.75em 2.25em hsl(0, 0%, 100%),
                0.75em 0.75em 2.25em hsl(var(--hue), 5%, 65%);
            padding: 0.75em;
            text-align: center;
            width: 100%;
        }
        .main__stat-block--lg {
            grid-column: 1 / 4;
            padding: 1.5em;
        }
        .main__stat-rows,
        .main__stat-row {
            margin-bottom: 1.5em;
        }
        .main__stat-row {
            align-items: center;
        }
        .main__stat-graph {
            margin: 0 auto 0.75em auto;
            position: relative;
            width: 3.75em;
            height: 3.75em;
        }
        .main__stat-graph .main__stat-detail {
            display: flex;
            flex-direction: column;
            justify-content: center;
            position: absolute;
            inset: 0;
        }
        .main__stat-block--lg .main__stat-graph {
            margin: auto;
            width: 11.25em;
            height: 11.25em;
        }
        .main__stat-block--lg .icon {
            margin: 0 auto;
            width: 2.25em;
            height: 2.25em;
        }
        .main__stat-row .main__stat-graph {
            background: var(--gradient);
            border-radius: 1em;
            box-shadow:
                -0.75em -0.75em 2.25em hsl(0, 0%, 100%),
                0.75em 0.75em 2.25em hsl(var(--hue), 5%, 65%);
            margin: 0;
            margin-inline-end: 1.5em;
        }
        .main__stat-value,
        .main__stat-unit {
            display: block;
        }
        .main__stat-value {
            font-size: 1.25em;
            line-height: 1.2;
        }
        .main__stat-block--lg .main__stat-value {
            font-size: 2em;
            line-height: 1.5;
        }
        .main__stat-unit,
        .main__stat-subtext {
            font-weight: 300;
        }
        .main__stat-subtext {
            color: hsl(var(--hue), 10%, 30%);
        }
        .footer {
            margin-top: auto;
        }
        .footer__nav-btn {
            background: var(--gradient);
            border-radius: 50%;
            box-shadow:
                1em 1em 2em hsl(var(--hue), 5%, 65%),
                -1em -1em 2em hsl(0, 0%, 100%);
            width: 3em;
            height: 3em;
        }
        .footer__nav-btn:active {
            box-shadow:
                0.75em 0.75em 1.5em hsl(var(--hue), 5%, 65%),
                -0.75em -0.75em 1.5em hsl(0, 0%, 100%);
            transform: scale(0.9);
        }
        .footer__nav-btn:focus .icon circle,
        .footer__nav-btn:focus .icon path {
            fill: var(--primary);
        }
        body.dark {
            --bg: hsl(var(--hue), 10%, 10%);
            --fg: hsl(var(--hue), 66%, 94%);
            --primary: hsl(var(--hue), 66%, 44%);
            --gradient: linear-gradient(145deg, hsl(var(--hue), 10%, 15%), hsl(var(--hue), 10%, 30%));
        }
        body.dark .app {
            background: hsl(var(--hue), 10%, 20%);
        }
        body.dark .icon--red path {
            fill: hsl(3, 90%, 65%);
        }
        body.dark .ring-track {
            stroke: hsl(var(--hue), 10%, 30%);
        }
        body.dark .main__stat-block,
        body.dark .main__stat-row .main__stat-graph {
            box-shadow:
                -0.75em -0.75em 2.25em hsl(var(--hue), 10%, 30%),
                0.75em 0.75em 2.25em hsl(var(--hue), 5%, 5%);
        }
        body.dark .main__stat-subtext {
            color: hsl(var(--hue), 10%, 70%);
        }
        body.dark .footer__nav-btn {
            box-shadow:
                -1em -1em 2em hsl(var(--hue), 10%, 30%),
                1em 1em 2em hsl(var(--hue), 5%, 5%);
        }
        body.dark .footer__nav-btn:active {
            box-shadow:
                -0.75em -0.75em 1.5em hsl(var(--hue), 10%, 30%),
                0.75em 0.75em 1.5em hsl(var(--hue), 5%, 5%);
        }
        @keyframes statFill {
            from { color: var(--fg); }
            to { color: hsl(var(--hue), 66%, 94%); }
        }
        @keyframes ringFill {
            from { r: 82px; stroke-width: 16; }
            to { r: 45px; stroke-width: 90; }
        }
        @keyframes stepCount {
            from { stroke-dashoffset: 515.22; }
            to { stroke-dashoffset: 0; }
        }
        @keyframes cals {
            from { stroke-dashoffset: 163.36; }
            to { stroke-dashoffset: 12.25; }
        }
        @keyframes miles {
            from { stroke-dashoffset: 163.36; }
            to { stroke-dashoffset: 35.39; }
        }
        @keyframes mins {
            from { stroke-dashoffset: 163.36; }
            to { stroke-dashoffset: 65.34; }
        }
        @keyframes bpm {
            from { transform: scale(0); }
            37.5% { transform: scale(1.2); }
            75%, to { transform: scale(1); }
        }
        @keyframes stepHrs {
            from { stroke-dashoffset: 131.95; }
            to { stroke-dashoffset: 52.78; }
        }
        @keyframes pulse {
            from, 75%, to { transform: scale(1); }
            25% { transform: scale(0.9); }
            50% { transform: scale(1.2); }
        }
        .nav-tabs {
            display: flex;
            gap: 0.5em;
            margin-bottom: 1.5em;
            justify-content: center;
        }
        .nav-btn {
            background: var(--gradient);
            border-radius: 1.5em;
            padding: 0.5em 1em;
            font-size: 0.7rem;
            font-weight: 600;
            cursor: pointer;
            box-shadow: -0.5em -0.5em 1.5em hsl(0, 0%, 100%), 0.5em 0.5em 1.5em hsl(var(--hue), 5%, 65%);
        }
        .nav-btn.active {
            background: var(--primary);
            color: white;
        }
        .tab-content {
            display: none;
        }
        .tab-content.active {
            display: block;
        }
        .card {
            background: var(--gradient);
            border-radius: 1.5em;
            padding: 1em;
            margin-bottom: 1em;
            box-shadow: -0.75em -0.75em 2.25em hsl(0, 0%, 100%), 0.75em 0.75em 2.25em hsl(var(--hue), 5%, 65%);
        }
        .card-title {
            font-weight: 700;
            margin-bottom: 0.5em;
            color: var(--primary);
        }
        .input-group {
            margin-bottom: 0.8em;
        }
        .input-group label {
            display: block;
            font-size: 0.7rem;
            font-weight: 600;
            margin-bottom: 0.2em;
        }
        .input-group input, .input-group select {
            width: 100%;
            padding: 0.4em 0.6em;
            background: hsl(var(--hue), 10%, 85%);
            border-radius: 1em;
            font-family: "Nunito", sans-serif;
            font-size: 0.8rem;
            border: 1px solid rgba(255, 255, 255, 0.3);
        }
        button {
            width: 100%;
            padding: 0.5em;
            background: var(--gradient);
            border-radius: 1.5em;
            font-family: "Nunito", sans-serif;
            font-weight: 600;
            cursor: pointer;
            margin-top: 0.3em;
            box-shadow: -0.5em -0.5em 1.5em hsl(0, 0%, 100%), 0.5em 0.5em 1.5em hsl(var(--hue), 5%, 65%);
            border: 1px solid rgba(255, 255, 255, 0.3);
        }
        button:active {
            transform: scale(0.95);
        }
        .btn-primary {
            background: var(--primary);
            color: white;
        }
        .resultado {
            margin-top: 0.5em;
            padding: 0.5em;
            background: hsl(var(--hue), 10%, 80%);
            border-radius: 1em;
            text-align: center;
            font-size: 0.7rem;
        }
        .graph-container {
            background: var(--gradient);
            border-radius: 1.5em;
            padding: 1em;
            margin-bottom: 1em;
            box-shadow: -0.75em -0.75em 2.25em hsl(0, 0%, 100%), 0.75em 0.75em 2.25em hsl(var(--hue), 5%, 65%);
        }
        .graph-title {
            font-size: 0.7rem;
            font-weight: 700;
            margin-bottom: 0.5em;
            text-align: center;
        }
        .simulation-canvas {
            width: 100%;
            height: 200px;
            background: hsl(var(--hue), 10%, 85%);
            border-radius: 1em;
        }
        .info-text {
            background: hsl(var(--hue), 10%, 80%);
            padding: 0.8em;
            border-radius: 1em;
            margin-bottom: 1em;
            font-size: 0.7rem;
            border-left: 3px solid var(--primary);
        }
        .formula {
            background: hsl(var(--hue), 10%, 80%);
            padding: 0.8em;
            border-radius: 1em;
            margin: 0.8em 0;
            font-size: 0.7rem;
            overflow-x: auto;
        }
        .nucleido-selector {
            display: flex;
            gap: 0.5em;
            margin-bottom: 0.8em;
            flex-wrap: wrap;
        }
        .nucleido-btn {
            flex: 1;
            padding: 0.3em;
            font-size: 0.7rem;
            margin: 0;
            background: hsl(var(--hue), 10%, 85%);
        }
        .nucleido-btn.active {
            background: var(--primary);
            color: white;
        }
        .flex-buttons {
            display: flex;
            gap: 0.5em;
        }
        .flex-buttons button {
            flex: 1;
        }
        .scene {
            display: flex;
            justify-content: center;
            align-items: center;
        }
        .images {
            position: absolute;
            opacity: 0;
            pointer-events: none;
        }
        .switch {
            display: flex;
            justify-content: center;
            align-items: center;
            width: 80px;
            cursor: pointer;
        }
        .switch input {
            position: absolute;
            opacity: 0;
        }
        .track {
            position: relative;
            width: 72px;
            height: 24px;
            background: #2c2c2c;
            border-radius: 60px;
            transition: 0.3s;
            box-shadow: inset 0 0 4px rgba(0,0,0,0.2);
        }
        body.dark .track {
            background: #1a1a1a;
        }
        .thumb {
            position: absolute;
            top: -6px;
            left: -6px;
            width: 36px;
            height: 36px;
            transition: 0.3s;
            display: flex;
            justify-content: center;
            align-items: center;
        }
        input:checked + .track .thumb {
            left: calc(100% - 36px + 6px);
        }
        .lightsaber {
            position: absolute;
            top: 0;
            left: 0;
            right: 0;
            bottom: 0;
            display: flex;
            justify-content: space-between;
            align-items: center;
            padding: 0 4px;
        }
        .lightsaber .light {
            width: 20px;
            height: 3px;
            background: #00ff9c;
            border-radius: 4px;
            box-shadow: 0 0 4px #00ff9c;
        }
        .lightsaber .dark {
            width: 20px;
            height: 3px;
            background: #ff3366;
            border-radius: 4px;
            box-shadow: 0 0 4px #ff3366;
        }
        .lightsaber .grip {
            width: 22px;
            height: 10px;
            background: #555;
            border-radius: 4px;
        }
        .circle {
            width: 36px;
            height: 36px;
            background: #1a1a1a;
            border-radius: 100%;
            display: flex;
            justify-content: center;
            align-items: center;
            transition: 0.2s;
            box-shadow: 0 2px 6px rgba(0,0,0,0.2);
        }
        body.dark .circle {
            background: #2a2a2a;
        }
        .sub-circle {
            width: 28px;
            height: 28px;
            background: #2c2c2c;
            border-radius: 100%;
        }
        body.dark .sub-circle {
            background: #3a3a3a;
        }
        .side {
            position: absolute;
            top: 0;
            left: 0;
            right: 0;
            bottom: 0;
            display: flex;
            justify-content: center;
            align-items: center;
            transition: 0.2s;
            transform: scale(0);
        }
        input:checked + .track .thumb .dark-side {
            transform: scale(1);
        }
        input:not(:checked) + .track .thumb .light-side {
            transform: scale(1);
        }
        .light-side .top {
            position: absolute;
            top: 4px;
            display: flex;
            gap: 4px;
        }
        .light-side .top .left,
        .light-side .top .right {
            width: 3px;
            height: 3px;
            background: #ffcc00;
            border-radius: 100%;
        }
        .light-side .center {
            display: flex;
            gap: 2px;
        }
        .light-side .center div {
            width: 2px;
            height: 10px;
            background: #ffcc00;
            border-radius: 2px;
        }
        .light-side .bottom {
            position: absolute;
            bottom: 4px;
        }
        .light-side .bottom .line {
            width: 12px;
            height: 2px;
            background: #ffcc00;
            border-radius: 2px;
        }
        .dark-side .circle {
            background: #ff3366;
        }
        .dark-side .sub-circle {
            background: #cc0033;
        }
        .app::-webkit-scrollbar {
            width: 6px;
        }
        .app::-webkit-scrollbar-track {
            background: rgba(0, 0, 0, 0.1);
            border-radius: 10px;
        }
        .app::-webkit-scrollbar-thumb {
            background: var(--primary);
            border-radius: 10px;
        }
    </style>
</head>
<body>
<div class="app">
    <div class="app__gradients">
        <svg style="width:0;height:0;position:absolute;" aria-hidden="true" focusable="false">
            <linearGradient id="ring" x1="0%" y1="0%" x2="100%" y2="100%">
                <stop offset="0%" stop-color="hsl(184, 66%, 44%)" />
                <stop offset="100%" stop-color="hsl(184, 66%, 74%)" />
            </linearGradient>
        </svg>
    </div>

    <div class="header">
        <button class="header__profile-btn" aria-label="Perfil">
            <svg class="icon header__profile-icon" viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="1.5">
                <circle cx="12" cy="8" r="4" />
                <path d="M5 20v-2a7 7 0 0 1 14 0v2" />
            </svg>
        </button>
        <h1>Laboratorio Nuclear</h1>
        <div class="scene">
            <div class="images">
                <img src="https://assets.codepen.io/907471/darkside.svg?v42" class="dark">
                <img src="https://assets.codepen.io/907471/lightside.svg?v42" class="light">
            </div>
            <label class="switch" id="switch" aria-label="Switch">
                <input type="checkbox" id="themeToggle">
                <div class="track">
                    <div class="lightsaber">
                        <div class="light"></div>
                        <div class="grip"></div>
                        <div class="dark"></div>
                    </div>
                    <div class="thumb">
                        <div class="side dark-side">
                            <div class="circle">
                                <div class="sub-circle"></div>
                            </div>
                        </div>
                        <div class="side light-side">
                            <div class="circle">
                                <div class="sub-circle"></div>
                            </div>
                            <div class="top">
                                <div class="left"></div>
                                <div class="right"></div>
                            </div>
                            <div class="center">
                                <div class="item-1"></div>
                                <div class="item-2"></div>
                                <div class="item-3"></div>
                                <div class="item-4"></div>
                                <div class="item-5"></div>
                            </div>
                            <div class="bottom">
                                <div class="line"></div>
                            </div>
                        </div>
                    </div>
                </div>
            </label>
        </div>
    </div>

    <div class="main__stat-blocks">
        <div class="main__stat-block">
            <div class="main__stat-graph">
                <svg class="ring" viewBox="0 0 120 120">
                    <circle class="ring-track" cx="60" cy="60" r="45" fill="none" stroke-width="8" />
                    <circle class="ring-stroke ring-stroke--steps" cx="60" cy="60" r="45" fill="none" stroke-width="8" stroke-dasharray="282.74" stroke-dashoffset="282.74" stroke-linecap="round" />
                </svg>
                <div class="main__stat-detail">
                    <span class="main__stat-value" id="statNeutrones">0</span>
                    <span class="main__stat-unit">n</span>
                </div>
            </div>
            <span class="main__stat-subtext">Neutrones</span>
        </div>
        <div class="main__stat-block">
            <div class="main__stat-graph">
                <svg class="ring" viewBox="0 0 120 120">
                    <circle class="ring-track" cx="60" cy="60" r="45" fill="none" stroke-width="8" />
                    <circle class="ring-stroke ring-stroke--cals" cx="60" cy="60" r="45" fill="none" stroke-width="8" stroke-dasharray="282.74" stroke-dashoffset="282.74" stroke-linecap="round" />
                </svg>
                <div class="main__stat-detail">
                    <span class="main__stat-value" id="statFisiones">0</span>
                    <span class="main__stat-unit">fis</span>
                </div>
            </div>
            <span class="main__stat-subtext">Fisiones</span>
        </div>
        <div class="main__stat-block">
            <div class="main__stat-graph">
                <svg class="ring" viewBox="0 0 120 120">
                    <circle class="ring-track" cx="60" cy="60" r="45" fill="none" stroke-width="8" />
                    <circle class="ring-stroke ring-stroke--miles" cx="60" cy="60" r="45" fill="none" stroke-width="8" stroke-dasharray="282.74" stroke-dashoffset="282.74" stroke-linecap="round" />
                </svg>
                <div class="main__stat-detail">
                    <span class="main__stat-value" id="statEnergia">0</span>
                    <span class="main__stat-unit">MeV</span>
                </div>
            </div>
            <span class="main__stat-subtext">Energia</span>
        </div>
    </div>

    <div class="nav-tabs">
        <button class="nav-btn active" onclick="showTab('montecarlo')">Monte Carlo</button>
        <button class="nav-btn" onclick="showTab('dispersión')">Dispersion</button>
        <button class="nav-btn" onclick="showTab('fision')">Fision 3D</button>
    </div>

    <div id="montecarlo" class="tab-content active">
        <div class="info-text">
            <strong>Metodo Monte Carlo</strong><br>
            Simulacion de poblacion de neutrones usando numeros aleatorios.
        </div>
        <div class="card">
            <div class="card-title">Parametros</div>
            <div class="input-group">
                <label>Probabilidad (p)</label>
                <input type="number" id="prob_p" value="0.5" step="0.05" min="0" max="1">
            </div>
            <div class="input-group">
                <label>Neutrones iniciales (n0)</label>
                <input type="number" id="neutrones_n0" value="100">
            </div>
            <div class="input-group">
                <label>Generaciones</label>
                <input type="number" id="generaciones" value="50" min="10" max="100">
            </div>
            <button class="btn-primary" onclick="simularMonteCarlo()">Ejecutar</button>
            <div id="resultado_mc" class="resultado">Listo</div>
        </div>
        <div class="card">
            <div class="card-title">Interpretacion</div>
            <div class="info-text" style="margin:0">
                <strong>Factor k:</strong><br>
                k < 1: Subcritico | k = 1: Critico | k > 1: Supercritico
            </div>
        </div>
        <div class="graph-container">
            <div class="graph-title">Evolucion de la Poblacion</div>
            <div id="grafica_mc" style="height: 250px;"></div>
        </div>
        <div class="graph-container">
            <div class="graph-title">Visualizacion</div>
            <canvas id="mcCanvas" class="simulation-canvas"></canvas>
        </div>
    </div>

    <div id="dispersión" class="tab-content">
        <div class="info-text">
            <strong>Dispersion Elastica de Neutrones</strong><br>
            Energia final vs angulo de dispersion (0° a 360°)
        </div>
        <div class="formula">
            $$E_l' = \\frac{1}{2}E_l[(1+\\alpha)+(1-\\alpha)\\cos\\theta_C]$$
            $$\\alpha = \\left(\\frac{A-1}{A+1}\\right)^2$$
        </div>
        <div class="card">
            <div class="card-title">Parametros</div>
            <div class="input-group">
                <label>Energia inicial (MeV)</label>
                <input type="number" id="energia_inicial_elastica" value="1.0" step="0.1">
            </div>
            <div class="input-group">
                <label>Nucleidos:</label>
                <div class="nucleido-selector" id="nucleidosList">
                    <button type="button" class="nucleido-btn" data-a="1" data-name="H-1">H-1</button>
                    <button type="button" class="nucleido-btn" data-a="12" data-name="C-12">C-12</button>
                    <button type="button" class="nucleido-btn" data-a="23" data-name="Na-23">Na-23</button>
                    <button type="button" class="nucleido-btn" data-a="238" data-name="U-238">U-238</button>
                </div>
            </div>
            <button class="btn-primary" onclick="agregarNucleido()">Agregar</button>
            <div class="flex-buttons">
                <button onclick="limpiarGrafica()">Limpiar</button>
            </div>
            <div id="nucleidos_activos" class="resultado">Activos: Ninguno</div>
        </div>
        <div class="graph-container">
            <div class="graph-title">Energia vs Angulo (0° a 360°)</div>
            <div id="grafica_elastica" style="height: 300px;"></div>
        </div>
    </div>

    <div id="fision" class="tab-content">
        <div class="info-text">
            <strong>Fision Nuclear 3D</strong><br>
            n + U-235 -> Ba-141 + Kr-92 + 2-3 n + 200 MeV
        </div>
        <div class="card">
            <div class="card-title">Simulacion 3D</div>
            <button class="btn-primary" onclick="abrirFision()">Abrir Simulacion</button>
            <div class="info-text" style="margin-top:0.8em">
                <strong>Controles:</strong><br>
                Disparar neutron | Reaccion en cadena | Reiniciar<br>
                Arrastrar: rotar | Scroll: zoom
            </div>
        </div>
    </div>
</div>

<script>
    let nucleidosActivos = [];

    const themeToggle = document.getElementById('themeToggle');
    
    function toggleTheme() {
        if (themeToggle.checked) {
            document.body.classList.add('dark');
            localStorage.setItem('theme', 'dark');
        } else {
            document.body.classList.remove('dark');
            localStorage.setItem('theme', 'light');
        }
        actualizarGraficaElastica();
        simularMonteCarlo();
    }

    themeToggle.addEventListener('change', toggleTheme);

    const savedTheme = localStorage.getItem('theme');
    if (savedTheme === 'dark') {
        document.body.classList.add('dark');
        themeToggle.checked = true;
    }

    function showTab(tabId) {
        document.querySelectorAll('.tab-content').forEach(tab => {
            tab.classList.remove('active');
        });
        document.getElementById(tabId).classList.add('active');
        
        document.querySelectorAll('.nav-btn').forEach(btn => {
            btn.classList.remove('active');
        });
        event.target.classList.add('active');
        
        if (tabId === 'montecarlo') {
            simularMonteCarlo();
        }
    }

    function energiaFinalElastica(E, A, theta) {
        const alpha = Math.pow((A - 1) / (A + 1), 2);
        const thetaRad = theta * Math.PI / 180;
        return 0.5 * E * ((1 + alpha) + (1 - alpha) * Math.cos(thetaRad));
    }

    function agregarNucleido() {
        const activeBtn = document.querySelector('.nucleido-btn.active');
        if (!activeBtn) {
            alert('Seleccione un nucleido');
            return;
        }
        
        const A = parseFloat(activeBtn.dataset.a);
        const name = activeBtn.dataset.name;
        const E = parseFloat(document.getElementById('energia_inicial_elastica').value);
        
        if (nucleidosActivos.some(n => n.A === A)) {
            alert(name + ' ya esta en la grafica');
            return;
        }
        
        nucleidosActivos.push({ A, name, E });
        actualizarListaNucleidos();
        actualizarGraficaElastica();
    }

    function actualizarListaNucleidos() {
        const container = document.getElementById('nucleidos_activos');
        if (nucleidosActivos.length === 0) {
            container.innerHTML = 'Activos: Ninguno';
            return;
        }
        container.innerHTML = 'Activos: ' + nucleidosActivos.map(n => n.name).join(', ');
    }

    function limpiarGrafica() {
        nucleidosActivos = [];
        actualizarListaNucleidos();
        actualizarGraficaElastica();
    }

    function actualizarGraficaElastica() {
        const theta_vals = Array.from({length: 361}, (_, i) => i);
        
        const traces = nucleidosActivos.map((nucleido, idx) => {
            const energy_ratio = theta_vals.map(theta => {
                const E_final = energiaFinalElastica(nucleido.E, nucleido.A, theta);
                return E_final / nucleido.E;
            });
            
            const colors = ['#00ff9c', '#667eea', '#ff6600', '#ff44cc'];
            
            return {
                x: theta_vals,
                y: energy_ratio,
                mode: 'lines',
                name: nucleido.name + ' (A=' + nucleido.A + ')',
                line: { color: colors[idx % colors.length], width: 2 }
            };
        });
        
        const isDark = document.body.classList.contains('dark');
        const fontColor = isDark ? '#fff' : '#2c3e4f';
        
        const layout = {
            template: 'plotly_dark',
            title: { text: 'Energia vs Angulo de Dispersion', font: { size: 10, color: fontColor } },
            xaxis: { title: "Angulo (grados)", range: [0, 360], dtick: 45, tickfont: { color: fontColor }, titlefont: { color: fontColor } },
            yaxis: { title: "E'/E0", range: [0, 1.05], tickfont: { color: fontColor }, titlefont: { color: fontColor } },
            plot_bgcolor: 'rgba(0,0,0,0)',
            paper_bgcolor: 'rgba(0,0,0,0)',
            legend: { font: { size: 8, color: fontColor } }
        };
        
        Plotly.newPlot('grafica_elastica', traces, layout, { responsive: true });
    }

    function monteCarlo(p, n0, pasos) {
        let neutrones = [n0];
        for (let step = 0; step < pasos; step++) {
            let nuevos = 0;
            for (let i = 0; i < neutrones[neutrones.length-1]; i++) {
                if (Math.random() < p) {
                    nuevos += Math.floor(Math.random() * 3);
                }
            }
            neutrones.push(nuevos);
        }
        return neutrones;
    }

    function simularMonteCarlo() {
        const p = parseFloat(document.getElementById('prob_p').value);
        const n0 = parseInt(document.getElementById('neutrones_n0').value);
        const pasos = parseInt(document.getElementById('generaciones').value);
        
        const datos = monteCarlo(p, n0, pasos);
        const k_efectivo = datos[datos.length-1] / datos[datos.length-2];
        
        let estado = "";
        if (k_efectivo < 0.95) estado = "Subcritico";
        else if (k_efectivo < 1.05) estado = "Critico";
        else estado = "Supercritico";
        
        document.getElementById('resultado_mc').innerHTML = 
            'Gen:' + datos.length + ' | Final:' + datos[datos.length-1] + ' | k=' + k_efectivo.toFixed(4) + '<br>' + estado;
        
        document.getElementById('statNeutrones').innerHTML = datos[datos.length-1];
        
        const isDark = document.body.classList.contains('dark');
        const fontColor = isDark ? '#fff' : '#2c3e4f';
        
        const trace = {
            y: datos,
            x: Array.from({length: datos.length}, (_, i) => i),
            mode: 'lines+markers',
            line: {color: '#00ff9c', width: 2},
            marker: {size: 2},
            fill: 'tozeroy',
            fillcolor: 'rgba(0, 255, 156, 0.1)'
        };
        
        const layout = {
            template: 'plotly_dark',
            title: { text: 'Monte Carlo', font: { size: 10, color: fontColor } },
            xaxis: { title: "Generacion", tickfont: { color: fontColor }, titlefont: { color: fontColor } },
            yaxis: { title: "Neutrones", type: 'log', tickfont: { color: fontColor }, titlefont: { color: fontColor } },
            plot_bgcolor: 'rgba(0,0,0,0)',
            paper_bgcolor: 'rgba(0,0,0,0)'
        };
        
        Plotly.newPlot('grafica_mc', [trace], layout);
        visualizarMonteCarlo3D(datos);
    }

    function visualizarMonteCarlo3D(datos) {
        const canvas = document.getElementById('mcCanvas');
        const ctx = canvas.getContext('2d');
        
        function draw() {
            canvas.width = canvas.clientWidth;
            canvas.height = canvas.clientHeight;
            const w = canvas.width;
            const h = canvas.height;
            
            const isDark = document.body.classList.contains('dark');
            const bgColor = isDark ? '#2a2a2a' : '#d0d0d0';
            
            ctx.fillStyle = bgColor;
            ctx.fillRect(0, 0, w, h);
            
            const maxVal = Math.max(...datos);
            const barWidth = w / datos.length * 0.6;
            const barSpacing = w / datos.length * 0.4;
            
            for (let i = 0; i < datos.length; i++) {
                const barHeight = Math.min((datos[i] / maxVal) * (h - 40), h - 40);
                const x = i * (barWidth + barSpacing) + barSpacing/2;
                const y = h - barHeight - 20;
                
                ctx.fillStyle = '#00ff9c';
                ctx.fillRect(x, y, barWidth, barHeight);
                
                ctx.fillStyle = isDark ? '#ccc' : '#333';
                ctx.font = '8px "Nunito"';
                ctx.textAlign = 'center';
                if (datos[i] < 100000) ctx.fillText(datos[i], x + barWidth/2, y - 2);
                ctx.fillStyle = isDark ? '#888' : '#666';
                ctx.fillText(i, x + barWidth/2, h - 8);
            }
        }
        
        draw();
        window.addEventListener('resize', () => draw());
    }

    function abrirFision() {
        window.open('/fision', '_blank');
    }

    document.querySelectorAll('.nucleido-btn').forEach(btn => {
        btn.addEventListener('click', function() {
            document.querySelectorAll('.nucleido-btn').forEach(b => b.classList.remove('active'));
            this.classList.add('active');
        });
    });

    document.getElementById('energia_inicial_elastica').addEventListener('input', () => {
        actualizarGraficaElastica();
    });

    window.onload = () => {
        simularMonteCarlo();
        document.querySelector('.nucleido-btn').classList.add('active');
        MathJax.Hub.Queue(["Typeset", MathJax.Hub]);
    };
</script>
</body>
</html>
"""

HTML_FISION = """
<!DOCTYPE html>
<html>
<head>
<meta charset="UTF-8">
<title>Simulador de Fision Nuclear 3D</title>
<style>
    * {
        border: 0;
        box-sizing: border-box;
        margin: 0;
        padding: 0;
    }
    :root {
        --hue: 184;
        --bg: hsl(var(--hue), 10%, 90%);
        --fg: hsl(var(--hue), 66%, 24%);
        --primary: hsl(var(--hue), 66%, 44%);
        --gradient: linear-gradient(145deg, hsl(var(--hue), 10%, 85%), hsl(var(--hue), 10%, 100%));
    }
    body { 
        margin: 0; 
        overflow: hidden; 
        background: var(--bg);
        font-family: "Nunito", sans-serif;
        position: relative;
    }
    body.dark {
        --bg: hsl(var(--hue), 10%, 10%);
        --fg: hsl(var(--hue), 66%, 94%);
        --gradient: linear-gradient(145deg, hsl(var(--hue), 10%, 15%), hsl(var(--hue), 10%, 30%));
    }
    #app { 
        width: 100%; 
        height: 100vh; 
    }
    .back-btn {
        position: absolute;
        top: 20px;
        left: 20px;
        z-index: 20;
        background: var(--gradient);
        color: var(--fg);
        padding: 8px 16px;
        cursor: pointer;
        font-family: "Nunito", sans-serif;
        font-size: 14px;
        font-weight: 600;
        text-decoration: none;
        border-radius: 1.5em;
        box-shadow: -0.5em -0.5em 1.5em hsl(0, 0%, 100%), 0.5em 0.5em 1.5em hsl(var(--hue), 5%, 65%);
        border: 1px solid rgba(255, 255, 255, 0.3);
    }
    .back-btn:active {
        transform: scale(0.95);
    }
    .controls {
        position: absolute;
        top: 20px;
        left: 120px;
        z-index: 10;
        display: flex;
        gap: 10px;
    }
    button {
        padding: 8px 16px;
        background: var(--gradient);
        color: var(--fg);
        border: none;
        cursor: pointer;
        font-family: "Nunito", sans-serif;
        font-size: 13px;
        font-weight: 600;
        border-radius: 1.5em;
        box-shadow: -0.5em -0.5em 1.5em hsl(0, 0%, 100%), 0.5em 0.5em 1.5em hsl(var(--hue), 5%, 65%);
        border: 1px solid rgba(255, 255, 255, 0.3);
    }
    button:active {
        transform: scale(0.95);
    }
    .info {
        position: absolute;
        bottom: 20px;
        left: 20px;
        color: var(--fg);
        background: var(--gradient);
        padding: 12px 16px;
        font-family: "Nunito", sans-serif;
        font-size: 12px;
        z-index: 10;
        border-radius: 1.5em;
        max-width: 280px;
        border: 1px solid rgba(255, 255, 255, 0.3);
    }
    .info h3 {
        margin: 0 0 8px 0;
        color: var(--primary);
    }
    .stats {
        position: absolute;
        top: 20px;
        right: 20px;
        color: var(--fg);
        background: var(--gradient);
        padding: 12px 16px;
        font-family: "Nunito", sans-serif;
        font-size: 12px;
        z-index: 10;
        border-radius: 1.5em;
        min-width: 180px;
        text-align: right;
        border: 1px solid rgba(255, 255, 255, 0.3);
    }
    .stats h3 {
        margin: 0 0 8px 0;
        color: var(--primary);
        text-align: left;
    }
    .stats .value {
        color: hsl(3, 90%, 55%);
        font-size: 18px;
        font-weight: bold;
    }
    .warning {
        color: hsl(3, 90%, 55%);
        animation: pulse 1s infinite;
        margin-top: 8px;
    }
    @keyframes pulse {
        0%, 100% { opacity: 1; }
        50% { opacity: 0.5; }
    }
    .scene {
        position: absolute;
        top: 20px;
        right: 20px;
        display: flex;
        justify-content: center;
        align-items: center;
        z-index: 30;
    }
    .images {
        position: absolute;
        opacity: 0;
        pointer-events: none;
    }
    .switch {
        display: flex;
        justify-content: center;
        align-items: center;
        width: 80px;
        cursor: pointer;
    }
    .switch input {
        position: absolute;
        opacity: 0;
    }
    .track {
        position: relative;
        width: 72px;
        height: 24px;
        background: #2c2c2c;
        border-radius: 60px;
        transition: 0.3s;
        box-shadow: inset 0 0 4px rgba(0,0,0,0.2);
    }
    body.dark .track {
        background: #1a1a1a;
    }
    .thumb {
        position: absolute;
        top: -6px;
        left: -6px;
        width: 36px;
        height: 36px;
        transition: 0.3s;
        display: flex;
        justify-content: center;
        align-items: center;
    }
    input:checked + .track .thumb {
        left: calc(100% - 36px + 6px);
    }
    .lightsaber {
        position: absolute;
        top: 0;
        left: 0;
        right: 0;
        bottom: 0;
        display: flex;
        justify-content: space-between;
        align-items: center;
        padding: 0 4px;
    }
    .lightsaber .light {
        width: 20px;
        height: 3px;
        background: #00ff9c;
        border-radius: 4px;
        box-shadow: 0 0 4px #00ff9c;
    }
    .lightsaber .dark {
        width: 20px;
        height: 3px;
        background: #ff3366;
        border-radius: 4px;
        box-shadow: 0 0 4px #ff3366;
    }
    .lightsaber .grip {
        width: 22px;
        height: 10px;
        background: #555;
        border-radius: 4px;
    }
    .circle {
        width: 36px;
        height: 36px;
        background: #1a1a1a;
        border-radius: 100%;
        display: flex;
        justify-content: center;
        align-items: center;
        transition: 0.2s;
        box-shadow: 0 2px 6px rgba(0,0,0,0.2);
    }
    body.dark .circle {
        background: #2a2a2a;
    }
    .sub-circle {
        width: 28px;
        height: 28px;
        background: #2c2c2c;
        border-radius: 100%;
    }
    body.dark .sub-circle {
        background: #3a3a3a;
    }
    .side {
        position: absolute;
        top: 0;
        left: 0;
        right: 0;
        bottom: 0;
        display: flex;
        justify-content: center;
        align-items: center;
        transition: 0.2s;
        transform: scale(0);
    }
    input:checked + .track .thumb .dark-side {
        transform: scale(1);
    }
    input:not(:checked) + .track .thumb .light-side {
        transform: scale(1);
    }
    .light-side .top {
        position: absolute;
        top: 4px;
        display: flex;
        gap: 4px;
    }
    .light-side .top .left,
    .light-side .top .right {
        width: 3px;
        height: 3px;
        background: #ffcc00;
        border-radius: 100%;
    }
    .light-side .center {
        display: flex;
        gap: 2px;
    }
    .light-side .center div {
        width: 2px;
        height: 10px;
        background: #ffcc00;
        border-radius: 2px;
    }
    .light-side .bottom {
        position: absolute;
        bottom: 4px;
    }
    .light-side .bottom .line {
        width: 12px;
        height: 2px;
        background: #ffcc00;
        border-radius: 2px;
    }
    .dark-side .circle {
        background: #ff3366;
    }
    .dark-side .sub-circle {
        background: #cc0033;
    }
</style>
</head>
<body>
    <div class="scene">
        <div class="images">
            <img src="https://assets.codepen.io/907471/darkside.svg?v42" class="dark">
            <img src="https://assets.codepen.io/907471/lightside.svg?v42" class="light">
        </div>
        <label class="switch" id="switch" aria-label="Switch">
            <input type="checkbox" id="themeToggleFision">
            <div class="track">
                <div class="lightsaber">
                    <div class="light"></div>
                    <div class="grip"></div>
                    <div class="dark"></div>
                </div>
                <div class="thumb">
                    <div class="side dark-side">
                        <div class="circle">
                            <div class="sub-circle"></div>
                        </div>
                    </div>
                    <div class="side light-side">
                        <div class="circle">
                            <div class="sub-circle"></div>
                        </div>
                        <div class="top">
                            <div class="left"></div>
                            <div class="right"></div>
                        </div>
                        <div class="center">
                            <div class="item-1"></div>
                            <div class="item-2"></div>
                            <div class="item-3"></div>
                            <div class="item-4"></div>
                            <div class="item-5"></div>
                        </div>
                        <div class="bottom">
                            <div class="line"></div>
                        </div>
                    </div>
                </div>
            </div>
        </label>
    </div>
    
    <a href="/" class="back-btn">Volver</a>
    <div class="controls">
        <button id="fireBtn">Disparar</button>
        <button id="resetBtn">Reiniciar</button>
        <button id="chainBtn">Cadena</button>
    </div>
    <div class="stats">
        <h3>Reactor</h3>
        <p>Uranio: <span id="atomCount" class="value">0</span></p>
        <p>Neutrones: <span id="neutronCount" class="value">0</span></p>
        <p>Fisiones: <span id="fissionCount" class="value">0</span></p>
        <p>Energia: <span id="energyReleased" class="value">0</span> MeV</p>
        <div id="chainWarning" style="display:none;" class="warning">CADENA ACTIVA</div>
    </div>
    <div class="info">
        <h3>Fision Nuclear</h3>
        <p>n + U-235 -> Ba-141 + Kr-92 + 2-3 n</p>
        <p>Energia: 200 MeV por fision</p>
        <p>150 atomos | Arrastrar: rotar | Scroll: zoom</p>
    </div>
    <div id="app"></div>
    <script>
        const themeToggleFision = document.getElementById('themeToggleFision');
        
        function toggleThemeFision() {
            if (themeToggleFision.checked) {
                document.body.classList.add('dark');
                localStorage.setItem('themeFision', 'dark');
            } else {
                document.body.classList.remove('dark');
                localStorage.setItem('themeFision', 'light');
            }
        }
        
        themeToggleFision.addEventListener('change', toggleThemeFision);
        
        const savedThemeFision = localStorage.getItem('themeFision');
        if (savedThemeFision === 'dark') {
            document.body.classList.add('dark');
            themeToggleFision.checked = true;
        }
    </script>
<script type="importmap">
{
  "imports": {
    "three": "https://esm.sh/three",
    "three/addons/": "https://esm.sh/three/addons/"
  }
}
</script>
<script type="module">
import * as THREE from "three";
import { OrbitControls } from "three/addons/controls/OrbitControls.js";

let scene, camera, renderer, controls;
let atoms = [];
let neutrons = [];
let fragments = [];
let energyParticles = [];
let fissionCount = 0;
let totalEnergy = 0;
let chainReactionActive = false;
let chainReactionInterval = null;

const ENERGY_PER_FISSION = 200;
const NEUTRON_SPEED = 0.35;
const FISSION_RADIUS = 1.3;

function init() {
    scene = new THREE.Scene();
    scene.background = new THREE.Color(0x101018);
    scene.fog = new THREE.FogExp2(0x101018, 0.0005);

    camera = new THREE.PerspectiveCamera(60, innerWidth/innerHeight, 0.1, 1000);
    camera.position.set(25, 20, 32);

    renderer = new THREE.WebGLRenderer({antialias: true});
    renderer.setSize(innerWidth, innerHeight);
    renderer.shadowMap.enabled = true;
    document.getElementById("app").appendChild(renderer.domElement);

    controls = new OrbitControls(camera, renderer.domElement);
    controls.enableDamping = true;
    controls.enableZoom = true;
    controls.zoomSpeed = 1.2;
    controls.target.set(0, 0, 0);

    const ambientLight = new THREE.AmbientLight(0x222222);
    scene.add(ambientLight);
    const directionalLight = new THREE.DirectionalLight(0xffffff, 0.6);
    directionalLight.position.set(5, 10, 7);
    directionalLight.castShadow = true;
    scene.add(directionalLight);
    const backLight = new THREE.PointLight(0x4466ff, 0.3);
    backLight.position.set(-5, 0, -5);
    scene.add(backLight);
    const fillLight = new THREE.PointLight(0xffaa66, 0.25);
    fillLight.position.set(3, 5, 4);
    scene.add(fillLight);
    const rimLight = new THREE.PointLight(0xff66aa, 0.2);
    rimLight.position.set(-3, 4, -4);
    scene.add(rimLight);

    createInitialAtoms();
    animate();
    setInterval(updateUI, 100);
}

const tex = new THREE.TextureLoader().load("https://threejs.org/examples/textures/sprites/disc.png");

const createShell = (count, radius, color) => {
    const pos = [];
    const col = [];
    const pointCount = Math.min(count * 6, 120);
    for (let i = 0; i < pointCount; i++){
        const r = radius + (Math.random() - 0.5) * 0.3;
        const theta = Math.acos(2 * Math.random() - 1);
        const phi = Math.random() * Math.PI * 2;
        const x = r * Math.sin(theta) * Math.cos(phi);
        const y = r * Math.sin(theta) * Math.sin(phi);
        const z = r * Math.cos(theta);
        pos.push(x, y, z);
        const c = new THREE.Color(color);
        const fade = 0.4 + Math.random() * 0.6;
        col.push(c.r * fade, c.g * fade, c.b * fade);
    }
    const geo = new THREE.BufferGeometry();
    geo.setAttribute("position", new THREE.Float32BufferAttribute(pos, 3));
    geo.setAttribute("color", new THREE.Float32BufferAttribute(col, 3));
    const mat = new THREE.PointsMaterial({ size: 0.07, map: tex, transparent: true, vertexColors: true, blending: THREE.AdditiveBlending, depthWrite: false });
    return new THREE.Points(geo, mat);
};

function createAtom(position, color = 0xff4466) {
    const group = new THREE.Group();
    const nucleus = new THREE.Mesh(new THREE.SphereGeometry(0.55, 32, 32), new THREE.MeshStandardMaterial({ color: color, emissive: 0x331100, emissiveIntensity: 0.15, metalness: 0.7, roughness: 0.3 }));
    nucleus.castShadow = true;
    group.add(nucleus);
    const shells = [{n:2,r:1.0,color:0xff6688},{n:8,r:1.6,color:0xff7799},{n:18,r:2.3,color:0xff88aa},{n:32,r:3.0,color:0xff99bb},{n:21,r:3.8,color:0xffaacc},{n:9,r:4.7,color:0xffbbdd},{n:2,r:5.6,color:0xffccee}];
    shells.forEach(s => { const c = createShell(s.n, s.r, s.color); group.add(c); });
    group.userData = { clouds: shells.map((_,i)=>group.children[i+1]), type: "uranium" };
    group.position.copy(position);
    scene.add(group);
    return group;
}

function createInitialAtoms() {
    atoms.forEach(atom => scene.remove(atom));
    atoms = [];
    const positions = [];
    const radius = 14;
    for(let i = 0; i < 150; i++) {
        let x, y, z, d;
        do { x = (Math.random() - 0.5) * radius * 2; y = (Math.random() - 0.5) * radius * 1.5; z = (Math.random() - 0.5) * radius * 2; d = Math.sqrt(x*x + y*y + z*z); } while(d > radius);
        positions.push(new THREE.Vector3(x, y, z));
    }
    positions.forEach(pos => atoms.push(createAtom(pos)));
    fissionCount = 0;
    totalEnergy = 0;
    updateUI();
}

function createNeutron(position, direction, speed = NEUTRON_SPEED) {
    const m = new THREE.Mesh(new THREE.SphereGeometry(0.18, 16, 16), new THREE.MeshStandardMaterial({ color: 0xffdd99, emissive: 0xff8844, emissiveIntensity: 0.8 }));
    m.position.copy(position);
    m.castShadow = true;
    m.userData = { velocity: direction.clone().normalize().multiplyScalar(speed) };
    scene.add(m);
    neutrons.push(m);
    updateUI();
}

function fission(atom, neutronPosition) {
    const atomPos = atom.position.clone();
    scene.remove(atom);
    const idx = atoms.indexOf(atom);
    if(idx !== -1) atoms.splice(idx, 1);
    fissionCount++;
    totalEnergy += ENERGY_PER_FISSION;
    updateUI();
    if(chainReactionActive && atoms.length > 0) document.getElementById("chainWarning").style.display = "block";
    
    const bariumGroup = new THREE.Group();
    const baNuc = new THREE.Mesh(new THREE.SphereGeometry(0.45, 24, 24), new THREE.MeshStandardMaterial({ color: 0x66cc99, emissive: 0x226644, emissiveIntensity: 0.25 }));
    bariumGroup.add(baNuc);
    [{n:2,r:0.8,c:0x88ffaa},{n:8,r:1.25,c:0xaaffcc},{n:18,r:1.75,c:0xccffdd},{n:18,r:2.3,c:0xddffee},{n:8,r:2.9,c:0xeeffff},{n:2,r:3.5,c:0xaaffff}].forEach(s => bariumGroup.add(createShell(s.n, s.r, s.c)));
    
    const kryptonGroup = new THREE.Group();
    const krNuc = new THREE.Mesh(new THREE.SphereGeometry(0.35, 24, 24), new THREE.MeshStandardMaterial({ color: 0xff9966, emissive: 0x442200, emissiveIntensity: 0.25 }));
    kryptonGroup.add(krNuc);
    [{n:2,r:0.7,c:0xffaa88},{n:8,r:1.1,c:0xffbb99},{n:18,r:1.6,c:0xffccaa},{n:8,r:2.1,c:0xffddbb}].forEach(s => kryptonGroup.add(createShell(s.n, s.r, s.c)));
    
    const dir = neutronPosition.clone().sub(atomPos).normalize();
    bariumGroup.position.copy(atomPos.clone().add(dir.clone().multiplyScalar(1.5)));
    kryptonGroup.position.copy(atomPos.clone().add(dir.clone().multiplyScalar(-1.5)));
    bariumGroup.userData = { velocity: dir.clone().multiplyScalar(0.06) };
    kryptonGroup.userData = { velocity: dir.clone().multiplyScalar(-0.06) };
    scene.add(bariumGroup);
    scene.add(kryptonGroup);
    fragments.push(bariumGroup, kryptonGroup);
    
    const nCount = 2 + Math.floor(Math.random() * 2);
    for(let i = 0; i < nCount; i++) {
        const rd = new THREE.Vector3((Math.random()-0.5)*2, (Math.random()-0.5)*2, (Math.random()-0.5)*2).normalize();
        createNeutron(atomPos, rd, NEUTRON_SPEED * (chainReactionActive ? 1.15 : 1.0));
    }
    
    for(let i = 0; i < 60; i++) {
        const p = new THREE.Mesh(new THREE.SphereGeometry(0.06, 6, 6), new THREE.MeshBasicMaterial({ color: [0xffaa44,0xff6644,0xffaa88][Math.floor(Math.random()*3)], transparent: true, blending: THREE.AdditiveBlending }));
        p.position.copy(atomPos);
        const a1 = Math.random() * Math.PI * 2, a2 = Math.random() * Math.PI * 2, sp = 0.12 + Math.random() * 0.18;
        p.userData = { velocity: new THREE.Vector3(Math.sin(a1)*Math.cos(a2)*sp, Math.sin(a1)*Math.sin(a2)*sp, Math.cos(a1)*sp), life: 1.0, decay: 0.012 };
        scene.add(p);
        energyParticles.push(p);
    }
    
    const explLight = new THREE.PointLight(0xff8844, 2.0, 20);
    explLight.position.copy(atomPos);
    scene.add(explLight);
    let int = 2.0;
    const li = setInterval(() => { int -= 0.25; explLight.intensity = int; if(int <= 0) { clearInterval(li); scene.remove(explLight); } }, 35);
}

function startChainReaction() {
    if(chainReactionActive) return;
    chainReactionActive = true;
    document.getElementById("chainWarning").style.display = "block";
    if(atoms.length > 0 && neutrons.length === 0) {
        const start = new THREE.Vector3(0, 5, 25);
        const target = atoms[Math.floor(Math.random() * atoms.length)].position;
        createNeutron(start, new THREE.Vector3().subVectors(target, start), NEUTRON_SPEED * 1.2);
    }
    chainReactionInterval = setInterval(() => {
        if(chainReactionActive && atoms.length > 0 && neutrons.length < 10) {
            const ra = atoms[Math.floor(Math.random() * atoms.length)];
            const start = new THREE.Vector3(ra.position.x + (Math.random()-0.5)*8, ra.position.y + (Math.random()-0.5)*6 + 6, ra.position.z + (Math.random()-0.5)*8 + 18);
            createNeutron(start, new THREE.Vector3().subVectors(ra.position, start), NEUTRON_SPEED * 1.25);
        }
        if(atoms.length === 0 && chainReactionActive) stopChainReaction();
    }, 1000);
}

function stopChainReaction() {
    chainReactionActive = false;
    document.getElementById("chainWarning").style.display = "none";
    if(chainReactionInterval) { clearInterval(chainReactionInterval); chainReactionInterval = null; }
}

function resetSimulation() {
    stopChainReaction();
    neutrons.forEach(n => scene.remove(n));
    fragments.forEach(f => scene.remove(f));
    energyParticles.forEach(p => scene.remove(p));
    atoms.forEach(a => scene.remove(a));
    neutrons = []; fragments = []; energyParticles = [];
    createInitialAtoms();
    updateUI();
}

function updateUI() {
    document.getElementById("atomCount").innerHTML = atoms.length;
    document.getElementById("neutronCount").innerHTML = neutrons.length;
    document.getElementById("fissionCount").innerHTML = fissionCount;
    document.getElementById("energyReleased").innerHTML = totalEnergy.toFixed(0);
    if(atoms.length === 0) document.getElementById("chainWarning").innerHTML = "COMPLETADA";
}

function animate() {
    requestAnimationFrame(animate);
    atoms.forEach(atom => { if(atom.userData && atom.userData.clouds) { atom.userData.clouds.forEach((c, i) => { c.rotation.y += 0.0008 * (i+1); c.rotation.x += 0.0005 * (i+1); }); } });
    for(let i=fragments.length-1; i>=0; i--) { const f = fragments[i]; if(f.userData && f.userData.velocity) { f.position.add(f.userData.velocity); f.children.forEach(c => { if(c.isPoints) { c.rotation.y += 0.015; c.rotation.x += 0.01; } }); } if(Math.abs(f.position.x) > 35 || Math.abs(f.position.y) > 25 || Math.abs(f.position.z) > 35) { scene.remove(f); fragments.splice(i,1); } }
    for(let i=energyParticles.length-1; i>=0; i--) { const p = energyParticles[i]; p.position.add(p.userData.velocity); p.userData.life -= p.userData.decay; const s = p.userData.life * 0.8; p.scale.set(s,s,s); p.material.opacity = p.userData.life; if(p.userData.life <= 0) { scene.remove(p); energyParticles.splice(i,1); } }
    for(let i=neutrons.length-1; i>=0; i--) { const n = neutrons[i]; n.position.add(n.userData.velocity); n.material.emissiveIntensity = 0.5 + Math.sin(Date.now()*0.02)*0.5; let hit = false; for(let j=0; j<atoms.length; j++) { const a = atoms[j]; if(a && n.position.distanceTo(a.position) < FISSION_RADIUS) { fission(a, n.position); scene.remove(n); neutrons.splice(i,1); hit = true; break; } } if(!hit && (Math.abs(n.position.x) > 45 || Math.abs(n.position.y) > 35 || Math.abs(n.position.z) > 45)) { scene.remove(n); neutrons.splice(i,1); } }
    controls.update();
    renderer.render(scene, camera);
}

document.getElementById("fireBtn").onclick = () => { if(atoms.length === 0) { alert("No quedan atomos. Reinicie."); return; } const start = new THREE.Vector3(0, 5, 28); const target = atoms[Math.floor(Math.random() * atoms.length)].position; createNeutron(start, new THREE.Vector3().subVectors(target, start)); };
document.getElementById("resetBtn").onclick = resetSimulation;
document.getElementById("chainBtn").onclick = startChainReaction;
window.addEventListener("resize", () => { camera.aspect = innerWidth / innerHeight; camera.updateProjectionMatrix(); renderer.setSize(innerWidth, innerHeight); });
init();
</script>
</body>
</html>
"""

class Handler(BaseHTTPRequestHandler):
    def do_GET(self):
        if self.path == '/':
            self.send_response(200)
            self.send_header('Content-type', 'text/html; charset=utf-8')
            self.end_headers()
            self.wfile.write(HTML_PRINCIPAL.encode('utf-8'))
        elif self.path == '/fision':
            self.send_response(200)
            self.send_header('Content-type', 'text/html; charset=utf-8')
            self.end_headers()
            self.wfile.write(HTML_FISION.encode('utf-8'))
        else:
            self.send_response(404)
            self.end_headers()
    
    def log_message(self, format, *args):
        pass

def open_browser():
    webbrowser.open('http://localhost:8080')

def main():
    port = 8080
    server = HTTPServer(('localhost', port), Handler)
    print("=" * 50)
    print(" LABORATORIO NUCLEAR")
    print("=" * 50)
    print(f" Servidor activo en: http://localhost:{port}")
    print("=" * 50)
    print(" Presiona Ctrl+C para detener")
    print("=" * 50)
    
    Timer(1, open_browser).start()
    
    try:
        server.serve_forever()
    except KeyboardInterrupt:
        print("\n Cerrando servidor...")
        server.shutdown()
        print(" Servidor detenido")

if __name__ == '__main__':
    main()

 LABORATORIO NUCLEAR
 Servidor activo en: http://localhost:8080
 Presiona Ctrl+C para detener

 Cerrando servidor...
 Servidor detenido


In [4]:
import webbrowser
import os

HTML_PRINCIPAL = """
<!DOCTYPE html>
<html lang="es">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Laboratorio Nuclear</title>
    <link href="https://fonts.googleapis.com/css2?family=Nunito:wght@300;400;600;700&display=swap" rel="stylesheet">
    <script src="https://cdn.plot.ly/plotly-3.1.0.min.js"></script>
    <script src="https://cdnjs.cloudflare.com/ajax/libs/mathjax/2.7.7/MathJax.js?config=TeX-MML-AM_CHTML"></script>
    <style>
        * {
            border: 0;
            box-sizing: border-box;
            margin: 0;
            padding: 0;
        }
        :root {
            --hue: 184;
            --bg: hsl(var(--hue), 10%, 90%);
            --fg: hsl(var(--hue), 66%, 24%);
            --primary: hsl(var(--hue), 66%, 44%);
            --gradient: linear-gradient(145deg, hsl(var(--hue), 10%, 85%), hsl(var(--hue), 10%, 100%));
            font-size: 16px;
        }
        body,
        button {
            color: var(--fg);
            font: 1em/1.5 "Nunito", sans-serif;
        }
        body {
            background: var(--bg);
            height: 100vh;
            display: grid;
            place-items: center;
            padding: 1.5em 0 0 0;
            position: relative;
        }
        body:after {
            content: "";
            display: block;
            height: 1.5em;
            width: 100%;
        }
        .app {
            background: hsl(var(--hue), 10%, 85%);
            border-radius: 3em;
            flex-direction: column;
            padding: 2.25em;
            width: 24.375em;
            height: auto;
            min-height: 52.75em;
            max-height: 90vh;
            overflow-y: auto;
            box-shadow: 0 8px 32px rgba(0, 0, 0, 0.1);
            border: 1px solid rgba(255, 255, 255, 0.5);
        }
        .app__gradients {
            position: absolute;
            width: 1px;
            height: 1px;
        }
        .icon {
            display: block;
            margin: auto;
            width: 1.5em;
            height: 1.5em;
        }
        .icon circle,
        .icon path {
            fill: currentColor;
            transition: fill 0.15s linear;
        }
        .icon ellipse,
        .icon polygon {
            stroke: currentColor;
            transition: stroke 0.15s linear;
        }
        .icon .no-fill {
            fill: none;
            stroke: currentColor;
        }
        .icon--red path {
            fill: hsl(3, 90%, 55%);
        }
        .icon--pulse {
            animation:
                bpm 1s linear,
                pulse 0.75s 1s linear infinite;
        }
        .ring,
        .sr-only {
            position: absolute;
        }
        .ring {
            display: block;
            inset: 0;
            width: 100%;
            height: auto;
        }
        .ring-fill,
        .ring-stroke {
            stroke: url("#ring");
        }
        .ring-stroke {
            animation-duration: 1s;
            animation-timing-function: ease-in-out;
        }
        .ring-track {
            stroke: hsl(var(--hue), 10%, 80%);
        }
        .sr-only {
            clip: rect(1px, 1px, 1px, 1px);
            overflow: hidden;
            width: 1px;
            height: 1px;
        }
        .header {
            display: flex;
            justify-content: space-between;
            align-items: center;
            margin-bottom: 1.5em;
        }
        .header__profile-btn,
        .header__notes-btn {
            width: 3em;
            height: 3em;
        }
        .header__profile-btn {
            border-radius: 1em;
            box-shadow: 0 0 0 0.125em inset;
        }
        .header__notes-btn {
            margin-inline-end: -1em;
        }
        .header__profile-btn:active,
        .header__notes-btn:active {
            transform: scale(0.9);
        }
        .header__profile-btn:focus {
            box-shadow: 0 0 0 0.125em var(--primary) inset;
        }
        .header__profile-icon {
            border-radius: 0.5em;
            margin: auto;
            width: 2em;
            height: 2em;
        }
        .header__notes-btn:focus .icon path {
            fill: var(--primary);
        }
        .header h1 {
            font-size: 1rem;
            text-align: center;
            flex: 1;
        }
        .main__date-nav {
            margin-bottom: 1.5em;
        }
        .main__date-arrow-btn,
        .main__date-edit-btn {
            height: 1.5em;
        }
        .main__date-arrow-btn {
            width: 1.5em;
        }
        .main__date-arrow-btn:active .icon path,
        .main__date-arrow-btn:focus .icon path {
            fill: var(--primary);
        }
        .main__date {
            text-transform: uppercase;
        }
        .main__date-edit-btn {
            min-width: 1.5em;
        }
        .main__date-edit-btn:active,
        .main__date-edit-btn:focus {
            color: var(--primary);
        }
        .main__stat-blocks {
            display: grid;
            grid-template-columns: repeat(3, 1fr);
            grid-gap: 1.5em;
            margin-bottom: 1.5em;
        }
        .main__stat-block {
            background: var(--gradient);
            border-radius: 1.5em;
            box-shadow:
                -0.75em -0.75em 2.25em hsl(0, 0%, 100%),
                0.75em 0.75em 2.25em hsl(var(--hue), 5%, 65%);
            padding: 0.75em;
            text-align: center;
            width: 100%;
        }
        .main__stat-block--lg {
            grid-column: 1 / 4;
            padding: 1.5em;
        }
        .main__stat-rows,
        .main__stat-row {
            margin-bottom: 1.5em;
        }
        .main__stat-row {
            align-items: center;
        }
        .main__stat-graph {
            margin: 0 auto 0.75em auto;
            position: relative;
            width: 3.75em;
            height: 3.75em;
        }
        .main__stat-graph .main__stat-detail {
            display: flex;
            flex-direction: column;
            justify-content: center;
            position: absolute;
            inset: 0;
        }
        .main__stat-block--lg .main__stat-graph {
            margin: auto;
            width: 11.25em;
            height: 11.25em;
        }
        .main__stat-block--lg .icon {
            margin: 0 auto;
            width: 2.25em;
            height: 2.25em;
        }
        .main__stat-row .main__stat-graph {
            background: var(--gradient);
            border-radius: 1em;
            box-shadow:
                -0.75em -0.75em 2.25em hsl(0, 0%, 100%),
                0.75em 0.75em 2.25em hsl(var(--hue), 5%, 65%);
            margin: 0;
            margin-inline-end: 1.5em;
        }
        .main__stat-value,
        .main__stat-unit {
            display: block;
        }
        .main__stat-value {
            font-size: 1.25em;
            line-height: 1.2;
        }
        .main__stat-block--lg .main__stat-value {
            font-size: 2em;
            line-height: 1.5;
        }
        .main__stat-unit,
        .main__stat-subtext {
            font-weight: 300;
        }
        .main__stat-subtext {
            color: hsl(var(--hue), 10%, 30%);
        }
        .footer {
            margin-top: auto;
        }
        .footer__nav-btn {
            background: var(--gradient);
            border-radius: 50%;
            box-shadow:
                1em 1em 2em hsl(var(--hue), 5%, 65%),
                -1em -1em 2em hsl(0, 0%, 100%);
            width: 3em;
            height: 3em;
        }
        .footer__nav-btn:active {
            box-shadow:
                0.75em 0.75em 1.5em hsl(var(--hue), 5%, 65%),
                -0.75em -0.75em 1.5em hsl(0, 0%, 100%);
            transform: scale(0.9);
        }
        .footer__nav-btn:focus .icon circle,
        .footer__nav-btn:focus .icon path {
            fill: var(--primary);
        }
        body.dark {
            --bg: hsl(var(--hue), 10%, 10%);
            --fg: hsl(var(--hue), 66%, 94%);
            --primary: hsl(var(--hue), 66%, 44%);
            --gradient: linear-gradient(145deg, hsl(var(--hue), 10%, 15%), hsl(var(--hue), 10%, 30%));
        }
        body.dark .app {
            background: hsl(var(--hue), 10%, 20%);
        }
        body.dark .icon--red path {
            fill: hsl(3, 90%, 65%);
        }
        body.dark .ring-track {
            stroke: hsl(var(--hue), 10%, 30%);
        }
        body.dark .main__stat-block,
        body.dark .main__stat-row .main__stat-graph {
            box-shadow:
                -0.75em -0.75em 2.25em hsl(var(--hue), 10%, 30%),
                0.75em 0.75em 2.25em hsl(var(--hue), 5%, 5%);
        }
        body.dark .main__stat-subtext {
            color: hsl(var(--hue), 10%, 70%);
        }
        body.dark .footer__nav-btn {
            box-shadow:
                -1em -1em 2em hsl(var(--hue), 10%, 30%),
                1em 1em 2em hsl(var(--hue), 5%, 5%);
        }
        body.dark .footer__nav-btn:active {
            box-shadow:
                -0.75em -0.75em 1.5em hsl(var(--hue), 10%, 30%),
                0.75em 0.75em 1.5em hsl(var(--hue), 5%, 5%);
        }
        @keyframes statFill {
            from { color: var(--fg); }
            to { color: hsl(var(--hue), 66%, 94%); }
        }
        @keyframes ringFill {
            from { r: 82px; stroke-width: 16; }
            to { r: 45px; stroke-width: 90; }
        }
        @keyframes stepCount {
            from { stroke-dashoffset: 515.22; }
            to { stroke-dashoffset: 0; }
        }
        @keyframes cals {
            from { stroke-dashoffset: 163.36; }
            to { stroke-dashoffset: 12.25; }
        }
        @keyframes miles {
            from { stroke-dashoffset: 163.36; }
            to { stroke-dashoffset: 35.39; }
        }
        @keyframes mins {
            from { stroke-dashoffset: 163.36; }
            to { stroke-dashoffset: 65.34; }
        }
        @keyframes bpm {
            from { transform: scale(0); }
            37.5% { transform: scale(1.2); }
            75%, to { transform: scale(1); }
        }
        @keyframes stepHrs {
            from { stroke-dashoffset: 131.95; }
            to { stroke-dashoffset: 52.78; }
        }
        @keyframes pulse {
            from, 75%, to { transform: scale(1); }
            25% { transform: scale(0.9); }
            50% { transform: scale(1.2); }
        }
        .nav-tabs {
            display: flex;
            gap: 0.5em;
            margin-bottom: 1.5em;
            justify-content: center;
        }
        .nav-btn {
            background: var(--gradient);
            border-radius: 1.5em;
            padding: 0.5em 1em;
            font-size: 0.7rem;
            font-weight: 600;
            cursor: pointer;
            box-shadow: -0.5em -0.5em 1.5em hsl(0, 0%, 100%), 0.5em 0.5em 1.5em hsl(var(--hue), 5%, 65%);
        }
        .nav-btn.active {
            background: var(--primary);
            color: white;
        }
        .tab-content {
            display: none;
        }
        .tab-content.active {
            display: block;
        }
        .card {
            background: var(--gradient);
            border-radius: 1.5em;
            padding: 1em;
            margin-bottom: 1em;
            box-shadow: -0.75em -0.75em 2.25em hsl(0, 0%, 100%), 0.75em 0.75em 2.25em hsl(var(--hue), 5%, 65%);
        }
        .card-title {
            font-weight: 700;
            margin-bottom: 0.5em;
            color: var(--primary);
        }
        .input-group {
            margin-bottom: 0.8em;
        }
        .input-group label {
            display: block;
            font-size: 0.7rem;
            font-weight: 600;
            margin-bottom: 0.2em;
        }
        .input-group input, .input-group select {
            width: 100%;
            padding: 0.4em 0.6em;
            background: hsl(var(--hue), 10%, 85%);
            border-radius: 1em;
            font-family: "Nunito", sans-serif;
            font-size: 0.8rem;
            border: 1px solid rgba(255, 255, 255, 0.3);
        }
        button {
            width: 100%;
            padding: 0.5em;
            background: var(--gradient);
            border-radius: 1.5em;
            font-family: "Nunito", sans-serif;
            font-weight: 600;
            cursor: pointer;
            margin-top: 0.3em;
            box-shadow: -0.5em -0.5em 1.5em hsl(0, 0%, 100%), 0.5em 0.5em 1.5em hsl(var(--hue), 5%, 65%);
            border: 1px solid rgba(255, 255, 255, 0.3);
        }
        button:active {
            transform: scale(0.95);
        }
        .btn-primary {
            background: var(--primary);
            color: white;
        }
        .resultado {
            margin-top: 0.5em;
            padding: 0.5em;
            background: hsl(var(--hue), 10%, 80%);
            border-radius: 1em;
            text-align: center;
            font-size: 0.7rem;
        }
        .graph-container {
            background: var(--gradient);
            border-radius: 1.5em;
            padding: 1em;
            margin-bottom: 1em;
            box-shadow: -0.75em -0.75em 2.25em hsl(0, 0%, 100%), 0.75em 0.75em 2.25em hsl(var(--hue), 5%, 65%);
        }
        .graph-title {
            font-size: 0.7rem;
            font-weight: 700;
            margin-bottom: 0.5em;
            text-align: center;
        }
        .simulation-canvas {
            width: 100%;
            height: 200px;
            background: hsl(var(--hue), 10%, 85%);
            border-radius: 1em;
        }
        .info-text {
            background: hsl(var(--hue), 10%, 80%);
            padding: 0.8em;
            border-radius: 1em;
            margin-bottom: 1em;
            font-size: 0.7rem;
            border-left: 3px solid var(--primary);
        }
        .formula {
            background: hsl(var(--hue), 10%, 80%);
            padding: 0.8em;
            border-radius: 1em;
            margin: 0.8em 0;
            font-size: 0.7rem;
            overflow-x: auto;
        }
        .nucleido-selector {
            display: flex;
            gap: 0.5em;
            margin-bottom: 0.8em;
            flex-wrap: wrap;
        }
        .nucleido-btn {
            flex: 1;
            padding: 0.3em;
            font-size: 0.7rem;
            margin: 0;
            background: hsl(var(--hue), 10%, 85%);
        }
        .nucleido-btn.active {
            background: var(--primary);
            color: white;
        }
        .flex-buttons {
            display: flex;
            gap: 0.5em;
        }
        .flex-buttons button {
            flex: 1;
        }
        .scene {
            display: flex;
            justify-content: center;
            align-items: center;
        }
        .images {
            position: absolute;
            opacity: 0;
            pointer-events: none;
        }
        .switch {
            display: flex;
            justify-content: center;
            align-items: center;
            width: 80px;
            cursor: pointer;
        }
        .switch input {
            position: absolute;
            opacity: 0;
        }
        .track {
            position: relative;
            width: 72px;
            height: 24px;
            background: #2c2c2c;
            border-radius: 60px;
            transition: 0.3s;
            box-shadow: inset 0 0 4px rgba(0,0,0,0.2);
        }
        body.dark .track {
            background: #1a1a1a;
        }
        .thumb {
            position: absolute;
            top: -6px;
            left: -6px;
            width: 36px;
            height: 36px;
            transition: 0.3s;
            display: flex;
            justify-content: center;
            align-items: center;
        }
        input:checked + .track .thumb {
            left: calc(100% - 36px + 6px);
        }
        .lightsaber {
            position: absolute;
            top: 0;
            left: 0;
            right: 0;
            bottom: 0;
            display: flex;
            justify-content: space-between;
            align-items: center;
            padding: 0 4px;
        }
        .lightsaber .light {
            width: 20px;
            height: 3px;
            background: #00ff9c;
            border-radius: 4px;
            box-shadow: 0 0 4px #00ff9c;
        }
        .lightsaber .dark {
            width: 20px;
            height: 3px;
            background: #ff3366;
            border-radius: 4px;
            box-shadow: 0 0 4px #ff3366;
        }
        .lightsaber .grip {
            width: 22px;
            height: 10px;
            background: #555;
            border-radius: 4px;
        }
        .circle {
            width: 36px;
            height: 36px;
            background: #1a1a1a;
            border-radius: 100%;
            display: flex;
            justify-content: center;
            align-items: center;
            transition: 0.2s;
            box-shadow: 0 2px 6px rgba(0,0,0,0.2);
        }
        body.dark .circle {
            background: #2a2a2a;
        }
        .sub-circle {
            width: 28px;
            height: 28px;
            background: #2c2c2c;
            border-radius: 100%;
        }
        body.dark .sub-circle {
            background: #3a3a3a;
        }
        .side {
            position: absolute;
            top: 0;
            left: 0;
            right: 0;
            bottom: 0;
            display: flex;
            justify-content: center;
            align-items: center;
            transition: 0.2s;
            transform: scale(0);
        }
        input:checked + .track .thumb .dark-side {
            transform: scale(1);
        }
        input:not(:checked) + .track .thumb .light-side {
            transform: scale(1);
        }
        .light-side .top {
            position: absolute;
            top: 4px;
            display: flex;
            gap: 4px;
        }
        .light-side .top .left,
        .light-side .top .right {
            width: 3px;
            height: 3px;
            background: #ffcc00;
            border-radius: 100%;
        }
        .light-side .center {
            display: flex;
            gap: 2px;
        }
        .light-side .center div {
            width: 2px;
            height: 10px;
            background: #ffcc00;
            border-radius: 2px;
        }
        .light-side .bottom {
            position: absolute;
            bottom: 4px;
        }
        .light-side .bottom .line {
            width: 12px;
            height: 2px;
            background: #ffcc00;
            border-radius: 2px;
        }
        .dark-side .circle {
            background: #ff3366;
        }
        .dark-side .sub-circle {
            background: #cc0033;
        }
        .app::-webkit-scrollbar {
            width: 6px;
        }
        .app::-webkit-scrollbar-track {
            background: rgba(0, 0, 0, 0.1);
            border-radius: 10px;
        }
        .app::-webkit-scrollbar-thumb {
            background: var(--primary);
            border-radius: 10px;
        }
    </style>
</head>
<body>
<div class="app">
    <div class="app__gradients">
        <svg style="width:0;height:0;position:absolute;" aria-hidden="true" focusable="false">
            <linearGradient id="ring" x1="0%" y1="0%" x2="100%" y2="100%">
                <stop offset="0%" stop-color="hsl(184, 66%, 44%)" />
                <stop offset="100%" stop-color="hsl(184, 66%, 74%)" />
            </linearGradient>
        </svg>
    </div>

    <div class="header">
        <button class="header__profile-btn" aria-label="Perfil">
            <svg class="icon header__profile-icon" viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="1.5">
                <circle cx="12" cy="8" r="4" />
                <path d="M5 20v-2a7 7 0 0 1 14 0v2" />
            </svg>
        </button>
        <h1>Laboratorio Nuclear</h1>
        <div class="scene">
            <div class="images">
                <img src="https://assets.codepen.io/907471/darkside.svg?v42" class="dark">
                <img src="https://assets.codepen.io/907471/lightside.svg?v42" class="light">
            </div>
            <label class="switch" id="switch" aria-label="Switch">
                <input type="checkbox" id="themeToggle">
                <div class="track">
                    <div class="lightsaber">
                        <div class="light"></div>
                        <div class="grip"></div>
                        <div class="dark"></div>
                    </div>
                    <div class="thumb">
                        <div class="side dark-side">
                            <div class="circle">
                                <div class="sub-circle"></div>
                            </div>
                        </div>
                        <div class="side light-side">
                            <div class="circle">
                                <div class="sub-circle"></div>
                            </div>
                            <div class="top">
                                <div class="left"></div>
                                <div class="right"></div>
                            </div>
                            <div class="center">
                                <div class="item-1"></div>
                                <div class="item-2"></div>
                                <div class="item-3"></div>
                                <div class="item-4"></div>
                                <div class="item-5"></div>
                            </div>
                            <div class="bottom">
                                <div class="line"></div>
                            </div>
                        </div>
                    </div>
                </div>
            </label>
        </div>
    </div>

    <div class="main__stat-blocks">
        <div class="main__stat-block">
            <div class="main__stat-graph">
                <svg class="ring" viewBox="0 0 120 120">
                    <circle class="ring-track" cx="60" cy="60" r="45" fill="none" stroke-width="8" />
                    <circle class="ring-stroke ring-stroke--steps" cx="60" cy="60" r="45" fill="none" stroke-width="8" stroke-dasharray="282.74" stroke-dashoffset="282.74" stroke-linecap="round" />
                </svg>
                <div class="main__stat-detail">
                    <span class="main__stat-value" id="statNeutrones">0</span>
                    <span class="main__stat-unit">n</span>
                </div>
            </div>
            <span class="main__stat-subtext">Neutrones</span>
        </div>
        <div class="main__stat-block">
            <div class="main__stat-graph">
                <svg class="ring" viewBox="0 0 120 120">
                    <circle class="ring-track" cx="60" cy="60" r="45" fill="none" stroke-width="8" />
                    <circle class="ring-stroke ring-stroke--cals" cx="60" cy="60" r="45" fill="none" stroke-width="8" stroke-dasharray="282.74" stroke-dashoffset="282.74" stroke-linecap="round" />
                </svg>
                <div class="main__stat-detail">
                    <span class="main__stat-value" id="statFisiones">0</span>
                    <span class="main__stat-unit">fis</span>
                </div>
            </div>
            <span class="main__stat-subtext">Fisiones</span>
        </div>
        <div class="main__stat-block">
            <div class="main__stat-graph">
                <svg class="ring" viewBox="0 0 120 120">
                    <circle class="ring-track" cx="60" cy="60" r="45" fill="none" stroke-width="8" />
                    <circle class="ring-stroke ring-stroke--miles" cx="60" cy="60" r="45" fill="none" stroke-width="8" stroke-dasharray="282.74" stroke-dashoffset="282.74" stroke-linecap="round" />
                </svg>
                <div class="main__stat-detail">
                    <span class="main__stat-value" id="statEnergia">0</span>
                    <span class="main__stat-unit">MeV</span>
                </div>
            </div>
            <span class="main__stat-subtext">Energia</span>
        </div>
    </div>

    <div class="nav-tabs">
        <button class="nav-btn active" onclick="showTab('montecarlo')">Monte Carlo</button>
        <button class="nav-btn" onclick="showTab('dispersión')">Dispersion</button>
        <button class="nav-btn" onclick="showTab('fision')">Fision 3D</button>
    </div>

    <div id="montecarlo" class="tab-content active">
        <div class="info-text">
            <strong>Metodo Monte Carlo</strong><br>
            Simulacion de poblacion de neutrones usando numeros aleatorios.
        </div>
        <div class="card">
            <div class="card-title">Parametros</div>
            <div class="input-group">
                <label>Probabilidad (p)</label>
                <input type="number" id="prob_p" value="0.5" step="0.05" min="0" max="1">
            </div>
            <div class="input-group">
                <label>Neutrones iniciales (n0)</label>
                <input type="number" id="neutrones_n0" value="100">
            </div>
            <div class="input-group">
                <label>Generaciones</label>
                <input type="number" id="generaciones" value="50" min="10" max="100">
            </div>
            <button class="btn-primary" onclick="simularMonteCarlo()">Ejecutar</button>
            <div id="resultado_mc" class="resultado">Listo</div>
        </div>
        <div class="card">
            <div class="card-title">Interpretacion</div>
            <div class="info-text" style="margin:0">
                <strong>Factor k:</strong><br>
                k < 1: Subcritico | k = 1: Critico | k > 1: Supercritico
            </div>
        </div>
        <div class="graph-container">
            <div class="graph-title">Evolucion de la Poblacion</div>
            <div id="grafica_mc" style="height: 250px;"></div>
        </div>
        <div class="graph-container">
            <div class="graph-title">Visualizacion</div>
            <canvas id="mcCanvas" class="simulation-canvas"></canvas>
        </div>
    </div>

    <div id="dispersión" class="tab-content">
        <div class="info-text">
            <strong>Dispersion Elastica de Neutrones</strong><br>
            Energia final vs angulo de dispersion (0° a 360°)
        </div>
        <div class="formula">
            $$E_l' = \\frac{1}{2}E_l[(1+\\alpha)+(1-\\alpha)\\cos\\theta_C]$$
            $$\\alpha = \\left(\\frac{A-1}{A+1}\\right)^2$$
        </div>
        <div class="card">
            <div class="card-title">Parametros</div>
            <div class="input-group">
                <label>Energia inicial (MeV)</label>
                <input type="number" id="energia_inicial_elastica" value="1.0" step="0.1">
            </div>
            <div class="input-group">
                <label>Nucleidos:</label>
                <div class="nucleido-selector" id="nucleidosList">
                    <button type="button" class="nucleido-btn" data-a="1" data-name="H-1">H-1</button>
                    <button type="button" class="nucleido-btn" data-a="12" data-name="C-12">C-12</button>
                    <button type="button" class="nucleido-btn" data-a="23" data-name="Na-23">Na-23</button>
                    <button type="button" class="nucleido-btn" data-a="238" data-name="U-238">U-238</button>
                </div>
            </div>
            <button class="btn-primary" onclick="agregarNucleido()">Agregar</button>
            <div class="flex-buttons">
                <button onclick="limpiarGrafica()">Limpiar</button>
            </div>
            <div id="nucleidos_activos" class="resultado">Activos: Ninguno</div>
        </div>
        <div class="graph-container">
            <div class="graph-title">Energia vs Angulo (0° a 360°)</div>
            <div id="grafica_elastica" style="height: 300px;"></div>
        </div>
    </div>

    <div id="fision" class="tab-content">
        <div class="info-text">
            <strong>Fision Nuclear 3D</strong><br>
            n + U-235 -> Ba-141 + Kr-92 + 2-3 n + 200 MeV
        </div>
        <div class="card">
            <div class="card-title">Simulacion 3D</div>
            <button class="btn-primary" onclick="abrirFision()">Abrir Simulacion</button>
            <div class="info-text" style="margin-top:0.8em">
                <strong>Controles:</strong><br>
                Disparar neutron | Reaccion en cadena | Reiniciar<br>
                Arrastrar: rotar | Scroll: zoom
            </div>
        </div>
    </div>
</div>

<script>
    let nucleidosActivos = [];

    const themeToggle = document.getElementById('themeToggle');
    
    function toggleTheme() {
        if (themeToggle.checked) {
            document.body.classList.add('dark');
            localStorage.setItem('theme', 'dark');
        } else {
            document.body.classList.remove('dark');
            localStorage.setItem('theme', 'light');
        }
        actualizarGraficaElastica();
        simularMonteCarlo();
    }

    themeToggle.addEventListener('change', toggleTheme);

    const savedTheme = localStorage.getItem('theme');
    if (savedTheme === 'dark') {
        document.body.classList.add('dark');
        themeToggle.checked = true;
    }

    function showTab(tabId) {
        document.querySelectorAll('.tab-content').forEach(tab => {
            tab.classList.remove('active');
        });
        document.getElementById(tabId).classList.add('active');
        
        document.querySelectorAll('.nav-btn').forEach(btn => {
            btn.classList.remove('active');
        });
        event.target.classList.add('active');
        
        if (tabId === 'montecarlo') {
            simularMonteCarlo();
        }
    }

    function energiaFinalElastica(E, A, theta) {
        const alpha = Math.pow((A - 1) / (A + 1), 2);
        const thetaRad = theta * Math.PI / 180;
        return 0.5 * E * ((1 + alpha) + (1 - alpha) * Math.cos(thetaRad));
    }

    function agregarNucleido() {
        const activeBtn = document.querySelector('.nucleido-btn.active');
        if (!activeBtn) {
            alert('Seleccione un nucleido');
            return;
        }
        
        const A = parseFloat(activeBtn.dataset.a);
        const name = activeBtn.dataset.name;
        const E = parseFloat(document.getElementById('energia_inicial_elastica').value);
        
        if (nucleidosActivos.some(n => n.A === A)) {
            alert(name + ' ya esta en la grafica');
            return;
        }
        
        nucleidosActivos.push({ A, name, E });
        actualizarListaNucleidos();
        actualizarGraficaElastica();
    }

    function actualizarListaNucleidos() {
        const container = document.getElementById('nucleidos_activos');
        if (nucleidosActivos.length === 0) {
            container.innerHTML = 'Activos: Ninguno';
            return;
        }
        container.innerHTML = 'Activos: ' + nucleidosActivos.map(n => n.name).join(', ');
    }

    function limpiarGrafica() {
        nucleidosActivos = [];
        actualizarListaNucleidos();
        actualizarGraficaElastica();
    }

    function actualizarGraficaElastica() {
        const theta_vals = Array.from({length: 361}, (_, i) => i);
        
        const traces = nucleidosActivos.map((nucleido, idx) => {
            const energy_ratio = theta_vals.map(theta => {
                const E_final = energiaFinalElastica(nucleido.E, nucleido.A, theta);
                return E_final / nucleido.E;
            });
            
            const colors = ['#00ff9c', '#667eea', '#ff6600', '#ff44cc'];
            
            return {
                x: theta_vals,
                y: energy_ratio,
                mode: 'lines',
                name: nucleido.name + ' (A=' + nucleido.A + ')',
                line: { color: colors[idx % colors.length], width: 2 }
            };
        });
        
        const isDark = document.body.classList.contains('dark');
        const fontColor = isDark ? '#fff' : '#2c3e4f';
        
        const layout = {
            template: 'plotly_dark',
            title: { text: 'Energia vs Angulo de Dispersion', font: { size: 10, color: fontColor } },
            xaxis: { title: "Angulo (grados)", range: [0, 360], dtick: 45, tickfont: { color: fontColor }, titlefont: { color: fontColor } },
            yaxis: { title: "E'/E0", range: [0, 1.05], tickfont: { color: fontColor }, titlefont: { color: fontColor } },
            plot_bgcolor: 'rgba(0,0,0,0)',
            paper_bgcolor: 'rgba(0,0,0,0)',
            legend: { font: { size: 8, color: fontColor } }
        };
        
        Plotly.newPlot('grafica_elastica', traces, layout, { responsive: true });
    }

    function monteCarlo(p, n0, pasos) {
        let neutrones = [n0];
        for (let step = 0; step < pasos; step++) {
            let nuevos = 0;
            for (let i = 0; i < neutrones[neutrones.length-1]; i++) {
                if (Math.random() < p) {
                    nuevos += Math.floor(Math.random() * 3);
                }
            }
            neutrones.push(nuevos);
        }
        return neutrones;
    }

    function simularMonteCarlo() {
        const p = parseFloat(document.getElementById('prob_p').value);
        const n0 = parseInt(document.getElementById('neutrones_n0').value);
        const pasos = parseInt(document.getElementById('generaciones').value);
        
        const datos = monteCarlo(p, n0, pasos);
        const k_efectivo = datos[datos.length-1] / datos[datos.length-2];
        
        let estado = "";
        if (k_efectivo < 0.95) estado = "Subcritico";
        else if (k_efectivo < 1.05) estado = "Critico";
        else estado = "Supercritico";
        
        document.getElementById('resultado_mc').innerHTML = 
            'Gen:' + datos.length + ' | Final:' + datos[datos.length-1] + ' | k=' + k_efectivo.toFixed(4) + '<br>' + estado;
        
        document.getElementById('statNeutrones').innerHTML = datos[datos.length-1];
        
        const isDark = document.body.classList.contains('dark');
        const fontColor = isDark ? '#fff' : '#2c3e4f';
        
        const trace = {
            y: datos,
            x: Array.from({length: datos.length}, (_, i) => i),
            mode: 'lines+markers',
            line: {color: '#00ff9c', width: 2},
            marker: {size: 2},
            fill: 'tozeroy',
            fillcolor: 'rgba(0, 255, 156, 0.1)'
        };
        
        const layout = {
            template: 'plotly_dark',
            title: { text: 'Monte Carlo', font: { size: 10, color: fontColor } },
            xaxis: { title: "Generacion", tickfont: { color: fontColor }, titlefont: { color: fontColor } },
            yaxis: { title: "Neutrones", type: 'log', tickfont: { color: fontColor }, titlefont: { color: fontColor } },
            plot_bgcolor: 'rgba(0,0,0,0)',
            paper_bgcolor: 'rgba(0,0,0,0)'
        };
        
        Plotly.newPlot('grafica_mc', [trace], layout);
        visualizarMonteCarlo3D(datos);
    }

    function visualizarMonteCarlo3D(datos) {
        const canvas = document.getElementById('mcCanvas');
        const ctx = canvas.getContext('2d');
        
        function draw() {
            canvas.width = canvas.clientWidth;
            canvas.height = canvas.clientHeight;
            const w = canvas.width;
            const h = canvas.height;
            
            const isDark = document.body.classList.contains('dark');
            const bgColor = isDark ? '#2a2a2a' : '#d0d0d0';
            
            ctx.fillStyle = bgColor;
            ctx.fillRect(0, 0, w, h);
            
            const maxVal = Math.max(...datos);
            const barWidth = w / datos.length * 0.6;
            const barSpacing = w / datos.length * 0.4;
            
            for (let i = 0; i < datos.length; i++) {
                const barHeight = Math.min((datos[i] / maxVal) * (h - 40), h - 40);
                const x = i * (barWidth + barSpacing) + barSpacing/2;
                const y = h - barHeight - 20;
                
                ctx.fillStyle = '#00ff9c';
                ctx.fillRect(x, y, barWidth, barHeight);
                
                ctx.fillStyle = isDark ? '#ccc' : '#333';
                ctx.font = '8px "Nunito"';
                ctx.textAlign = 'center';
                if (datos[i] < 100000) ctx.fillText(datos[i], x + barWidth/2, y - 2);
                ctx.fillStyle = isDark ? '#888' : '#666';
                ctx.fillText(i, x + barWidth/2, h - 8);
            }
        }
        
        draw();
        window.addEventListener('resize', () => draw());
    }

    function abrirFision() {
        window.open('fision_nuclear.html', '_blank');
    }

    document.querySelectorAll('.nucleido-btn').forEach(btn => {
        btn.addEventListener('click', function() {
            document.querySelectorAll('.nucleido-btn').forEach(b => b.classList.remove('active'));
            this.classList.add('active');
        });
    });

    document.getElementById('energia_inicial_elastica').addEventListener('input', () => {
        actualizarGraficaElastica();
    });

    window.onload = () => {
        simularMonteCarlo();
        document.querySelector('.nucleido-btn').classList.add('active');
        MathJax.Hub.Queue(["Typeset", MathJax.Hub]);
    };
</script>
</body>
</html>
"""

HTML_FISION = """
<!DOCTYPE html>
<html>
<head>
<meta charset="UTF-8">
<title>Simulador de Fision Nuclear 3D</title>
<style>
    * {
        border: 0;
        box-sizing: border-box;
        margin: 0;
        padding: 0;
    }
    :root {
        --hue: 184;
        --bg: hsl(var(--hue), 10%, 90%);
        --fg: hsl(var(--hue), 66%, 24%);
        --primary: hsl(var(--hue), 66%, 44%);
        --gradient: linear-gradient(145deg, hsl(var(--hue), 10%, 85%), hsl(var(--hue), 10%, 100%));
    }
    body { 
        margin: 0; 
        overflow: hidden; 
        background: var(--bg);
        font-family: "Nunito", sans-serif;
        position: relative;
    }
    body.dark {
        --bg: hsl(var(--hue), 10%, 10%);
        --fg: hsl(var(--hue), 66%, 94%);
        --gradient: linear-gradient(145deg, hsl(var(--hue), 10%, 15%), hsl(var(--hue), 10%, 30%));
    }
    #app { 
        width: 100%; 
        height: 100vh; 
    }
    .back-btn {
        position: absolute;
        top: 20px;
        left: 20px;
        z-index: 20;
        background: var(--gradient);
        color: var(--fg);
        padding: 8px 16px;
        cursor: pointer;
        font-family: "Nunito", sans-serif;
        font-size: 14px;
        font-weight: 600;
        text-decoration: none;
        border-radius: 1.5em;
        box-shadow: -0.5em -0.5em 1.5em hsl(0, 0%, 100%), 0.5em 0.5em 1.5em hsl(var(--hue), 5%, 65%);
        border: 1px solid rgba(255, 255, 255, 0.3);
    }
    .back-btn:active {
        transform: scale(0.95);
    }
    .controls {
        position: absolute;
        top: 20px;
        left: 120px;
        z-index: 10;
        display: flex;
        gap: 10px;
    }
    button {
        padding: 8px 16px;
        background: var(--gradient);
        color: var(--fg);
        border: none;
        cursor: pointer;
        font-family: "Nunito", sans-serif;
        font-size: 13px;
        font-weight: 600;
        border-radius: 1.5em;
        box-shadow: -0.5em -0.5em 1.5em hsl(0, 0%, 100%), 0.5em 0.5em 1.5em hsl(var(--hue), 5%, 65%);
        border: 1px solid rgba(255, 255, 255, 0.3);
    }
    button:active {
        transform: scale(0.95);
    }
    .info {
        position: absolute;
        bottom: 20px;
        left: 20px;
        color: var(--fg);
        background: var(--gradient);
        padding: 12px 16px;
        font-family: "Nunito", sans-serif;
        font-size: 12px;
        z-index: 10;
        border-radius: 1.5em;
        max-width: 280px;
        border: 1px solid rgba(255, 255, 255, 0.3);
    }
    .info h3 {
        margin: 0 0 8px 0;
        color: var(--primary);
    }
    .stats {
        position: absolute;
        top: 20px;
        right: 20px;
        color: var(--fg);
        background: var(--gradient);
        padding: 12px 16px;
        font-family: "Nunito", sans-serif;
        font-size: 12px;
        z-index: 10;
        border-radius: 1.5em;
        min-width: 180px;
        text-align: right;
        border: 1px solid rgba(255, 255, 255, 0.3);
    }
    .stats h3 {
        margin: 0 0 8px 0;
        color: var(--primary);
        text-align: left;
    }
    .stats .value {
        color: hsl(3, 90%, 55%);
        font-size: 18px;
        font-weight: bold;
    }
    .warning {
        color: hsl(3, 90%, 55%);
        animation: pulse 1s infinite;
        margin-top: 8px;
    }
    @keyframes pulse {
        0%, 100% { opacity: 1; }
        50% { opacity: 0.5; }
    }
    .scene {
        position: absolute;
        top: 20px;
        right: 20px;
        display: flex;
        justify-content: center;
        align-items: center;
        z-index: 30;
    }
    .images {
        position: absolute;
        opacity: 0;
        pointer-events: none;
    }
    .switch {
        display: flex;
        justify-content: center;
        align-items: center;
        width: 80px;
        cursor: pointer;
    }
    .switch input {
        position: absolute;
        opacity: 0;
    }
    .track {
        position: relative;
        width: 72px;
        height: 24px;
        background: #2c2c2c;
        border-radius: 60px;
        transition: 0.3s;
        box-shadow: inset 0 0 4px rgba(0,0,0,0.2);
    }
    body.dark .track {
        background: #1a1a1a;
    }
    .thumb {
        position: absolute;
        top: -6px;
        left: -6px;
        width: 36px;
        height: 36px;
        transition: 0.3s;
        display: flex;
        justify-content: center;
        align-items: center;
    }
    input:checked + .track .thumb {
        left: calc(100% - 36px + 6px);
    }
    .lightsaber {
        position: absolute;
        top: 0;
        left: 0;
        right: 0;
        bottom: 0;
        display: flex;
        justify-content: space-between;
        align-items: center;
        padding: 0 4px;
    }
    .lightsaber .light {
        width: 20px;
        height: 3px;
        background: #00ff9c;
        border-radius: 4px;
        box-shadow: 0 0 4px #00ff9c;
    }
    .lightsaber .dark {
        width: 20px;
        height: 3px;
        background: #ff3366;
        border-radius: 4px;
        box-shadow: 0 0 4px #ff3366;
    }
    .lightsaber .grip {
        width: 22px;
        height: 10px;
        background: #555;
        border-radius: 4px;
    }
    .circle {
        width: 36px;
        height: 36px;
        background: #1a1a1a;
        border-radius: 100%;
        display: flex;
        justify-content: center;
        align-items: center;
        transition: 0.2s;
        box-shadow: 0 2px 6px rgba(0,0,0,0.2);
    }
    body.dark .circle {
        background: #2a2a2a;
    }
    .sub-circle {
        width: 28px;
        height: 28px;
        background: #2c2c2c;
        border-radius: 100%;
    }
    body.dark .sub-circle {
        background: #3a3a3a;
    }
    .side {
        position: absolute;
        top: 0;
        left: 0;
        right: 0;
        bottom: 0;
        display: flex;
        justify-content: center;
        align-items: center;
        transition: 0.2s;
        transform: scale(0);
    }
    input:checked + .track .thumb .dark-side {
        transform: scale(1);
    }
    input:not(:checked) + .track .thumb .light-side {
        transform: scale(1);
    }
    .light-side .top {
        position: absolute;
        top: 4px;
        display: flex;
        gap: 4px;
    }
    .light-side .top .left,
    .light-side .top .right {
        width: 3px;
        height: 3px;
        background: #ffcc00;
        border-radius: 100%;
    }
    .light-side .center {
        display: flex;
        gap: 2px;
    }
    .light-side .center div {
        width: 2px;
        height: 10px;
        background: #ffcc00;
        border-radius: 2px;
    }
    .light-side .bottom {
        position: absolute;
        bottom: 4px;
    }
    .light-side .bottom .line {
        width: 12px;
        height: 2px;
        background: #ffcc00;
        border-radius: 2px;
    }
    .dark-side .circle {
        background: #ff3366;
    }
    .dark-side .sub-circle {
        background: #cc0033;
    }
</style>
</head>
<body>
    <div class="scene">
        <div class="images">
            <img src="https://assets.codepen.io/907471/darkside.svg?v42" class="dark">
            <img src="https://assets.codepen.io/907471/lightside.svg?v42" class="light">
        </div>
        <label class="switch" id="switch" aria-label="Switch">
            <input type="checkbox" id="themeToggleFision">
            <div class="track">
                <div class="lightsaber">
                    <div class="light"></div>
                    <div class="grip"></div>
                    <div class="dark"></div>
                </div>
                <div class="thumb">
                    <div class="side dark-side">
                        <div class="circle">
                            <div class="sub-circle"></div>
                        </div>
                    </div>
                    <div class="side light-side">
                        <div class="circle">
                            <div class="sub-circle"></div>
                        </div>
                        <div class="top">
                            <div class="left"></div>
                            <div class="right"></div>
                        </div>
                        <div class="center">
                            <div class="item-1"></div>
                            <div class="item-2"></div>
                            <div class="item-3"></div>
                            <div class="item-4"></div>
                            <div class="item-5"></div>
                        </div>
                        <div class="bottom">
                            <div class="line"></div>
                        </div>
                    </div>
                </div>
            </div>
        </label>
    </div>
    
    <a href="laboratorio_nuclear.html" class="back-btn">Volver</a>
    <div class="controls">
        <button id="fireBtn">Disparar</button>
        <button id="resetBtn">Reiniciar</button>
        <button id="chainBtn">Cadena</button>
    </div>
    <div class="stats">
        <h3>Reactor</h3>
        <p>Uranio: <span id="atomCount" class="value">0</span></p>
        <p>Neutrones: <span id="neutronCount" class="value">0</span></p>
        <p>Fisiones: <span id="fissionCount" class="value">0</span></p>
        <p>Energia: <span id="energyReleased" class="value">0</span> MeV</p>
        <div id="chainWarning" style="display:none;" class="warning">CADENA ACTIVA</div>
    </div>
    <div class="info">
        <h3>Fision Nuclear</h3>
        <p>n + U-235 -> Ba-141 + Kr-92 + 2-3 n</p>
        <p>Energia: 200 MeV por fision</p>
        <p>150 atomos | Arrastrar: rotar | Scroll: zoom</p>
    </div>
    <div id="app"></div>
    <script>
        const themeToggleFision = document.getElementById('themeToggleFision');
        
        function toggleThemeFision() {
            if (themeToggleFision.checked) {
                document.body.classList.add('dark');
                localStorage.setItem('themeFision', 'dark');
            } else {
                document.body.classList.remove('dark');
                localStorage.setItem('themeFision', 'light');
            }
        }
        
        themeToggleFision.addEventListener('change', toggleThemeFision);
        
        const savedThemeFision = localStorage.getItem('themeFision');
        if (savedThemeFision === 'dark') {
            document.body.classList.add('dark');
            themeToggleFision.checked = true;
        }
    </script>
<script type="importmap">
{
  "imports": {
    "three": "https://esm.sh/three",
    "three/addons/": "https://esm.sh/three/addons/"
  }
}
</script>
<script type="module">
import * as THREE from "three";
import { OrbitControls } from "three/addons/controls/OrbitControls.js";

let scene, camera, renderer, controls;
let atoms = [];
let neutrons = [];
let fragments = [];
let energyParticles = [];
let fissionCount = 0;
let totalEnergy = 0;
let chainReactionActive = false;
let chainReactionInterval = null;

const ENERGY_PER_FISSION = 200;
const NEUTRON_SPEED = 0.35;
const FISSION_RADIUS = 1.3;

function init() {
    scene = new THREE.Scene();
    scene.background = new THREE.Color(0x101018);
    scene.fog = new THREE.FogExp2(0x101018, 0.0005);

    camera = new THREE.PerspectiveCamera(60, innerWidth/innerHeight, 0.1, 1000);
    camera.position.set(25, 20, 32);

    renderer = new THREE.WebGLRenderer({antialias: true});
    renderer.setSize(innerWidth, innerHeight);
    renderer.shadowMap.enabled = true;
    document.getElementById("app").appendChild(renderer.domElement);

    controls = new OrbitControls(camera, renderer.domElement);
    controls.enableDamping = true;
    controls.enableZoom = true;
    controls.zoomSpeed = 1.2;
    controls.target.set(0, 0, 0);

    const ambientLight = new THREE.AmbientLight(0x222222);
    scene.add(ambientLight);
    const directionalLight = new THREE.DirectionalLight(0xffffff, 0.6);
    directionalLight.position.set(5, 10, 7);
    directionalLight.castShadow = true;
    scene.add(directionalLight);
    const backLight = new THREE.PointLight(0x4466ff, 0.3);
    backLight.position.set(-5, 0, -5);
    scene.add(backLight);
    const fillLight = new THREE.PointLight(0xffaa66, 0.25);
    fillLight.position.set(3, 5, 4);
    scene.add(fillLight);
    const rimLight = new THREE.PointLight(0xff66aa, 0.2);
    rimLight.position.set(-3, 4, -4);
    scene.add(rimLight);

    createInitialAtoms();
    animate();
    setInterval(updateUI, 100);
}

const tex = new THREE.TextureLoader().load("https://threejs.org/examples/textures/sprites/disc.png");

const createShell = (count, radius, color) => {
    const pos = [];
    const col = [];
    const pointCount = Math.min(count * 6, 120);
    for (let i = 0; i < pointCount; i++){
        const r = radius + (Math.random() - 0.5) * 0.3;
        const theta = Math.acos(2 * Math.random() - 1);
        const phi = Math.random() * Math.PI * 2;
        const x = r * Math.sin(theta) * Math.cos(phi);
        const y = r * Math.sin(theta) * Math.sin(phi);
        const z = r * Math.cos(theta);
        pos.push(x, y, z);
        const c = new THREE.Color(color);
        const fade = 0.4 + Math.random() * 0.6;
        col.push(c.r * fade, c.g * fade, c.b * fade);
    }
    const geo = new THREE.BufferGeometry();
    geo.setAttribute("position", new THREE.Float32BufferAttribute(pos, 3));
    geo.setAttribute("color", new THREE.Float32BufferAttribute(col, 3));
    const mat = new THREE.PointsMaterial({ size: 0.07, map: tex, transparent: true, vertexColors: true, blending: THREE.AdditiveBlending, depthWrite: false });
    return new THREE.Points(geo, mat);
};

function createAtom(position, color = 0xff4466) {
    const group = new THREE.Group();
    const nucleus = new THREE.Mesh(new THREE.SphereGeometry(0.55, 32, 32), new THREE.MeshStandardMaterial({ color: color, emissive: 0x331100, emissiveIntensity: 0.15, metalness: 0.7, roughness: 0.3 }));
    nucleus.castShadow = true;
    group.add(nucleus);
    const shells = [{n:2,r:1.0,color:0xff6688},{n:8,r:1.6,color:0xff7799},{n:18,r:2.3,color:0xff88aa},{n:32,r:3.0,color:0xff99bb},{n:21,r:3.8,color:0xffaacc},{n:9,r:4.7,color:0xffbbdd},{n:2,r:5.6,color:0xffccee}];
    shells.forEach(s => { const c = createShell(s.n, s.r, s.color); group.add(c); });
    group.userData = { clouds: shells.map((_,i)=>group.children[i+1]), type: "uranium" };
    group.position.copy(position);
    scene.add(group);
    return group;
}

function createInitialAtoms() {
    atoms.forEach(atom => scene.remove(atom));
    atoms = [];
    const positions = [];
    const radius = 14;
    for(let i = 0; i < 150; i++) {
        let x, y, z, d;
        do { x = (Math.random() - 0.5) * radius * 2; y = (Math.random() - 0.5) * radius * 1.5; z = (Math.random() - 0.5) * radius * 2; d = Math.sqrt(x*x + y*y + z*z); } while(d > radius);
        positions.push(new THREE.Vector3(x, y, z));
    }
    positions.forEach(pos => atoms.push(createAtom(pos)));
    fissionCount = 0;
    totalEnergy = 0;
    updateUI();
}

function createNeutron(position, direction, speed = NEUTRON_SPEED) {
    const m = new THREE.Mesh(new THREE.SphereGeometry(0.18, 16, 16), new THREE.MeshStandardMaterial({ color: 0xffdd99, emissive: 0xff8844, emissiveIntensity: 0.8 }));
    m.position.copy(position);
    m.castShadow = true;
    m.userData = { velocity: direction.clone().normalize().multiplyScalar(speed) };
    scene.add(m);
    neutrons.push(m);
    updateUI();
}

function fission(atom, neutronPosition) {
    const atomPos = atom.position.clone();
    scene.remove(atom);
    const idx = atoms.indexOf(atom);
    if(idx !== -1) atoms.splice(idx, 1);
    fissionCount++;
    totalEnergy += ENERGY_PER_FISSION;
    updateUI();
    if(chainReactionActive && atoms.length > 0) document.getElementById("chainWarning").style.display = "block";
    
    const bariumGroup = new THREE.Group();
    const baNuc = new THREE.Mesh(new THREE.SphereGeometry(0.45, 24, 24), new THREE.MeshStandardMaterial({ color: 0x66cc99, emissive: 0x226644, emissiveIntensity: 0.25 }));
    bariumGroup.add(baNuc);
    [{n:2,r:0.8,c:0x88ffaa},{n:8,r:1.25,c:0xaaffcc},{n:18,r:1.75,c:0xccffdd},{n:18,r:2.3,c:0xddffee},{n:8,r:2.9,c:0xeeffff},{n:2,r:3.5,c:0xaaffff}].forEach(s => bariumGroup.add(createShell(s.n, s.r, s.c)));
    
    const kryptonGroup = new THREE.Group();
    const krNuc = new THREE.Mesh(new THREE.SphereGeometry(0.35, 24, 24), new THREE.MeshStandardMaterial({ color: 0xff9966, emissive: 0x442200, emissiveIntensity: 0.25 }));
    kryptonGroup.add(krNuc);
    [{n:2,r:0.7,c:0xffaa88},{n:8,r:1.1,c:0xffbb99},{n:18,r:1.6,c:0xffccaa},{n:8,r:2.1,c:0xffddbb}].forEach(s => kryptonGroup.add(createShell(s.n, s.r, s.c)));
    
    const dir = neutronPosition.clone().sub(atomPos).normalize();
    bariumGroup.position.copy(atomPos.clone().add(dir.clone().multiplyScalar(1.5)));
    kryptonGroup.position.copy(atomPos.clone().add(dir.clone().multiplyScalar(-1.5)));
    bariumGroup.userData = { velocity: dir.clone().multiplyScalar(0.06) };
    kryptonGroup.userData = { velocity: dir.clone().multiplyScalar(-0.06) };
    scene.add(bariumGroup);
    scene.add(kryptonGroup);
    fragments.push(bariumGroup, kryptonGroup);
    
    const nCount = 2 + Math.floor(Math.random() * 2);
    for(let i = 0; i < nCount; i++) {
        const rd = new THREE.Vector3((Math.random()-0.5)*2, (Math.random()-0.5)*2, (Math.random()-0.5)*2).normalize();
        createNeutron(atomPos, rd, NEUTRON_SPEED * (chainReactionActive ? 1.15 : 1.0));
    }
    
    for(let i = 0; i < 60; i++) {
        const p = new THREE.Mesh(new THREE.SphereGeometry(0.06, 6, 6), new THREE.MeshBasicMaterial({ color: [0xffaa44,0xff6644,0xffaa88][Math.floor(Math.random()*3)], transparent: true, blending: THREE.AdditiveBlending }));
        p.position.copy(atomPos);
        const a1 = Math.random() * Math.PI * 2, a2 = Math.random() * Math.PI * 2, sp = 0.12 + Math.random() * 0.18;
        p.userData = { velocity: new THREE.Vector3(Math.sin(a1)*Math.cos(a2)*sp, Math.sin(a1)*Math.sin(a2)*sp, Math.cos(a1)*sp), life: 1.0, decay: 0.012 };
        scene.add(p);
        energyParticles.push(p);
    }
    
    const explLight = new THREE.PointLight(0xff8844, 2.0, 20);
    explLight.position.copy(atomPos);
    scene.add(explLight);
    let int = 2.0;
    const li = setInterval(() => { int -= 0.25; explLight.intensity = int; if(int <= 0) { clearInterval(li); scene.remove(explLight); } }, 35);
}

function startChainReaction() {
    if(chainReactionActive) return;
    chainReactionActive = true;
    document.getElementById("chainWarning").style.display = "block";
    if(atoms.length > 0 && neutrons.length === 0) {
        const start = new THREE.Vector3(0, 5, 25);
        const target = atoms[Math.floor(Math.random() * atoms.length)].position;
        createNeutron(start, new THREE.Vector3().subVectors(target, start), NEUTRON_SPEED * 1.2);
    }
    chainReactionInterval = setInterval(() => {
        if(chainReactionActive && atoms.length > 0 && neutrons.length < 10) {
            const ra = atoms[Math.floor(Math.random() * atoms.length)];
            const start = new THREE.Vector3(ra.position.x + (Math.random()-0.5)*8, ra.position.y + (Math.random()-0.5)*6 + 6, ra.position.z + (Math.random()-0.5)*8 + 18);
            createNeutron(start, new THREE.Vector3().subVectors(ra.position, start), NEUTRON_SPEED * 1.25);
        }
        if(atoms.length === 0 && chainReactionActive) stopChainReaction();
    }, 1000);
}

function stopChainReaction() {
    chainReactionActive = false;
    document.getElementById("chainWarning").style.display = "none";
    if(chainReactionInterval) { clearInterval(chainReactionInterval); chainReactionInterval = null; }
}

function resetSimulation() {
    stopChainReaction();
    neutrons.forEach(n => scene.remove(n));
    fragments.forEach(f => scene.remove(f));
    energyParticles.forEach(p => scene.remove(p));
    atoms.forEach(a => scene.remove(a));
    neutrons = []; fragments = []; energyParticles = [];
    createInitialAtoms();
    updateUI();
}

function updateUI() {
    document.getElementById("atomCount").innerHTML = atoms.length;
    document.getElementById("neutronCount").innerHTML = neutrons.length;
    document.getElementById("fissionCount").innerHTML = fissionCount;
    document.getElementById("energyReleased").innerHTML = totalEnergy.toFixed(0);
    if(atoms.length === 0) document.getElementById("chainWarning").innerHTML = "COMPLETADA";
}

function animate() {
    requestAnimationFrame(animate);
    atoms.forEach(atom => { if(atom.userData && atom.userData.clouds) { atom.userData.clouds.forEach((c, i) => { c.rotation.y += 0.0008 * (i+1); c.rotation.x += 0.0005 * (i+1); }); } });
    for(let i=fragments.length-1; i>=0; i--) { const f = fragments[i]; if(f.userData && f.userData.velocity) { f.position.add(f.userData.velocity); f.children.forEach(c => { if(c.isPoints) { c.rotation.y += 0.015; c.rotation.x += 0.01; } }); } if(Math.abs(f.position.x) > 35 || Math.abs(f.position.y) > 25 || Math.abs(f.position.z) > 35) { scene.remove(f); fragments.splice(i,1); } }
    for(let i=energyParticles.length-1; i>=0; i--) { const p = energyParticles[i]; p.position.add(p.userData.velocity); p.userData.life -= p.userData.decay; const s = p.userData.life * 0.8; p.scale.set(s,s,s); p.material.opacity = p.userData.life; if(p.userData.life <= 0) { scene.remove(p); energyParticles.splice(i,1); } }
    for(let i=neutrons.length-1; i>=0; i--) { const n = neutrons[i]; n.position.add(n.userData.velocity); n.material.emissiveIntensity = 0.5 + Math.sin(Date.now()*0.02)*0.5; let hit = false; for(let j=0; j<atoms.length; j++) { const a = atoms[j]; if(a && n.position.distanceTo(a.position) < FISSION_RADIUS) { fission(a, n.position); scene.remove(n); neutrons.splice(i,1); hit = true; break; } } if(!hit && (Math.abs(n.position.x) > 45 || Math.abs(n.position.y) > 35 || Math.abs(n.position.z) > 45)) { scene.remove(n); neutrons.splice(i,1); } }
    controls.update();
    renderer.render(scene, camera);
}

document.getElementById("fireBtn").onclick = () => { if(atoms.length === 0) { alert("No quedan atomos. Reinicie."); return; } const start = new THREE.Vector3(0, 5, 28); const target = atoms[Math.floor(Math.random() * atoms.length)].position; createNeutron(start, new THREE.Vector3().subVectors(target, start)); };
document.getElementById("resetBtn").onclick = resetSimulation;
document.getElementById("chainBtn").onclick = startChainReaction;
window.addEventListener("resize", () => { camera.aspect = innerWidth / innerHeight; camera.updateProjectionMatrix(); renderer.setSize(innerWidth, innerHeight); });
init();
</script>
</body>
</html>
"""

usuario = os.getlogin()

# Guardar archivo principal
destino_principal = fr"C:\Users\{usuario}\Downloads\laboratorio_nuclear.html"

try:
    with open(destino_principal, "w", encoding="utf-8") as file:
        file.write(HTML_PRINCIPAL)
    print(f" Archivo principal creado: {destino_principal}")
    webbrowser.open(destino_principal)
except PermissionError:
    destino_alt = fr"C:\Users\{usuario}\Desktop\laboratorio_nuclear.html"
    with open(destino_alt, "w", encoding="utf-8") as file:
        file.write(HTML_PRINCIPAL)
    print(f" Archivo guardado en escritorio: {destino_alt}")
    webbrowser.open(destino_alt)

# Guardar archivo de fisión
destino_fision = fr"C:\Users\{usuario}\Downloads\fision_nuclear.html"

try:
    with open(destino_fision, "w", encoding="utf-8") as file:
        file.write(HTML_FISION)
    print(f" Archivo de fisión creado: {destino_fision}")
except PermissionError:
    destino_alt2 = fr"C:\Users\{usuario}\Desktop\fision_nuclear.html"
    with open(destino_alt2, "w", encoding="utf-8") as file:
        file.write(HTML_FISION)
    print(f" Archivo de fisión guardado en escritorio: {destino_alt2}")

print("\n Archivos guardados correctamente!")
print(" Abre 'laboratorio_nuclear.html' en tu navegador")

 Archivo principal creado: C:\Users\jose.valdez\Downloads\laboratorio_nuclear.html
 Archivo de fisión creado: C:\Users\jose.valdez\Downloads\fision_nuclear.html

 Archivos guardados correctamente!
 Abre 'laboratorio_nuclear.html' en tu navegador
